# 🛢️ ROGII Wellbore Geology Prediction

## EDA + Residual Modeling v2

**Goal:** predict the missing tail segment of `TVT_input` for each horizontal well.

**Core statistical question:** how far should each tail row move away from the last known TVT anchor?

**Modeling boundary:** strict drilling-time features and Kaggle offline batch features are evaluated separately, so target-free future covariates never leak into the strict policy.

## 📌 Statistical Summary

> **Main idea:** `last_known_TVT` is a strong flat anchor. The model should predict only the residual that is justified by trajectory, GR/typewell alignment, and spatial geology evidence.

### 🎯 Task Shape

| Item | Value | Statistical meaning |
|---|---:|---|
| Train wells | 773 | Validation must separate wells |
| Visible test wells | 3 | Public diagnostics can be optimistic because visible test wells overlap train well ids |
| Submission rows | 14,151 | Exactly the rows where `TVT_input` is missing |
| Train tail rows | 3,783,989 | Row-level RMSE gives long tails more weight |
| Missing blocks per train well | 1 | The problem is tail forecasting, not arbitrary interpolation |

### 📈 Target Behavior

| Diagnostic | Value | Interpretation |
|---|---:|---|
| Median tail TVT range | 26.37 ft | Most wells move within a modest vertical interval |
| p90 tail TVT range | 46.82 ft | A minority of tails need real residual correction |
| Max tail TVT range | 121.84 ft | Large drift wells drive RMSE risk |
| Median tail-end drift | +1.40 ft | Flat anchor is a strong prior |
| p10 to p90 tail-end drift | -21.51 ft to +22.69 ft | Residuals must allow both directions |
| Row `dTVT` p1 to p99 | -0.06 ft to +1.07 ft | Predictions should usually be smooth |

### 🧪 Stored Grouped-CV Evidence

These numbers come from the grouped HGB residual pipeline and are kept as local validation evidence. They do **not** score the all-row LightGBM submission branch directly.

| Policy | Row-weighted RMSE | Role |
|---|---:|---|
| Constant anchor | 15.9099 | Strong null model |
| Strict `calibrated_typewell_alignment` | **14.4315** | Best stored drilling-time-compatible policy |
| Offline `offline_candidate_path_alignment` | **13.6172** | Best stored HGB batch policy without beam |

<details>
<summary>Row-weighted CV aggregation</summary>

$$
\mathrm{RMSE}_{CV}
=
\sqrt{\frac{\sum_f n_f\,\mathrm{RMSE}_f^2}{\sum_f n_f}}.
$$

</details>

### 🚀 Current Modeling Branches

| Branch | Main signal | Default status | Interpretation |
|---|---|---:|---|
| Strict HGB | prefix + trailing GR + calibrated typewell alignment | local diagnostic | conservative leakage-safe baseline |
| Offline HGB | candidate TVT paths + gap/GR texture | local diagnostic | stored best HGB CV branch |
| All-row GPU LightGBM | compact offline texture + mild post-process | Kaggle submission path | memory-safe high-capacity branch |
| Formation / plane-blend candidate | mild LGBM + spatial formation formula | opt-in experiment | heuristic branch; not OOF-selected as the default |
| 4-prior beam branch | sequence alignment of hidden GR to typewell GR | opt-in only | high-value but heavier experiment |

### 🧬 Signal Stack

| Signal family | Why it matters | Leakage boundary |
|---|---|---|
| Flat anchor | most tails stay near `last_known_TVT` | prefix target only |
| Typewell GR alignment | horizontal GR often correlates with reference GR | use prefix-derived TVT baselines, never true tail TVT |
| Offline GR texture | centered/lead/lag GR captures batch-only local shape | offline only, target-free |
| Candidate TVT paths | compares plausible residual curves against typewell GR and Geology context | offline only |
| Formation-plane/KNN imputation | recovers spatial formation-top geometry from train surfaces | opt-in; use surfaces only as auxiliary labels and fit fold-safe in CV |
| Beam alignment | treats GR matching as a sequence/path problem | offline only; disabled by default for memory safety |

### 🛡️ Decision Rules

| Rule | Reason |
|---|---|
| Train/validation split by `well_id` | avoids same-well row autocorrelation leakage |
| Evaluate only `TVT_input.isna()` rows | matches the submission target |
| Direct train-only surfaces are excluded | hidden test horizontal files do not provide them |
| Spatial formation features are allowed only through fitted imputers | the imputer can be reproduced for hidden test and is fold-safe in CV |
| Public-clean files are diagnostics | public overlap can inflate same-well LB estimates |


## 🧭 Statistical Questions

| Section | Question | Output |
|---|---|---|
| 0-0.1 | What information is observable? | Leakage rules + validation policy |
| 1 | Are files matched by well id? | File inventory checks |
| 2-3 | Which rows are targets? | Prediction ids and tail masks |
| 4-7 | How difficult are tails? | Tail length, missingness, drift, smoothness |
| 8-9 | Does typewell GR align with horizontal GR? | Prefix GR correlation diagnostics |
| 10 | How strong is the flat-anchor prior? | Constant and linear baselines |
| 11 | Which feature families are statistically admissible? | Strict and offline feature tables |
| 12-13 | Do curve shape or spatial proximity contain signal? | Knot targets and KNN drift diagnostics |
| 14 | Which well-level failure modes are visible? | Representative well plots |
| 15 | Which model form follows from the EDA? | Residual prediction with constraints |
| 16 | Which leakage-safe policy wins grouped CV? | Feature ablations and OOF diagnostics |
| 17 | Do offline path features and stronger learners add signal? | Candidate-path and model-family ablations |
| 18 | Which all-row submission candidates are written? | GPU LightGBM, formation formula, and plane-blend artifacts |


## 🔒 0. Information Policies and Leakage Rules

### 🧭 Two-track information model

| Policy | Uses | Does **not** use | Best use |
|---|---|---|---|
| **Strict drilling-time** | prefix + current/trailing row evidence | future tail shape, centered windows, tail length | conservative geosteering-style validation |
| **Offline batch** | full provided test CSV covariates | future `TVT`, target-derived summaries | Kaggle submission candidates |

> **Practical rule:** offline features may look at future **GR / trajectory** rows because they are provided in test files, but they must never look at future **TVT**.

### ✅ Allowed vs 🚫 excluded

| Feature family | Strict | Offline | Reason |
|---|---:|---:|---|
| current `MD/X/Y/Z/GR` | ✅ | ✅ | observed covariates |
| prefix `TVT_input` | ✅ | ✅ | known target prefix |
| trailing GR windows | ✅ | ✅ | no future row access |
| centered GR / lead-lag GR | 🚫 | ✅ | future covariates, target-free |
| tail length / tail fraction | 🚫 | ✅ | known only in batch mode |
| candidate-path typewell features | 🚫 | ✅ | path uses full tail position |
| beam alignment | 🚫 | ✅ | sequence feature from hidden GR |
| direct train-only surfaces | 🚫 | 🚫 | hidden-test columns unavailable |
| fold-safe formation imputer outputs | 🚫 | ✅ | reproducible spatial reference model |
| tail `TVT` labels | 🚫 | 🚫 | direct target leakage |

<details>
<summary>Notation and validation boundary</summary>

| Symbol | Meaning |
|---|---|
| $i$ | row index inside one horizontal well |
| $\mathcal{P}_w$ | known prefix where `TVT_input` is present |
| $\mathcal{T}_w$ | prediction tail where `TVT_input` is missing |
| $y_{w,i}$ | true `TVT` |
| $g_w(t)$ | typewell GR curve indexed by typewell TVT |

Strict policy allows values up to row $i$ plus known-prefix target values. Offline policy can additionally use target-free covariates from the full provided test file.

Validation separates wells:

$$
\mathcal{W}_{train}\cap\mathcal{W}_{valid}=\emptyset.
$$

</details>


## 🧯 0.1 Leakage and Statistical Risk Table

| Risk source | Statistical failure mode | Decision |
|---|---|---|
| Visible test mirrors train wells | Same-well diagnostics can be optimistic | Produce an all-train table and a public-clean table |
| Row-level random split | Same-well autocorrelation leakage | Use `GroupKFold` by `well_id` |
| Train-only surfaces | Hidden-test feature mismatch if used directly | Never use as direct row features; use only as fold-safe auxiliary labels for spatial imputers |
| Prefix target values | Valid only before Prediction Start | Use `TVT_input` only in the known prefix |
| Tail length / tail fraction | Future-tail information in strict mode | Offline features only |
| Full-tail GR aggregates | Future GR in strict mode | Offline features only if target-free |
| Full-well endpoint azimuth | Uses future trajectory endpoint | Exclude from strict features |
| Centered rolling windows | Reads future rows | Offline features only |
| Future-looking GR derivatives | Reads future GR events | Use backward differences |
| `TVT_input_bfill` | Backfills from future rows | Exclude |
| Typewell alignment to true tail TVT | Direct target leakage | Use prefix-derived TVT baselines only |
| GR affine calibration on tail | Uses prediction-region data | Fit only on known-prefix pairs |
| Centered smoothing | Future prediction dependence | Strict mode uses causal clipping |
| Nearby wells | Fold leakage if validation targets enter reference table | Require fold-safe reference tables |
| Linear extrapolation | High-variance slope error | Baseline or constrained feature only |

### Anchor hypothesis

$$
H_0: y_{w,i}=y_{w,\mathrm{PS}-1}, \quad i\in\mathcal{T}_w.
$$

📌 Improvements are useful only when residual movement beats this flat anchor without damaging flat wells.



In [ ]:
# Configure imports, data locations, and display settings.

from pathlib import Path
from collections import Counter
import json
import gc
import math
import re
import zipfile
import xml.etree.ElementTree as ET
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.errors import PerformanceWarning

try:
    from scipy.spatial import cKDTree
except Exception:
    cKDTree = None

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore', category=PerformanceWarning)

# Resolve competition data. Kaggle input mounts are preferred; local runs fall back to ./.
KAGGLE_DATA_DIRS = [
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
]
LOCAL_DATA_DIR = Path('.')
CANDIDATE_DATA_DIRS = KAGGLE_DATA_DIRS + [LOCAL_DATA_DIR]
DATA_DIR = next(
    (
        p
        for p in CANDIDATE_DATA_DIRS
        if (p / 'train').exists() and (p / 'sample_submission.csv').exists()
    ),
    LOCAL_DATA_DIR,
)
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
SAMPLE_SUBMISSION = DATA_DIR / 'sample_submission.csv'
PPTX_PATH = DATA_DIR / 'AI_wellbore_geology_prediction_task_en.pptx'
KAGGLE_WORKING_DIR = Path('/kaggle/working')
KAGGLE_NOTEBOOK_RUN = KAGGLE_WORKING_DIR.exists()
OUTPUT_DIR = KAGGLE_WORKING_DIR if KAGGLE_NOTEBOOK_RUN else DATA_DIR
LIGHTGBM_DEVICE_TYPE = 'gpu' if KAGGLE_NOTEBOOK_RUN else 'cpu'


def lightgbm_accelerator_params(device_type: str | None = None) -> dict:
    device_type = device_type or LIGHTGBM_DEVICE_TYPE
    if device_type == 'gpu':
        memory_safe = bool(globals().get('KAGGLE_MEMORY_SAFE_MODE', False))
        return {
            'device_type': 'gpu',
            'gpu_use_dp': False,
            'max_bin': 127 if memory_safe else 255,
        }
    return {
        'device_type': 'cpu',
        'force_col_wise': True,
    }


DATA_DIR_LABEL = './' if DATA_DIR == LOCAL_DATA_DIR else DATA_DIR.as_posix()
print('DATA_DIR:', DATA_DIR_LABEL)
print('train exists:', TRAIN_DIR.exists())
print('test exists:', TEST_DIR.exists())
print('sample_submission exists:', SAMPLE_SUBMISSION.exists())
print('OUTPUT_DIR:', OUTPUT_DIR.as_posix() if OUTPUT_DIR != DATA_DIR else DATA_DIR_LABEL)
print('LightGBM device type:', LIGHTGBM_DEVICE_TYPE)

## 1. File Inventory

### Statistical role

Check whether the actual file layout matches the competition documentation. All file-level statistics assume that files can be joined correctly by `well_id`.

### Checks

- Do train horizontal, typewell, and PNG id sets match?
- Do test horizontal and typewell id sets match?
- Are visible test ids a subset of train ids?

### Statistical reason

A file-matching error is more damaging than most row-level modeling mistakes. In this well-level task, a mismatched `horizontal_well` and `typewell` pair would invalidate GR correlation analysis.


In [ ]:
# Build file lists, extract well ids, and verify horizontal/typewell/image matching.

def well_id_from_path(path: Path) -> str:
    name = path.name
    if '__' in name:
        return name.split('__')[0]
    return path.stem

train_horizontal_files = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
train_typewell_files = sorted(TRAIN_DIR.glob('*__typewell.csv'))
train_png_files = sorted(TRAIN_DIR.glob('*.png'))
test_horizontal_files = sorted(TEST_DIR.glob('*__horizontal_well.csv')) if TEST_DIR.exists() else []
test_typewell_files = sorted(TEST_DIR.glob('*__typewell.csv')) if TEST_DIR.exists() else []

file_inventory = pd.DataFrame({
    'group': ['train_horizontal', 'train_typewell', 'train_png', 'test_horizontal', 'test_typewell'],
    'count': [len(train_horizontal_files), len(train_typewell_files), len(train_png_files), len(test_horizontal_files), len(test_typewell_files)],
})
display(file_inventory)

train_h_ids = {well_id_from_path(p) for p in train_horizontal_files}
train_t_ids = {well_id_from_path(p) for p in train_typewell_files}
train_png_ids = {p.stem for p in train_png_files}
test_h_ids = {well_id_from_path(p) for p in test_horizontal_files}
test_t_ids = {well_id_from_path(p) for p in test_typewell_files}

checks = {
    'train horizontal/typewell/png id sets equal': train_h_ids == train_t_ids == train_png_ids,
    'test horizontal/typewell id sets equal': test_h_ids == test_t_ids,
    'visible test ids subset of train ids': test_h_ids.issubset(train_h_ids),
}
print(json.dumps(checks, indent=2))
print('visible test ids:', sorted(test_h_ids)[:10])

### 1.1 Task Description Signals

### Statistical role

Extract the domain hints from the task PPTX before designing features.

### Main modeling implication

The central signal is matching horizontal-well GR against typewell GR on the TVT axis.

<details>
<summary>Local alignment cost</summary>

$$
\mathrm{cost}(i, t)=\left|GR^{horizontal}_{w,i}-GR^{typewell}_{w}(t)\right|
$$

Here $t$ is a candidate TVT. This cost defines the local term for a dynamic-programming or DTW-style alignment model.

</details>


In [ ]:
# Extract task-description text used to identify domain signals and constraints.

def extract_pptx_slide_text(pptx_path: Path) -> pd.DataFrame:
    if not pptx_path.exists():
        return pd.DataFrame(columns=['slide', 'text'])
    rows = []
    with zipfile.ZipFile(pptx_path) as zf:
        slide_names = sorted(
            [name for name in zf.namelist() if re.match(r'ppt/slides/slide\d+\.xml$', name)],
            key=lambda name: int(re.search(r'slide(\d+)\.xml$', name).group(1)),
        )
        for slide_name in slide_names:
            root = ET.fromstring(zf.read(slide_name))
            ns_text = '{http://schemas.openxmlformats.org/drawingml/2006/main}t'
            text_parts = [node.text.strip() for node in root.iter(ns_text) if node.text and node.text.strip()]
            rows.append({'slide': slide_name, 'text': ' | '.join(text_parts)})
    return pd.DataFrame(rows)

pptx_text = extract_pptx_slide_text(PPTX_PATH)
print('slide_count:', len(pptx_text))
display(pptx_text.head(14))

## 2. Schema and Representative Well Inspection

### Statistical role

Read one representative well to inspect the train/test schema difference and the missing-value structure.

### Main check

Train includes target and surface diagnostics that are absent from hidden test horizontal files. Safe model features must be reproducible from the hidden test schema.

<details>
<summary>Schema sets</summary>

Train-only columns:

$$
\{\mathrm{TVT},\mathrm{ANCC},\mathrm{ASTNU},\mathrm{ASTNL},\mathrm{EGFDU},\mathrm{EGFDL},\mathrm{BUDA}\}
$$

Safe test-time columns:

$$
\{\mathrm{MD},X,Y,Z,\mathrm{GR},\mathrm{TVT}_{\mathrm{input}}\}
$$

</details>

### Why this matters

The safe feature set must be fixed before modeling. Otherwise local CV may learn information that cannot be reproduced on the hidden test set.


In [ ]:
# Inspect one representative well for schema, dtypes, and missingness.

representative_well_id = sorted(train_h_ids)[0]
print('representative_well_id:', representative_well_id)

representative_train_h = pd.read_csv(TRAIN_DIR / f'{representative_well_id}__horizontal_well.csv')
representative_typewell = pd.read_csv(TRAIN_DIR / f'{representative_well_id}__typewell.csv')
representative_test_h = pd.read_csv(TEST_DIR / f'{representative_well_id}__horizontal_well.csv') if (TEST_DIR / f'{representative_well_id}__horizontal_well.csv').exists() else None

print('train horizontal shape:', representative_train_h.shape)
print('train horizontal columns:', list(representative_train_h.columns))
print('typewell shape:', representative_typewell.shape)
print('typewell columns:', list(representative_typewell.columns))
if representative_test_h is not None:
    print('visible test horizontal shape:', representative_test_h.shape)
    print('visible test horizontal columns:', list(representative_test_h.columns))

print('\nTrain horizontal missing counts:')
display(representative_train_h.isna().sum().to_frame('missing'))
print('\nTypewell missing counts:')
display(representative_typewell.isna().sum().to_frame('missing'))

print('\nTrain horizontal head:')
display(representative_train_h.head())
print('\nTypewell head:')
display(representative_typewell.head())

## 3. Prediction Zone and Submission Mapping

### Statistical role

Verify exactly which rows appear in `sample_submission.csv`.

### Main check

The submission rows should match exactly the tail rows where `TVT_input` is missing.

<details>
<summary>ID and tail-row definition</summary>

Each prediction id is built as the string `{well_id}_{row_index}`:

$$
\mathrm{id}_{w,i}=\mathrm{concat}(\mathrm{wellId}_w,\;\mathrm{underscore},\;i)
$$

The prediction row set is:

$$
\mathcal{T}_w=\{i: \mathrm{TVT}_{\mathrm{input},w,i}\;\mathrm{is}\;\mathrm{NaN}\}
$$

</details>

### Statistical reason

If the submitted rows form one tail block per well, the problem is prefix-conditioned forecasting rather than interpolation.


In [ ]:
# Define prediction tails and verify sample_submission ids match missing TVT_input rows.

def prediction_zone_info(df: pd.DataFrame) -> dict:
    mask = df['TVT_input'].isna().to_numpy()
    null_indices = np.flatnonzero(mask)
    groups = 0
    in_group = False
    for value in mask:
        if value and not in_group:
            groups += 1
            in_group = True
        elif not value:
            in_group = False
    return {
        'n_rows': len(df),
        'known_rows': int((~mask).sum()),
        'prediction_rows': int(mask.sum()),
        'first_prediction_index': int(null_indices[0]) if len(null_indices) else None,
        'last_prediction_index': int(null_indices[-1]) if len(null_indices) else None,
        'n_missing_groups': groups,
    }

print('Representative prediction zone:')
display(pd.Series(prediction_zone_info(representative_train_h)).to_frame('value'))

if SAMPLE_SUBMISSION.exists() and test_horizontal_files:
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    expected_ids = []
    for path in test_horizontal_files:
        wid = well_id_from_path(path)
        df = pd.read_csv(path)
        pred_idx = np.flatnonzero(df['TVT_input'].isna().to_numpy())
        expected_ids.extend([f'{wid}_{i}' for i in pred_idx])
    print('sample_submission rows:', len(sample))
    print('visible test prediction ids:', len(expected_ids))
    print('exact id set match:', set(sample['id']) == set(expected_ids))
    display(sample.head())

## 📊 4. Horizontal Well Aggregate Summary

### Statistical role

Summarize each train horizontal well into one row of prefix length, tail length, GR availability, geometry, and baseline difficulty.

### Key reading

Tail length, GR missingness, and constant-baseline difficulty define how much each well can influence row-level RMSE.

<details>
<summary>Well-level quantities</summary>

| Quantity | Definition |
|---|---|
| Known prefix length | $n^{\mathrm{known}}_w$ |
| Prediction tail length | $n^{\mathrm{tail}}_w$ |
| GR missing rate | $\#\mathrm{NaN}(GR) / n_w$ |
| Tail TVT range | $\max_{i\in\mathcal{T}_w} y_{w,i}-\min_{i\in\mathcal{T}_w} y_{w,i}$ |
| Constant baseline RMSE | $\sqrt{\frac{1}{n^{\mathrm{tail}}_w}\sum_{i\in\mathcal{T}_w}(y_{w,i}-y_{w,\mathrm{PS}-1})^2}$ |

</details>

> **Leakage boundary:** `tail_tvt_range`, `tail_end_delta_from_last_known`, `constant_tail_rmse`, `tail_rows`, and full-well endpoint-derived `azimuth_deg` are EDA descriptors, not strict drilling-time features.

📌 Tail length affects row-level RMSE weight: long wells can dominate the score.


In [ ]:
# Build well-level summaries for prefix length, tail length, GR missingness, geometry, and baseline difficulty.

def summarize_horizontal_file(path: Path) -> dict:
    wid = well_id_from_path(path)
    df = pd.read_csv(path)
    mask = df['TVT_input'].isna().to_numpy()
    pred_idx = np.flatnonzero(mask)
    first_pred = int(pred_idx[0]) if len(pred_idx) else len(df)
    last_known_idx = max(first_pred - 1, 0)
    y = df['TVT'].to_numpy() if 'TVT' in df.columns else np.full(len(df), np.nan)
    tail_y = y[mask] if 'TVT' in df.columns else np.array([])
    tvt_d = pd.Series(y).diff().to_numpy() if 'TVT' in df.columns else np.array([])
    dx = float(df['X'].iloc[-1] - df['X'].iloc[0])
    dy = float(df['Y'].iloc[-1] - df['Y'].iloc[0])
    dz = float(df['Z'].iloc[-1] - df['Z'].iloc[0])
    azimuth = (np.degrees(np.arctan2(dx, dy)) + 360.0) % 360.0
    prefix = df.loc[~mask]
    tail = df.loc[mask]
    return {
        'well_id': wid,
        'n_rows': len(df),
        'md_start': float(df['MD'].iloc[0]),
        'md_end': float(df['MD'].iloc[-1]),
        'md_span': float(df['MD'].iloc[-1] - df['MD'].iloc[0]),
        'known_rows': int((~mask).sum()),
        'tail_rows': int(mask.sum()),
        'ps_index': first_pred,
        'missing_tvt_input_groups': prediction_zone_info(df)['n_missing_groups'],
        'last_known_tvt': float(df['TVT_input'].iloc[last_known_idx]) if len(prefix) else np.nan,
        'ps_x': float(df['X'].iloc[first_pred]) if len(pred_idx) else np.nan,
        'ps_y': float(df['Y'].iloc[first_pred]) if len(pred_idx) else np.nan,
        'ps_z': float(df['Z'].iloc[first_pred]) if len(pred_idx) else np.nan,
        'x_start': float(df['X'].iloc[0]),
        'y_start': float(df['Y'].iloc[0]),
        'z_start': float(df['Z'].iloc[0]),
        'x_end': float(df['X'].iloc[-1]),
        'y_end': float(df['Y'].iloc[-1]),
        'z_end': float(df['Z'].iloc[-1]),
        'xy_span': float(np.hypot(dx, dy)),
        'z_delta': dz,
        'azimuth_deg': float(azimuth),
        'gr_missing_rows': int(df['GR'].isna().sum()),
        'gr_missing_rate': float(df['GR'].isna().mean()),
        'gr_missing_prefix_rate': float(prefix['GR'].isna().mean()) if len(prefix) else np.nan,
        'gr_missing_tail_rate': float(tail['GR'].isna().mean()) if len(tail) else np.nan,
        'tvt_start': float(y[0]) if len(y) else np.nan,
        'tvt_end': float(y[-1]) if len(y) else np.nan,
        'tvt_end_minus_start': float(y[-1] - y[0]) if len(y) else np.nan,
        'tail_tvt_range': float(np.nanmax(tail_y) - np.nanmin(tail_y)) if len(tail_y) else np.nan,
        'tail_end_delta_from_last_known': float(tail_y[-1] - df['TVT_input'].iloc[last_known_idx]) if len(tail_y) else np.nan,
        'tail_median_abs_step': float(np.nanmedian(np.abs(np.diff(tail_y)))) if len(tail_y) > 1 else np.nan,
        'whole_median_abs_step': float(np.nanmedian(np.abs(tvt_d[1:]))) if len(tvt_d) > 1 else np.nan,
        'constant_tail_rmse': float(np.sqrt(np.nanmean((tail_y - df['TVT_input'].iloc[last_known_idx]) ** 2))) if len(tail_y) else np.nan,
    }

h_summary = pd.DataFrame([summarize_horizontal_file(path) for path in train_horizontal_files])
print('h_summary shape:', h_summary.shape)
display(h_summary.head())

In [ ]:
# Summarize well-level distributions that affect validation and model design.

summary_cols = [
    'n_rows', 'known_rows', 'tail_rows', 'ps_index',
    'gr_missing_rate', 'gr_missing_prefix_rate', 'gr_missing_tail_rate',
    'tail_tvt_range', 'tail_end_delta_from_last_known', 'tail_median_abs_step',
    'constant_tail_rmse', 'xy_span', 'z_delta', 'azimuth_deg',
]

display(h_summary[summary_cols].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T)

print('Prediction zone missing group counts:')
print(h_summary['missing_tvt_input_groups'].value_counts().sort_index())

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
plot_specs = [
    ('n_rows', 'Rows per horizontal well'),
    ('known_rows', 'Known TVT_input prefix length'),
    ('tail_rows', 'Prediction tail length'),
    ('gr_missing_rate', 'GR missing rate'),
    ('tail_tvt_range', 'Tail TVT range'),
    ('constant_tail_rmse', 'Last-known constant RMSE by well'),
]
for ax, (col, title) in zip(axes.ravel(), plot_specs):
    sns.histplot(h_summary[col].dropna(), bins=40, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 4.1 Interpretation: Horizontal Summary

| Finding | Modeling implication |
|---|---|
| Missing `TVT_input` forms one tail block | Treat the task as prefix-conditioned forecasting |
| Tail lengths are often thousands of rows | Small slope bias can accumulate |
| GR missingness is substantial | Avoid GR-only models |
| Tail TVT ranges are often tens of feet | `last_known_TVT` is a strong anchor |

Residual target:

$$
\Delta y_{w,i}=y_{w,i}-y_{w,\mathrm{PS}-1}.
$$

🚫 Diagnostic labels such as `tail_rows`, `tail_tvt_range`, `tail_end_delta_from_last_known`, and `constant_tail_rmse` are not strict features.

## 5. Leakage Boundary and Column Roles

### 🗂️ Column role map

| Role | Examples | How the notebook uses it |
|---|---|---|
| **Observable covariates** | `MD`, `X`, `Y`, `Z`, `GR`, typewell logs | direct row features |
| **Known-prefix target** | prefix `TVT_input` | anchor, prefix statistics, GR calibration pairs |
| **Hidden-tail target** | tail `TVT` | labels for train/CV only; never features |
| **Train-only formation surfaces** | `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, `BUDA` | auxiliary labels for fold-safe spatial imputers |

### ✅ Safe pattern

| Step | Action | Leakage status |
|---|---|---:|
| 1 | fit `(X,Y) -> formation top` from training wells | ✅ reference model |
| 2 | project `formation_hat(X,Y)` onto validation/test rows | ✅ reproducible feature |
| 3 | build `TVT ≈ -Z + formation_hat + prefix_bias` | ✅ target-free formula |

### 🚫 Unsafe pattern

| Pattern | Problem |
|---|---|
| use `ANCC` directly as a feature | hidden test horizontal files do not provide it |
| fit the imputer with validation wells inside GroupKFold | fold leakage |
| use tail `TVT` summaries or bfilled target values | direct answer-key leakage |

📌 **Clean distinction:** formation tops are not observed test features, but they can define a fold-safe spatial reference model.


In [ ]:
# Classify columns by hidden-test availability and leakage risk.

train_horizontal_columns = set(representative_train_h.columns)
test_horizontal_columns = set(representative_test_h.columns) if representative_test_h is not None else {'MD', 'X', 'Y', 'Z', 'GR', 'TVT_input'}

column_roles = []
for col in representative_train_h.columns:
    if col == 'TVT':
        role = 'target_train_only'
    elif col in {'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'}:
        role = 'train_only_surface_diagnostic'
    elif col in test_horizontal_columns:
        role = 'safe_hidden_test_feature'
    else:
        role = 'unknown_train_only'
    column_roles.append({'column': col, 'role': role})

display(pd.DataFrame(column_roles))

## 6. TVT_input Consistency Check

### Statistical role

Verify that `TVT_input` is an exact copy of the target `TVT` in the known prefix.

### Why this matters

If this check passes, prefix trends, last-known TVT, and residual targets can safely be built from `TVT_input`.

<details>
<summary>Consistency condition</summary>

Let $\mathcal{K}_w$ be the row set before Prediction Start. The expected condition is:

$$
\max_{i\in\mathcal{K}_w}\left|\mathrm{TVT}_{w,i}-\mathrm{TVT}_{\mathrm{input},w,i}\right|=0
$$

</details>


In [ ]:
# Verify that TVT_input equals TVT throughout the known prefix.

max_abs_prefix_errors = []
for path in train_horizontal_files:
    df = pd.read_csv(path, usecols=['TVT', 'TVT_input'])
    known = df['TVT_input'].notna()
    if known.any():
        max_abs_prefix_errors.append(float((df.loc[known, 'TVT'] - df.loc[known, 'TVT_input']).abs().max()))

print('max over wells of max_abs(TVT - TVT_input) in known prefix:', np.nanmax(max_abs_prefix_errors))
print('wells with non-zero prefix mismatch:', sum(err > 1e-9 for err in max_abs_prefix_errors))

## 7. Target Behavior, Smoothness, and Jumps

### Statistical role

Measure how smooth the TVT curve is, how much it drifts in the tail, and whether abnormal jumps exist.

### Why this matters

If the target curve is smooth, curve/knot prediction plus smoothing may be more stable than raw row-wise prediction.

<details>
<summary>Target diagnostics</summary>

Row-level step:

$$
\Delta^{step}_{w,i}=y_{w,i}-y_{w,i-1}
$$

Tail end drift:

$$
\Delta^{end}_w=y_{w,\mathrm{end}}-y_{w,\mathrm{PS}-1}
$$

Tail range:

$$
R_w=\max_{i\in\mathcal{T}_w} y_{w,i}-\min_{i\in\mathcal{T}_w} y_{w,i}
$$

</details>


In [ ]:
# Measure TVT step smoothness, tail drift, and jump frequency.

# Collect dTVT step distribution and jump counts.
dtvt_values = []
jump_counts = Counter()
for path in train_horizontal_files:
    df = pd.read_csv(path, usecols=['TVT'])
    d = df['TVT'].diff().dropna().to_numpy()
    dtvt_values.append(d)
    for threshold in [0.1, 0.5, 1, 2, 5, 10, 25, 50, 100]:
        jump_counts[f'abs_dTVT_gt_{threshold}'] += int(np.sum(np.abs(d) > threshold))

dtvt_values = np.concatenate(dtvt_values)
print('dTVT step percentiles:')
display(pd.Series(dtvt_values).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame('dTVT'))
print('\nLarge jump counts:')
display(pd.Series(jump_counts).to_frame('count'))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(dtvt_values[(dtvt_values > -2) & (dtvt_values < 2)], bins=120, ax=axes[0])
axes[0].set_title('dTVT per 1 ft MD, clipped to [-2, 2]')
sns.histplot(h_summary['tail_end_delta_from_last_known'], bins=50, ax=axes[1])
axes[1].set_title('Tail end delta from last known TVT')
sns.histplot(h_summary['tail_tvt_range'], bins=50, ax=axes[2])
axes[2].set_title('Tail TVT range by well')
plt.tight_layout()
plt.show()

### 7.1 Interpretation: Target Behavior

### What we learned

Most row-level `dTVT` values are small, but wells are not always monotonic. TVT can increase, decrease, or remain nearly constant.

### Modeling implications

1. Blindly extrapolating the prefix slope is risky.
2. Row-wise predictions should use continuity or smoothing postprocessing.
3. A knot-level or curve-level target may be more stable than independent row-level predictions.


## 8. Typewell Data Analysis

### Statistical role

Analyze the typewell vertical reference logs: TVT sampling, GR range, and geology label distribution.

### Key relation

A typewell is a reference GR curve on the TVT axis. The horizontal-well GR sequence can help locate a candidate TVT path along that curve.

<details>
<summary>Reference-curve notation</summary>

$$
GR^{typewell}_w(t), \quad t=TVT
$$

The typewell sampling interval and GR range determine how interpolation and alignment costs should be designed.

</details>


In [ ]:
# Summarize typewell TVT sampling, GR ranges, and geology labels.

def summarize_typewell_file(path: Path) -> dict:
    wid = well_id_from_path(path)
    df = pd.read_csv(path)
    steps = df['TVT'].diff().dropna().round(6)
    labels = df['Geology'].fillna('<blank>').astype(str)
    return {
        'well_id': wid,
        'n_rows': len(df),
        'tvt_min': float(df['TVT'].min()),
        'tvt_max': float(df['TVT'].max()),
        'tvt_span': float(df['TVT'].max() - df['TVT'].min()),
        'gr_min': float(df['GR'].min()),
        'gr_median': float(df['GR'].median()),
        'gr_max': float(df['GR'].max()),
        'geology_unique_count': int(labels.nunique()),
        'mode_tvt_step': float(steps.mode().iloc[0]) if len(steps) else np.nan,
    }

t_summary = pd.DataFrame([summarize_typewell_file(path) for path in train_typewell_files])
print('t_summary shape:', t_summary.shape)
display(t_summary.head())
display(t_summary.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

geology_counter = Counter()
tvt_step_counter = Counter()
for path in train_typewell_files:
    df = pd.read_csv(path)
    geology_counter.update(df['Geology'].fillna('<blank>').astype(str).tolist())
    tvt_step_counter.update(df['TVT'].diff().dropna().round(6).tolist())

print('Top geology labels:')
display(pd.DataFrame(geology_counter.most_common(25), columns=['Geology', 'row_count']))
print('Most common TVT sampling steps:')
display(pd.DataFrame(tvt_step_counter.most_common(10), columns=['TVT_step', 'count']))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(t_summary['n_rows'], bins=50, ax=axes[0])
axes[0].set_title('Typewell rows per well')
sns.histplot(t_summary['tvt_span'], bins=50, ax=axes[1])
axes[1].set_title('Typewell TVT span')
sns.histplot(t_summary['gr_median'], bins=50, ax=axes[2])
axes[2].set_title('Typewell median GR')
plt.tight_layout()
plt.show()

## 9. Known-Prefix GR Alignment Signal

### Statistical role

Measure how well horizontal GR matches typewell GR on the same TVT axis in the known prefix.

### Statistical caution

GR has many missing values and a noisy response, so the prefix correlation should be treated as a confidence feature. Low correlation does not prove that the typewell is useless.

<details>
<summary>Prefix alignment diagnostic</summary>

For known-prefix rows $\mathcal{K}_w$, interpolate typewell GR at the `TVT_input` positions:

$$
\tilde{g}^{\mathrm{type}}_{w,i}=GR^{\mathrm{typewell}}_w(\mathrm{TVT}_{\mathrm{input},w,i})
$$

Correlation diagnostic:

$$
\rho_w=\mathrm{corr}\left(GR^{\mathrm{horizontal}}_{w,i},\tilde{g}^{\mathrm{type}}_{w,i}\right), \quad i\in\mathcal{K}_w
$$

</details>


In [ ]:
# Interpolate typewell GR onto known-prefix TVT positions and compute horizontal/typewell GR correlation.

def typewell_gr_at_tvt(typewell_df: pd.DataFrame, tvt_values: np.ndarray) -> np.ndarray:
    tw = typewell_df[['TVT', 'GR']].dropna().sort_values('TVT')
    if len(tw) < 2:
        return np.full(len(tvt_values), np.nan)
    x = tw['TVT'].to_numpy()
    y = tw['GR'].to_numpy()
    pred = np.interp(tvt_values, x, y, left=np.nan, right=np.nan)
    return pred

def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    valid = np.isfinite(a) & np.isfinite(b)
    if valid.sum() < 30:
        return np.nan
    aa = a[valid]
    bb = b[valid]
    if np.nanstd(aa) < 1e-9 or np.nanstd(bb) < 1e-9:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])

alignment_rows = []
for path in train_horizontal_files:
    wid = well_id_from_path(path)
    h = pd.read_csv(path, usecols=['GR', 'TVT_input'])
    tw = pd.read_csv(TRAIN_DIR / f'{wid}__typewell.csv')
    known = h['TVT_input'].notna()
    hv = h.loc[known, 'GR'].to_numpy()
    tvt = h.loc[known, 'TVT_input'].to_numpy()
    tw_gr = typewell_gr_at_tvt(tw, tvt)
    valid = np.isfinite(hv) & np.isfinite(tw_gr)
    alignment_rows.append({
        'well_id': wid,
        'known_valid_gr_points': int(valid.sum()),
        'prefix_horizontal_vs_typewell_gr_corr': safe_corr(hv, tw_gr),
        'prefix_horizontal_gr_mean': float(np.nanmean(hv)) if np.isfinite(hv).any() else np.nan,
        'prefix_typewell_gr_mean_at_tvt': float(np.nanmean(tw_gr)) if np.isfinite(tw_gr).any() else np.nan,
    })

alignment = pd.DataFrame(alignment_rows).merge(h_summary[['well_id', 'constant_tail_rmse', 'gr_missing_prefix_rate']], on='well_id', how='left')
display(alignment.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(alignment['prefix_horizontal_vs_typewell_gr_corr'].dropna(), bins=50, ax=axes[0])
axes[0].set_title('Known-prefix horizontal GR vs typewell GR correlation')
sns.scatterplot(
    data=alignment,
    x='prefix_horizontal_vs_typewell_gr_corr',
    y='constant_tail_rmse',
    hue='gr_missing_prefix_rate',
    palette='viridis',
    ax=axes[1],
)
axes[1].set_title('Alignment correlation vs constant baseline difficulty')
plt.tight_layout()
plt.show()

## 10. Baseline Evaluation

### Statistical role

Quantify simple baselines before fitting any model.

### Why this matters

If the constant baseline is strong, a model must avoid damaging flat wells. If linear extrapolation is weak, directly extending the prefix slope is dangerous.

<details>
<summary>Baseline definitions</summary>

Constant baseline:

$$
\hat{y}^{\mathrm{const}}_{w,i}=\mathrm{TVT}_{\mathrm{input},w,\mathrm{PS}-1}
$$

Linear prefix extrapolation baseline:

$$
\hat{y}^{\mathrm{linear}}_{w,i}=a_w x_{w,i}+b_w
$$

where $a_w,b_w$ are fitted only on the known prefix `TVT_input` values.

</details>


In [ ]:
# Evaluate constant and linear baselines on the supervised prediction tail.

def rmse_from_sse(sse: float, n: int) -> float:
    return float(np.sqrt(sse / n)) if n else np.nan

baseline_sse = Counter()
baseline_n = Counter()
per_well_baseline = []

for path in train_horizontal_files:
    wid = well_id_from_path(path)
    df = pd.read_csv(path, usecols=['MD', 'Z', 'TVT', 'TVT_input'])
    mask = df['TVT_input'].isna().to_numpy()
    if not mask.any():
        continue
    known_idx = np.flatnonzero(~mask)
    tail_idx = np.flatnonzero(mask)
    tvt = df['TVT'].to_numpy()
    tvt_input = df['TVT_input'].to_numpy()
    md = df['MD'].to_numpy()
    z = df['Z'].to_numpy()
    known_y = tvt_input[known_idx]
    last_known = float(known_y[-1])
    yy = tvt[tail_idx]
    predictions = {}
    predictions['last_known_constant'] = np.full(len(tail_idx), last_known)
    
    # Intentionally naive baselines: useful to prove that blind extrapolation is dangerous.
    for name, x, idx in [
        ('prefix_linear_md', md, known_idx),
        ('prefix_linear_z', z, known_idx),
    ]:
        coef = np.polyfit(x[idx], known_y, 1)
        predictions[name] = np.polyval(coef, x[tail_idx])
    
    last200_idx = known_idx[-min(200, len(known_idx)):]
    for name, x in [('last200_linear_md', md), ('last200_linear_z', z)]:
        last200_y = tvt_input[last200_idx]
        coef = np.polyfit(x[last200_idx], last200_y, 1)
        predictions[name] = np.polyval(coef, x[tail_idx])
    
    row = {'well_id': wid, 'tail_rows': len(tail_idx)}
    for name, pred in predictions.items():
        err = pred - yy
        sse = float(np.sum(err ** 2))
        baseline_sse[name] += sse
        baseline_n[name] += len(err)
        row[f'{name}_rmse'] = float(np.sqrt(np.mean(err ** 2)))
    per_well_baseline.append(row)

baseline_report = pd.DataFrame({
    'baseline': list(baseline_sse.keys()),
    'global_row_rmse': [rmse_from_sse(baseline_sse[k], baseline_n[k]) for k in baseline_sse.keys()],
    'n_rows': [baseline_n[k] for k in baseline_sse.keys()],
}).sort_values('global_row_rmse')
per_well_baseline = pd.DataFrame(per_well_baseline)

display(baseline_report)
display(per_well_baseline.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=baseline_report, x='global_row_rmse', y='baseline', ax=ax)
ax.set_title('Global row-level RMSE of simple baselines')
plt.tight_layout()
plt.show()

### 10.1 Interpretation: Baseline Evaluation

### What the baseline means

`last_known_constant` is the null model: the tail stays at the final known TVT anchor.

### Modeling implication

To beat this baseline, a model must gain on drifting wells without creating unnecessary movement on flat wells. Residual prediction with shrinkage and clipping directly controls this trade-off.

<details>
<summary>Null residual form</summary>

$$
H_0: \Delta y_{w,i}=0
$$

</details>


## 🧱 11. Row-Level Features

### Domain framing

The target is a hidden coordinate transform from measured depth to typewell TVT. The feature table combines anchor context, current-row geometry, GR texture, typewell matching, and optional spatial geology references.

<details>
<summary>Coordinate-transform framing</summary>

$$
t_i=\phi_w(MD_i).
$$

A simplified log-correlation model is:

$$
GR^{horizontal}_{w,i}\approx a_w\,GR^{typewell}_w(t_i)+b_w+\epsilon_i.
$$

The calibration coefficients $a_w,b_w$ are estimated only on the known prefix.

</details>

### Strict feature groups

| Group | Examples | Leakage boundary |
|---|---|---|
| Prefix context | prefix length, GR stats, TVT range, prefix slopes | prefix only |
| Row position | `MD_i - MD_PS`, `X_i - X_PS`, `Y_i - Y_PS`, `Z_i - Z_PS` | current row |
| Current GR | raw GR, prefix-normalized GR, missing flag | current row |
| GR events | backward differences, MD-normalized slopes | rows up to `i` |
| Trailing GR context | rolling mean/std/range | rows up to `i` |
| Typewell alignment | typewell GR at prefix-derived TVT baselines | no true tail TVT |
| Local offset search | best GR match around prefix-derived baseline | target-free GR only |
| Calibrated typewell | affine-calibrated typewell residuals | prefix fit only |
| Typewell interval context | Geology interval phase at prefix-derived TVT positions | reference data + prefix baseline |

### Offline feature groups

| Feature | Why offline-only |
|---|---|
| Tail length / tail fraction | uses the full prediction tail |
| Full-row fraction and MD tail fraction | uses full test-file geometry |
| Gap geometry and GR quantiles | summarizes the full hidden interval |
| Centered GR rolling and lead/lag GR | uses future GR covariates, not future TVT labels |
| Candidate-path typewell endpoints | uses tail fraction to compare plausible TVT drift paths |
| Formation-plane/KNN references | projects train formation-top geometry onto target-free coordinates |
| Beam-path typewell alignment | uses the full hidden GR sequence to form a path feature |

<details>
<summary>Residual target and local alignment equations</summary>

Residual target:

$$
\Delta^{\mathrm{target}}_{w,i}=\mathrm{TVT}_{w,i}-\mathrm{TVT}_{\mathrm{input},w,\mathrm{PS}-1}.
$$

Prefix-derived baseline:

$$
B_{w,i}=y_{w,\mathrm{PS}-1}+\hat{\beta}^{\mathrm{prefix}}_w(MD_i-MD_{PS}).
$$

Local offset search:

$$
\delta^*_{w,i}=\arg\min_{\delta\in\mathcal{D}}
\left|GR^{horizontal}_{w,i}-GR^{typewell}_w(B_{w,i}+\delta)\right|.
$$

</details>

🚫 Excluded from all policies: `TVT_input_bfill`, true tail `TVT`, target-derived tail summaries, true future TVT knots, and direct train-only surface columns.


In [ ]:
# Construct leakage-safe row-level tail features and residual targets for one well.

def safe_slope(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 2:
        return np.nan
    if np.nanstd(x[valid]) < 1e-9:
        return np.nan
    return float(np.polyfit(x[valid], y[valid], 1)[0])


def normalize_required_feature_columns(required_feature_columns) -> set[str] | None:
    if required_feature_columns is None:
        return None
    return {str(col) for col in required_feature_columns}


def needs_any(required_columns: set[str] | None, prefixes=(), exact=()) -> bool:
    if required_columns is None:
        return True
    prefixes = tuple(prefixes)
    exact = set(exact)
    for col in required_columns:
        if col in exact or any(col.startswith(prefix) for prefix in prefixes):
            return True
    return False




def typewell_gr_at_tvt(typewell_df: pd.DataFrame, tvt_values: np.ndarray) -> np.ndarray:
    if not {'TVT', 'GR'}.issubset(typewell_df.columns):
        return np.full(len(tvt_values), np.nan)
    tw = typewell_df[['TVT', 'GR']].dropna().sort_values('TVT')
    if len(tw) < 2:
        return np.full(len(tvt_values), np.nan)
    x = tw['TVT'].to_numpy(dtype=float)
    y = tw['GR'].to_numpy(dtype=float)
    return np.interp(np.asarray(tvt_values, dtype=float), x, y, left=np.nan, right=np.nan)


def safe_corr(a: np.ndarray, b: np.ndarray, min_points: int = 30) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    valid = np.isfinite(a) & np.isfinite(b)
    if valid.sum() < min_points:
        return np.nan
    aa = a[valid]
    bb = b[valid]
    if np.nanstd(aa) < 1e-9 or np.nanstd(bb) < 1e-9:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])


def recent_mean_diff(values: np.ndarray, window: int) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[-(window + 1):]
    if len(values) < 2:
        return np.nan
    return float(np.diff(values).mean())


def recent_slope_window(y_values: np.ndarray, x_values: np.ndarray, window: int) -> float:
    y_values = np.asarray(y_values, dtype=float)[-window:]
    x_values = np.asarray(x_values, dtype=float)[-window:]
    valid = np.isfinite(y_values) & np.isfinite(x_values)
    if valid.sum() < 2:
        return np.nan
    x = x_values[valid]
    y = y_values[valid]
    centered_x = x - x.mean()
    denominator = float(np.dot(centered_x, centered_x))
    if denominator <= 1e-12:
        return np.nan
    return float(np.dot(centered_x, y - y.mean()) / denominator)


def nearest_sorted_index(sorted_values: np.ndarray, target: float) -> int:
    values = np.asarray(sorted_values, dtype=float)
    if len(values) == 0:
        return 0
    idx = int(np.searchsorted(values, target, side='left'))
    if idx >= len(values):
        return len(values) - 1
    if idx > 0 and abs(values[idx - 1] - target) <= abs(values[idx] - target):
        return idx - 1
    return idx


def smooth_gr_for_beam(values: np.ndarray, fallback: float, radius: int) -> np.ndarray:
    series = pd.Series(values, dtype='float64').interpolate(limit_direction='both').fillna(fallback)
    if radius <= 0:
        return series.to_numpy(dtype=float)
    return series.rolling(radius * 2 + 1, center=True, min_periods=1).mean().to_numpy(dtype=float)


def beam_typewell_path(
    gr_values: np.ndarray,
    tw_tvt: np.ndarray,
    tw_gr: np.ndarray,
    start_tvt: float,
    beam_size: int = 10,
    move_cost: float = 20.0,
    emit_scale: float = 144.0,
    radius: int = 2,
) -> np.ndarray:
    tw_tvt = np.asarray(tw_tvt, dtype=float)
    tw_gr = np.asarray(tw_gr, dtype=float)
    valid_tw = np.isfinite(tw_tvt) & np.isfinite(tw_gr)
    tw_tvt = tw_tvt[valid_tw]
    tw_gr = tw_gr[valid_tw]
    order = np.argsort(tw_tvt)
    tw_tvt = tw_tvt[order]
    tw_gr = tw_gr[order]
    n = len(gr_values)
    if n == 0 or len(tw_tvt) < 2 or not np.isfinite(start_tvt):
        return np.full(n, np.nan)

    fallback = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    smoothed_gr = smooth_gr_for_beam(gr_values, fallback=fallback, radius=radius)
    start_idx = nearest_sorted_index(tw_tvt, start_tvt)
    states = {start_idx: 0.0}
    backpointers: list[dict[int, int]] = []

    for gr_value in smoothed_gr:
        candidates: dict[int, float] = {}
        parents: dict[int, int] = {}
        if not np.isfinite(gr_value):
            gr_value = fallback
        for idx, cost in states.items():
            for delta in (-1, 0, 1):
                next_idx = idx + delta
                if next_idx < 0 or next_idx >= len(tw_tvt):
                    continue
                emit_cost = ((gr_value - tw_gr[next_idx]) ** 2) / max(emit_scale, 1e-6)
                total_cost = cost + emit_cost + move_cost * abs(delta)
                if next_idx not in candidates or total_cost < candidates[next_idx]:
                    candidates[next_idx] = float(total_cost)
                    parents[next_idx] = idx
        kept = sorted(candidates.items(), key=lambda item: item[1])[:beam_size]
        if not kept:
            return np.full(n, np.nan)
        states = {idx: cost for idx, cost in kept}
        backpointers.append({idx: parents[idx] for idx, _ in kept})

    final_idx = min(states, key=states.get)
    path = [final_idx]
    for step in range(len(backpointers) - 1, 0, -1):
        path.append(backpointers[step][path[-1]])
    path.reverse()
    return tw_tvt[np.asarray(path, dtype=int)]


def offline_beam_feature_names(prefix: str = 'tw_beam') -> list[str]:
    names = [
        f'{prefix}_tight_delta',
        f'{prefix}_conservative_delta',
        f'{prefix}_loose_delta',
        f'{prefix}_vloose_delta',
        f'{prefix}_gap',
        f'{prefix}_spread',
        f'{prefix}_mean_delta',
        f'{prefix}_std_delta',
        f'{prefix}_tight_step',
        f'{prefix}_conservative_step',
        f'{prefix}_loose_step',
        f'{prefix}_vloose_step',
        f'{prefix}_gr_at_conservative',
        f'{prefix}_gr_at_loose',
        f'{prefix}_gr_minus_conservative',
        f'{prefix}_gr_minus_loose',
    ]
    if prefix == 'tw_beam':
        names += [
            'beam_tight_delta',
            'beam_cons_delta',
            'beam_loose_delta',
            'beam_vloose_delta',
            'beam_mean_delta',
            'beam_std_delta',
            'beam_spread',
            'beam_gap',
            'gr_minus_tw_beam_cons',
            'gr_minus_tw_beam_loose',
        ]
    return names

def safe_stat(series, func) -> float:
    values = pd.to_numeric(series, errors='coerce').to_numpy(dtype=float)
    valid = np.isfinite(values)
    if not valid.any():
        return np.nan
    return float(func(values[valid]))


def safe_affine_fit(x: np.ndarray, y: np.ndarray, min_points: int = 30) -> tuple[float, float]:
    """Fit y ~= a*x + b using only finite prefix points."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < min_points or np.nanstd(x[valid]) < 1e-9:
        return np.nan, np.nan
    a, b = np.polyfit(x[valid], y[valid], 1)
    return float(a), float(b)


def apply_affine(x: np.ndarray | float, a: float, b: float):
    if np.isfinite(a) and np.isfinite(b):
        return a * x + b
    return np.full_like(np.asarray(x, dtype=float), np.nan, dtype=float)



def local_typewell_offset_features(
    typewell_df: pd.DataFrame,
    baseline_tvt: np.ndarray,
    horizontal_gr: np.ndarray,
    offsets: np.ndarray,
    affine_a: float | None = None,
    affine_b: float | None = None,
    score_scale: float = 20.0,
    prefix: str = 'typewell_local',
) -> dict[str, np.ndarray]:
    baseline_tvt = np.asarray(baseline_tvt, dtype=float)
    horizontal_gr = np.asarray(horizontal_gr, dtype=float)
    offsets = np.asarray(offsets, dtype=float)
    n = len(baseline_tvt)
    if n == 0 or len(offsets) == 0:
        return {
            f'{prefix}_best_delta': np.full(n, np.nan),
            f'{prefix}_best_abs_resid': np.full(n, np.nan),
            f'{prefix}_zero_abs_resid': np.full(n, np.nan),
            f'{prefix}_top2_gap': np.full(n, np.nan),
            f'{prefix}_soft_delta_mean': np.full(n, np.nan),
            f'{prefix}_best_gr': np.full(n, np.nan),
        }
    candidate_tvt = baseline_tvt[:, None] + offsets[None, :]
    candidate_gr = typewell_gr_at_tvt(typewell_df, candidate_tvt.reshape(-1)).reshape(n, len(offsets))
    if affine_a is not None and affine_b is not None and np.isfinite(affine_a) and np.isfinite(affine_b):
        candidate_gr = affine_a * candidate_gr + affine_b
    abs_resid = np.abs(horizontal_gr[:, None] - candidate_gr)
    valid = np.isfinite(abs_resid)
    masked = np.where(valid, abs_resid, np.inf)
    any_valid = valid.any(axis=1)
    best_idx = np.argmin(masked, axis=1)
    best_abs = np.where(any_valid, masked[np.arange(n), best_idx], np.nan)
    best_delta = np.where(any_valid, offsets[best_idx], np.nan)
    best_gr = np.where(any_valid, candidate_gr[np.arange(n), best_idx], np.nan)
    zero_idx = int(np.argmin(np.abs(offsets)))
    zero_abs = abs_resid[:, zero_idx] if len(offsets) else np.full(n, np.nan)
    if len(offsets) >= 2:
        sorted_abs = np.sort(masked, axis=1)
        top2_gap = np.full(n, np.nan)
        top2_ok = any_valid & np.isfinite(sorted_abs[:, 1])
        top2_gap[top2_ok] = sorted_abs[top2_ok, 1] - sorted_abs[top2_ok, 0]
    else:
        top2_gap = np.full(n, np.nan)
    weights = np.zeros_like(abs_resid, dtype=float)
    weights[valid] = np.exp(-abs_resid[valid] / score_scale)
    denom = weights.sum(axis=1)
    soft_delta = np.full(n, np.nan)
    denom_ok = denom > 0
    soft_delta[denom_ok] = (weights[denom_ok] * offsets[None, :]).sum(axis=1) / denom[denom_ok]
    return {
        f'{prefix}_best_delta': best_delta,
        f'{prefix}_best_abs_resid': best_abs,
        f'{prefix}_zero_abs_resid': zero_abs,
        f'{prefix}_top2_gap': top2_gap,
        f'{prefix}_soft_delta_mean': soft_delta,
        f'{prefix}_best_gr': best_gr,
    }

CANDIDATE_PATH_ENDPOINTS = np.array([-60.0, -40.0, -25.0, -15.0, -8.0, 0.0, 8.0, 15.0, 25.0, 40.0, 60.0])


def candidate_endpoint_label(endpoint: float) -> str:
    sign = 'm' if endpoint < 0 else 'p'
    value = str(int(abs(endpoint))) if float(endpoint).is_integer() else str(abs(endpoint)).replace('.', 'p')
    return f'{sign}{value}'


def candidate_path_feature_names(prefix: str = 'tw_path', endpoints: np.ndarray = CANDIDATE_PATH_ENDPOINTS) -> list[str]:
    names: list[str] = []
    for endpoint in endpoints:
        label = candidate_endpoint_label(float(endpoint))
        names.extend([
            f'{prefix}_gr_diff_{label}',
            f'{prefix}_gr_absdiff_{label}',
            f'{prefix}_boundary_count_{label}',
            f'{prefix}_boundary_nearest_{label}',
        ])
    names.extend([
        f'{prefix}_min_absdiff',
        f'{prefix}_best_endpoint',
        f'{prefix}_best_endpoint_centered',
        f'{prefix}_top2_absdiff_gap',
        f'{prefix}_soft_endpoint_mean',
        f'{prefix}_best_gr_resid',
    ])
    return names


def typewell_boundary_tvt(typewell_df: pd.DataFrame) -> np.ndarray:
    if 'Geology' not in typewell_df.columns or 'TVT' not in typewell_df.columns:
        return np.array([], dtype=float)
    tw = typewell_df[['TVT', 'Geology']].dropna().sort_values('TVT')
    if len(tw) < 2:
        return np.array([], dtype=float)
    tvt = pd.to_numeric(tw['TVT'], errors='coerce').to_numpy(dtype=float)
    geology = tw['Geology'].astype(str).to_numpy()
    valid = np.isfinite(tvt)
    tvt = tvt[valid]
    geology = geology[valid]
    if len(tvt) < 2:
        return np.array([], dtype=float)
    change_idx = np.flatnonzero(geology[1:] != geology[:-1]) + 1
    return tvt[change_idx]




def typewell_interval_boundaries(typewell_df: pd.DataFrame) -> np.ndarray:
    if 'TVT' not in typewell_df.columns:
        return np.array([], dtype=float)
    tvt = pd.to_numeric(typewell_df['TVT'], errors='coerce').dropna().to_numpy(dtype=float)
    if len(tvt) < 2:
        return np.array([], dtype=float)
    boundaries = typewell_boundary_tvt(typewell_df)
    bounds = np.unique(np.r_[np.nanmin(tvt), boundaries, np.nanmax(tvt)])
    return np.sort(bounds[np.isfinite(bounds)])


def typewell_interval_context_features(typewell_df: pd.DataFrame, target_tvt, prefix: str) -> dict[str, np.ndarray]:
    target = np.asarray(target_tvt, dtype=float)
    original_shape = target.shape
    target_flat = target.reshape(-1)
    bounds = typewell_interval_boundaries(typewell_df)
    out = {
        f'{prefix}_prev_boundary_dist': np.full(len(target_flat), np.nan),
        f'{prefix}_next_boundary_dist': np.full(len(target_flat), np.nan),
        f'{prefix}_interval_thickness': np.full(len(target_flat), np.nan),
        f'{prefix}_interval_phase': np.full(len(target_flat), np.nan),
        f'{prefix}_interval_phase_sin': np.full(len(target_flat), np.nan),
        f'{prefix}_interval_phase_cos': np.full(len(target_flat), np.nan),
        f'{prefix}_boundary_balance': np.full(len(target_flat), np.nan),
        f'{prefix}_boundary_proximity': np.full(len(target_flat), np.nan),
    }
    if len(bounds) < 2:
        return {k: v.reshape(original_shape) for k, v in out.items()}
    valid = np.isfinite(target_flat)
    if valid.any():
        idx = np.searchsorted(bounds, target_flat[valid], side='right') - 1
        idx = np.clip(idx, 0, len(bounds) - 2)
        top = bounds[idx]
        base = bounds[idx + 1]
        thickness = base - top
        ok = np.isfinite(thickness) & (np.abs(thickness) > 1e-9)
        valid_positions = np.flatnonzero(valid)
        rows = valid_positions[ok]
        phase = (target_flat[rows] - top[ok]) / thickness[ok]
        prev_dist = target_flat[rows] - top[ok]
        next_dist = base[ok] - target_flat[rows]
        out[f'{prefix}_prev_boundary_dist'][rows] = prev_dist
        out[f'{prefix}_next_boundary_dist'][rows] = next_dist
        out[f'{prefix}_interval_thickness'][rows] = thickness[ok]
        out[f'{prefix}_interval_phase'][rows] = phase
        out[f'{prefix}_interval_phase_sin'][rows] = np.sin(np.pi * phase)
        out[f'{prefix}_interval_phase_cos'][rows] = np.cos(np.pi * phase)
        out[f'{prefix}_boundary_balance'][rows] = (next_dist - prev_dist) / (thickness[ok] + 1e-6)
        out[f'{prefix}_boundary_proximity'][rows] = np.minimum(prev_dist, next_dist) / (thickness[ok] + 1e-6)
    return {k: v.reshape(original_shape) for k, v in out.items()}


def typewell_interval_context_feature_names(prefix: str) -> list[str]:
    return [
        f'{prefix}_prev_boundary_dist',
        f'{prefix}_next_boundary_dist',
        f'{prefix}_interval_thickness',
        f'{prefix}_interval_phase',
        f'{prefix}_interval_phase_sin',
        f'{prefix}_interval_phase_cos',
        f'{prefix}_boundary_balance',
        f'{prefix}_boundary_proximity',
    ]

def boundary_count_between(boundaries: np.ndarray, start_tvt: float, target_tvt: np.ndarray) -> np.ndarray:
    target_tvt = np.asarray(target_tvt, dtype=float)
    if len(boundaries) == 0 or not np.isfinite(start_tvt):
        return np.full(len(target_tvt), np.nan)
    lo = np.minimum(start_tvt, target_tvt)
    hi = np.maximum(start_tvt, target_tvt)
    valid = np.isfinite(lo) & np.isfinite(hi)
    counts = np.full(len(target_tvt), np.nan)
    if valid.any():
        counts[valid] = ((boundaries[None, :] > lo[valid, None]) & (boundaries[None, :] <= hi[valid, None])).sum(axis=1)
    return counts


def nearest_boundary_distance(boundaries: np.ndarray, target_tvt: np.ndarray) -> np.ndarray:
    target_tvt = np.asarray(target_tvt, dtype=float)
    if len(boundaries) == 0:
        return np.full(len(target_tvt), np.nan)
    dist = np.full(len(target_tvt), np.nan)
    valid = np.isfinite(target_tvt)
    if valid.any():
        dist[valid] = np.min(np.abs(target_tvt[valid, None] - boundaries[None, :]), axis=1)
    return dist


def typewell_candidate_path_features(
    typewell_df: pd.DataFrame,
    last_known_tvt: float,
    tail_frac: np.ndarray,
    horizontal_gr: np.ndarray,
    endpoints: np.ndarray = CANDIDATE_PATH_ENDPOINTS,
    prefix: str = 'tw_path',
    score_scale: float = 20.0,
) -> dict[str, np.ndarray]:
    tail_frac = np.asarray(tail_frac, dtype=float)
    horizontal_gr = np.asarray(horizontal_gr, dtype=float)
    endpoints = np.asarray(endpoints, dtype=float)
    n = len(tail_frac)
    if n == 0 or len(endpoints) == 0 or not np.isfinite(last_known_tvt):
        return {name: np.full(n, np.nan) for name in candidate_path_feature_names(prefix, endpoints)}

    candidate_tvt = last_known_tvt + tail_frac[:, None] * endpoints[None, :]
    candidate_gr = typewell_gr_at_tvt(typewell_df, candidate_tvt.reshape(-1)).reshape(n, len(endpoints))
    gr_diff = horizontal_gr[:, None] - candidate_gr
    absdiff = np.abs(gr_diff)
    valid = np.isfinite(absdiff)
    masked = np.where(valid, absdiff, np.inf)
    any_valid = valid.any(axis=1)
    best_idx = np.argmin(masked, axis=1)
    best_abs = np.where(any_valid, masked[np.arange(n), best_idx], np.nan)
    best_endpoint = np.where(any_valid, endpoints[best_idx], np.nan)
    best_gr = np.where(any_valid, candidate_gr[np.arange(n), best_idx], np.nan)

    if len(endpoints) >= 2:
        sorted_abs = np.sort(masked, axis=1)
        top2_gap = np.full(n, np.nan)
        top2_ok = any_valid & np.isfinite(sorted_abs[:, 1])
        top2_gap[top2_ok] = sorted_abs[top2_ok, 1] - sorted_abs[top2_ok, 0]
    else:
        top2_gap = np.full(n, np.nan)

    weights = np.zeros_like(absdiff, dtype=float)
    weights[valid] = np.exp(-absdiff[valid] / score_scale)
    denom = weights.sum(axis=1)
    soft_endpoint = np.full(n, np.nan)
    denom_ok = denom > 0
    soft_endpoint[denom_ok] = (weights[denom_ok] * endpoints[None, :]).sum(axis=1) / denom[denom_ok]

    boundaries = typewell_boundary_tvt(typewell_df)
    max_endpoint = np.nanmax(np.abs(endpoints)) if len(endpoints) else np.nan
    features: dict[str, np.ndarray] = {}
    for j, endpoint in enumerate(endpoints):
        label = candidate_endpoint_label(float(endpoint))
        features[f'{prefix}_gr_diff_{label}'] = gr_diff[:, j]
        features[f'{prefix}_gr_absdiff_{label}'] = absdiff[:, j]
        features[f'{prefix}_boundary_count_{label}'] = boundary_count_between(boundaries, last_known_tvt, candidate_tvt[:, j])
        features[f'{prefix}_boundary_nearest_{label}'] = nearest_boundary_distance(boundaries, candidate_tvt[:, j])
    features[f'{prefix}_min_absdiff'] = best_abs
    features[f'{prefix}_best_endpoint'] = best_endpoint
    features[f'{prefix}_best_endpoint_centered'] = best_endpoint / max_endpoint if np.isfinite(max_endpoint) and max_endpoint > 0 else np.nan
    features[f'{prefix}_top2_absdiff_gap'] = top2_gap
    features[f'{prefix}_soft_endpoint_mean'] = soft_endpoint
    features[f'{prefix}_best_gr_resid'] = horizontal_gr - best_gr
    return features


FORMATION_TOP_COLUMNS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']
FORMATION_LABELS = [name.lower() for name in FORMATION_TOP_COLUMNS]


def formation_feature_names(include_row: bool = True) -> list[str]:
    plane_names = [f'formation_plane_{label}' for label in FORMATION_LABELS]
    names = [
        *plane_names,
        'formation_plane_min_dist',
        'formation_plane_anchor_b',
        'formation_plane_prefix_rmse',
        'formation_plane_prefix_mae',
        'formation_plane_tvt_formula',
        'formation_plane_delta_formula',
        'formation_plane_delta_from_slope_last200',
    ]
    if include_row:
        names += [
            'formation_row_ancc',
            'formation_row_ancc_std',
            'formation_row_min_dist',
            'formation_row_anchor_b',
            'formation_row_prefix_rmse',
            'formation_row_prefix_mae',
            'formation_row_tvt_formula',
            'formation_row_delta_formula',
            'formation_row_delta_from_plane',
            'formation_formula_mean_delta',
            'formation_formula_abs_gap',
        ]
    return names


def _safe_weighted_plane_predict(xy_query: np.ndarray, xy_neighbors: np.ndarray, values_neighbors: np.ndarray, weights: np.ndarray) -> np.ndarray:
    xy_query = np.asarray(xy_query, dtype=float)
    n_query = len(xy_query)
    n_targets = values_neighbors.shape[2]
    out = np.full((n_query, n_targets), np.nan, dtype=np.float32)
    for r in range(n_query):
        valid = np.isfinite(weights[r]) & (weights[r] > 0)
        if valid.sum() < 3:
            if valid.any():
                sw = weights[r, valid].sum()
                out[r] = (values_neighbors[r, valid] * weights[r, valid, None]).sum(axis=0) / max(sw, 1e-12)
            continue
        x = xy_neighbors[r, valid, 0]
        y = xy_neighbors[r, valid, 1]
        w = weights[r, valid]
        X = np.column_stack([x, y, np.ones(len(x))])
        WX = X * w[:, None]
        ata = X.T @ WX
        ata.flat[::4] += 1e-8
        atb = X.T @ (values_neighbors[r, valid] * w[:, None])
        try:
            coef = np.linalg.solve(ata, atb)
        except np.linalg.LinAlgError:
            coef = np.linalg.pinv(ata) @ atb
        qx, qy = xy_query[r]
        out[r] = (qx * coef[0] + qy * coef[1] + coef[2]).astype(np.float32)
    return out


class FormationPlaneKNN:
    """Spatial plane-fit imputer for train-only formation top columns."""

    def __init__(self, well_ids, split_dir: Path, formations=FORMATION_TOP_COLUMNS, k: int = 10):
        self.formations = list(formations)
        self.k = int(k)
        rows = []
        for well_id in sorted(well_ids):
            path = split_dir / f'{well_id}__horizontal_well.csv'
            if not path.exists():
                continue
            try:
                df = pd.read_csv(path, usecols=lambda c: c in {'X', 'Y', *self.formations}).dropna(subset=['X', 'Y'])
            except Exception:
                continue
            if df.empty or not set(self.formations).issubset(df.columns):
                continue
            valid = df[['X', 'Y', *self.formations]].dropna()
            if valid.empty:
                continue
            row = {
                'well_id': well_id,
                'x': float(valid['X'].median()),
                'y': float(valid['Y'].median()),
            }
            for col in self.formations:
                row[col] = float(valid[col].median())
            rows.append(row)
        self.df = pd.DataFrame(rows)
        self.ready = cKDTree is not None and len(self.df) >= 3
        if not self.ready:
            self.well_to_idx = {}
            self.xy = np.empty((0, 2), dtype=float)
            self.values = np.empty((0, len(self.formations)), dtype=float)
            self.scale = np.ones(2, dtype=float)
            self.tree = None
            return
        self.well_to_idx = {well_id: i for i, well_id in enumerate(self.df['well_id'].to_numpy())}
        self.xy = self.df[['x', 'y']].to_numpy(dtype=float)
        self.values = self.df[self.formations].to_numpy(dtype=float)
        self.scale = np.nanstd(self.xy, axis=0)
        self.scale = np.where(self.scale < 1e-6, 1.0, self.scale)
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_query, self_wid=None, k: int | None = None):
        xy_query = np.atleast_2d(np.asarray(xy_query, dtype=float))
        n = len(xy_query)
        if not self.ready:
            return np.full((n, len(self.formations)), np.nan, dtype=np.float32), np.full(n, np.nan, dtype=np.float32)
        k = int(k or self.k)
        n_fetch = min(max(k + 5, k), len(self.df))
        dist, idx = self.tree.query(xy_query / self.scale, k=n_fetch)
        dist = np.atleast_2d(dist)
        idx = np.atleast_2d(idx)
        if dist.shape[0] != n:
            dist = dist.reshape(n, -1)
            idx = idx.reshape(n, -1)
        if self_wid is not None and self_wid in self.well_to_idx:
            self_idx = self.well_to_idx[self_wid]
            dist = np.where(idx == self_idx, np.inf, dist)
        valid = np.isfinite(dist)
        safe_order_k = min(max(k - 1, 0), dist.shape[1] - 1)
        order = np.argpartition(np.where(valid, dist, np.inf), kth=safe_order_k, axis=1)[:, :k]
        d_k = np.take_along_axis(dist, order, axis=1)
        idx_k = np.take_along_axis(idx, order, axis=1)
        valid_k = np.isfinite(d_k)
        weights = np.where(valid_k, 1.0 / (d_k + 1e-3), 0.0)
        xy_neighbors = self.xy[idx_k]
        value_neighbors = self.values[idx_k]
        pred = _safe_weighted_plane_predict(xy_query, xy_neighbors, value_neighbors, weights)
        no_neighbor = weights.sum(axis=1) <= 1e-12
        if no_neighbor.any():
            pred[no_neighbor] = np.nanmean(self.values, axis=0).astype(np.float32)
        min_dist = np.where(valid_k, d_k, np.inf).min(axis=1).astype(np.float32)
        min_dist[~np.isfinite(min_dist)] = np.nan
        return pred.astype(np.float32), min_dist


class RowANCCKNN:
    """Sampled row-level KNN imputer for ANCC only."""

    def __init__(self, well_ids, split_dir: Path, samples_per_well: int = 400, seed: int = 42):
        xs, ys, vals, wids = [], [], [], []
        rng = np.random.default_rng(seed)
        for well_id in sorted(well_ids):
            path = split_dir / f'{well_id}__horizontal_well.csv'
            if not path.exists():
                continue
            try:
                df = pd.read_csv(path, usecols=lambda c: c in {'X', 'Y', 'ANCC'}).dropna()
            except Exception:
                continue
            if df.empty:
                continue
            if len(df) > samples_per_well:
                idx = np.sort(rng.choice(df.index.to_numpy(), size=samples_per_well, replace=False))
                df = df.loc[idx]
            xs.append(df['X'].to_numpy(dtype=float))
            ys.append(df['Y'].to_numpy(dtype=float))
            vals.append(df['ANCC'].to_numpy(dtype=float))
            wids.extend([well_id] * len(df))
        self.ready = cKDTree is not None and bool(xs)
        if not self.ready:
            self.xy = np.empty((0, 2), dtype=float)
            self.ancc = np.empty(0, dtype=float)
            self.wids = np.empty(0, dtype=object)
            self.scale = np.ones(2, dtype=float)
            self.tree = None
            return
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(vals).astype(np.float32)
        self.wids = np.asarray(wids, dtype=object)
        self.scale = np.nanstd(self.xy, axis=0)
        self.scale = np.where(self.scale < 1e-6, 1.0, self.scale)
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_query, self_wid=None, k: int = 20, extra_fetch: int = 450):
        xy_query = np.atleast_2d(np.asarray(xy_query, dtype=float))
        n = len(xy_query)
        if not self.ready:
            return (
                np.full(n, np.nan, dtype=np.float32),
                np.full(n, np.nan, dtype=np.float32),
                np.full(n, np.nan, dtype=np.float32),
            )
        n_fetch = min(max(k + extra_fetch, k), len(self.ancc))
        dist, idx = self.tree.query(xy_query / self.scale, k=n_fetch, workers=-1)
        dist = np.atleast_2d(dist)
        idx = np.atleast_2d(idx)
        if dist.shape[0] != n:
            dist = dist.reshape(n, -1)
            idx = idx.reshape(n, -1)
        if self_wid is not None:
            dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        valid = np.isfinite(dist)
        safe_order_k = min(max(k - 1, 0), dist.shape[1] - 1)
        order = np.argpartition(np.where(valid, dist, np.inf), kth=safe_order_k, axis=1)[:, :k]
        d_k = np.take_along_axis(dist, order, axis=1)
        idx_k = np.take_along_axis(idx, order, axis=1)
        valid_k = np.isfinite(d_k)
        weights = np.where(valid_k, 1.0 / (d_k + 1e-3), 0.0)
        sw = weights.sum(axis=1)
        safe = np.where(sw <= 1e-12, 1.0, sw)
        neighbor_vals = self.ancc[idx_k]
        pred = (neighbor_vals * weights).sum(axis=1) / safe
        no_neighbor = sw <= 1e-12
        if no_neighbor.any():
            pred[no_neighbor] = float(np.nanmean(self.ancc))
        var = (((neighbor_vals - pred[:, None]) ** 2) * weights).sum(axis=1) / safe
        std = np.sqrt(np.maximum(var, 0.0))
        min_dist = np.where(valid_k, d_k, np.inf).min(axis=1)
        min_dist[~np.isfinite(min_dist)] = np.nan
        return pred.astype(np.float32), std.astype(np.float32), min_dist.astype(np.float32)


def make_formation_imputers(well_ids, split_dir: Path, need_row_ancc: bool = False, seed: int = 42):
    plane = FormationPlaneKNN(well_ids, split_dir=split_dir, k=10)
    row = RowANCCKNN(well_ids, split_dir=split_dir, samples_per_well=400, seed=seed) if need_row_ancc else None
    return plane, row


def make_tail_features_for_well(
    well_id: str,
    split_dir: Path,
    include_target: bool = True,
    use_beam_features: bool | None = None,
    required_feature_columns=None,
    formation_plane_imputer=None,
    row_ancc_imputer=None,
    exclude_query_well_from_formation: bool = True,
) -> pd.DataFrame:
    if use_beam_features is None:
        use_beam_features = bool(globals().get('ENABLE_OFFLINE_BEAM_FEATURES', False))
    required_feature_columns = normalize_required_feature_columns(required_feature_columns)
    need_trailing_gr_roll = needs_any(required_feature_columns, prefixes=('gr_roll_',))
    need_offline_gr_context = needs_any(
        required_feature_columns,
        prefixes=('gr_center_',),
        exact=('gr_cumsum_since_ps',),
    )
    need_typewell_features = needs_any(
        required_feature_columns,
        prefixes=(
            'typewell_', 'gr_minus_typewell_', 'prefix_horizontal_vs_typewell_',
            'calibrated_', 'gr_minus_calibrated_', 'prefix_typewell_',
            'tw_path_', 'tw_path_ease_', 'tw_beam_', 'beam_',
        ),
    )
    need_typewell_slope_features = needs_any(
        required_feature_columns,
        prefixes=('gr_minus_typewell_slope_',),
        exact=('typewell_gr_at_slope_baseline_all', 'typewell_gr_at_slope_baseline_last200'),
    )
    need_local_typewell = needs_any(required_feature_columns, prefixes=('typewell_local_last200_',))
    need_calibrated_typewell = needs_any(
        required_feature_columns,
        prefixes=('calibrated_', 'gr_minus_calibrated_'),
        exact=(
            'prefix_typewell_gr_affine_a',
            'prefix_typewell_gr_affine_b',
            'prefix_horizontal_vs_calibrated_typewell_gr_mae',
            'prefix_horizontal_vs_calibrated_typewell_gr_rmse',
            'typewell_calibrated_gr_at_last_known_tvt',
            'typewell_calibrated_gr_at_slope_baseline_all',
            'typewell_calibrated_gr_at_slope_baseline_last200',
        ),
    )
    need_local_calibrated_typewell = needs_any(required_feature_columns, prefixes=('calibrated_typewell_local_last200_',))
    need_typewell_interval_context = needs_any(
        required_feature_columns,
        prefixes=('typewell_last_geo_', 'typewell_baseline_last200_geo_'),
    )
    need_typewell_anchor_offsets = needs_any(required_feature_columns, prefixes=('typewell_anchor_gr_diff_',))
    need_calibrated_anchor_offsets = needs_any(required_feature_columns, prefixes=('calibrated_typewell_anchor_gr_diff_',))
    need_candidate_path = needs_any(required_feature_columns, prefixes=('tw_path_', 'tw_path_ease_'))
    need_beam = bool(use_beam_features and needs_any(required_feature_columns, prefixes=('tw_beam_', 'beam_')))
    need_formation_plane = needs_any(required_feature_columns, prefixes=('formation_plane_',))
    need_row_ancc = needs_any(required_feature_columns, prefixes=('formation_row_', 'formation_formula_'))
    need_formation_features = need_formation_plane or need_row_ancc
    path = split_dir / f'{well_id}__horizontal_well.csv'
    df = pd.read_csv(path)
    mask = df['TVT_input'].isna().to_numpy()
    pred_idx = np.flatnonzero(mask)
    if len(pred_idx) == 0:
        return pd.DataFrame()
    ps = int(pred_idx[0])
    tail = df.iloc[pred_idx].copy()
    prefix = df.iloc[:ps].copy()
    last_known_tvt = float(prefix['TVT_input'].iloc[-1])
    last_known_md = float(prefix['MD'].iloc[-1])
    ps_md = float(df['MD'].iloc[ps])
    ps_x = float(df['X'].iloc[ps])
    ps_y = float(df['Y'].iloc[ps])
    ps_z = float(df['Z'].iloc[ps])
    prefix_tvt = pd.to_numeric(prefix['TVT_input'], errors='coerce')
    prefix_gr = pd.to_numeric(prefix['GR'], errors='coerce')
    tail_gr = pd.to_numeric(tail['GR'], errors='coerce').to_numpy(dtype=float)
    prefix_gr_mean = safe_stat(prefix_gr, np.mean)
    prefix_gr_std = safe_stat(prefix_gr, np.std)
    prefix_tvt_values = prefix_tvt.to_numpy(dtype=float)
    prefix_md_values = pd.to_numeric(prefix['MD'], errors='coerce').to_numpy(dtype=float)
    prefix_z_values = pd.to_numeric(prefix['Z'], errors='coerce').to_numpy(dtype=float)
    prefix_tvt_slope_all = safe_slope(prefix_md_values, prefix_tvt_values)
    prefix_tvt_slope_last200 = safe_slope(prefix['MD'].tail(200).to_numpy(), prefix_tvt.tail(200).to_numpy())
    prefix_tvt_step20 = recent_mean_diff(prefix_tvt_values, 20)
    prefix_tvt_step100 = recent_mean_diff(prefix_tvt_values, 100)
    prefix_tvt_md_slope100 = recent_slope_window(prefix_tvt_values, prefix_md_values, 100)
    prefix_tvt_z_slope100 = recent_slope_window(prefix_tvt_values, prefix_z_values, 100)
    
    # Use only the known prefix to estimate orientation. Full-well endpoint azimuth would use future tail information.
    if len(prefix) >= 2:
        dx_prefix = float(prefix['X'].iloc[-1] - prefix['X'].iloc[0])
        dy_prefix = float(prefix['Y'].iloc[-1] - prefix['Y'].iloc[0])
        prefix_azimuth = (np.degrees(np.arctan2(dx_prefix, dy_prefix)) + 360.0) % 360.0
    else:
        prefix_azimuth = np.nan
    
    md_since_ps = tail['MD'].to_numpy(dtype=float) - ps_md
    tail_len = len(tail)
    tail_row_number = np.arange(tail_len, dtype=float)
    tail_frac = tail_row_number / max(tail_len - 1, 1)
    n_rows = len(df)
    row_frac = pred_idx.astype(float) / max(n_rows - 1, 1)
    md_tail_span = float(tail['MD'].iloc[-1] - ps_md) if len(tail) else np.nan
    md_tail_frac = md_since_ps / md_tail_span if np.isfinite(md_tail_span) and abs(md_tail_span) > 1e-9 else np.full(tail_len, np.nan)
    tail_gr_missing_rate = float(pd.to_numeric(tail['GR'], errors='coerce').isna().mean())
    tail_frac2 = tail_frac ** 2
    tail_frac3 = tail_frac ** 3
    sqrt_tail_frac = np.sqrt(np.clip(tail_frac, 0.0, None))
    log1p_tail_row = np.log1p(tail_row_number)
    sin_tail_frac_pi = np.sin(np.pi * tail_frac)
    sin_tail_frac_2pi = np.sin(2.0 * np.pi * tail_frac)
    cos_tail_frac_3pi = np.cos(3.0 * np.pi * tail_frac)
    gap_x_delta = float(tail['X'].iloc[-1] - tail['X'].iloc[0]) if len(tail) else np.nan
    gap_y_delta = float(tail['Y'].iloc[-1] - tail['Y'].iloc[0]) if len(tail) else np.nan
    gap_z_delta = float(tail['Z'].iloc[-1] - tail['Z'].iloc[0]) if len(tail) else np.nan
    gap_xy_span = float(np.hypot(gap_x_delta, gap_y_delta)) if np.isfinite(gap_x_delta) and np.isfinite(gap_y_delta) else np.nan
    gap_z_over_xy = gap_z_delta / (gap_xy_span + 1.0) if np.isfinite(gap_z_delta) and np.isfinite(gap_xy_span) else np.nan
    finite_tail_gr = tail_gr[np.isfinite(tail_gr)]
    if len(finite_tail_gr):
        gap_gr_mean = float(np.mean(finite_tail_gr))
        gap_gr_std = float(np.std(finite_tail_gr))
        gap_gr_min = float(np.min(finite_tail_gr))
        gap_gr_max = float(np.max(finite_tail_gr))
        gap_gr_q = {q: float(np.quantile(finite_tail_gr, q)) for q in [0.05, 0.25, 0.50, 0.75, 0.95]}
    else:
        gap_gr_mean = gap_gr_std = gap_gr_min = gap_gr_max = np.nan
        gap_gr_q = {q: np.nan for q in [0.05, 0.25, 0.50, 0.75, 0.95]}
    x_delta_ps = tail['X'].to_numpy(dtype=float) - ps_x
    y_delta_ps = tail['Y'].to_numpy(dtype=float) - ps_y
    z_delta_ps = tail['Z'].to_numpy(dtype=float) - ps_z
    safe_md_since_ps = np.where(np.abs(md_since_ps) > 1e-9, md_since_ps, np.nan)
    dist_xyz_ps = np.sqrt(x_delta_ps ** 2 + y_delta_ps ** 2 + z_delta_ps ** 2)
    slope_delta_all = prefix_tvt_slope_all * md_since_ps if np.isfinite(prefix_tvt_slope_all) else np.full(len(tail), np.nan)
    slope_delta_last200 = prefix_tvt_slope_last200 * md_since_ps if np.isfinite(prefix_tvt_slope_last200) else np.full(len(tail), np.nan)
    slope_baseline_tvt_all = last_known_tvt + slope_delta_all
    slope_baseline_tvt_last200 = last_known_tvt + slope_delta_last200
    prefix_valid_gr = prefix_gr.dropna()
    if len(prefix_valid_gr):
        prefix_last_valid_gr = float(prefix_valid_gr.iloc[-1])
        prefix_last_valid_gr_index = int(prefix_valid_gr.index[-1])
        prefix_last_valid_gr_md = float(df['MD'].iloc[prefix_last_valid_gr_index])
    else:
        prefix_last_valid_gr = np.nan
        prefix_last_valid_gr_index = -1
        prefix_last_valid_gr_md = np.nan
    gr_prefix_z = (tail_gr - prefix_gr_mean) / (prefix_gr_std + 1e-6) if np.isfinite(prefix_gr_std) else np.full(len(tail), np.nan)
    
    md_full = pd.to_numeric(df['MD'], errors='coerce')
    x_full = pd.to_numeric(df['X'], errors='coerce')
    y_full = pd.to_numeric(df['Y'], errors='coerce')
    z_full = pd.to_numeric(df['Z'], errors='coerce')
    gr_full_numeric = pd.to_numeric(df['GR'], errors='coerce')
    md_step_1 = md_full.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    x_step_1 = x_full.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    y_step_1 = y_full.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    z_step_1 = z_full.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    gr_diff_1 = gr_full_numeric.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    gr_diff_5 = gr_full_numeric.diff(5).iloc[pred_idx].to_numpy(dtype=float)
    safe_md_step_1 = np.where(np.abs(md_step_1) > 1e-9, md_step_1, np.nan)
    
    out = pd.DataFrame({
        'well_id': well_id,
        'row_index': pred_idx,
        'id': [f'{well_id}_{i}' for i in pred_idx],
        'MD': tail['MD'].to_numpy(),
        'X': tail['X'].to_numpy(),
        'Y': tail['Y'].to_numpy(),
        'Z': tail['Z'].to_numpy(),
        'GR': tail_gr,
        'GR_isna': tail['GR'].isna().astype(int).to_numpy(),
        'GR_prefix_z': gr_prefix_z,
        'gr_diff_1': gr_diff_1,
        'gr_diff_5': gr_diff_5,
        'gr_slope_md_1': gr_diff_1 / safe_md_step_1,
        'md_step_1': md_step_1,
        'x_step_1': x_step_1,
        'y_step_1': y_step_1,
        'z_step_1': z_step_1,
        'trajectory_step_1': np.sqrt(x_step_1 ** 2 + y_step_1 ** 2 + z_step_1 ** 2),
        'z_slope_md_1': z_step_1 / safe_md_step_1,
        'last_known_TVT': last_known_tvt,
        'last_known_MD': last_known_md,
        'tail_len': tail_len,
        'tail_row_number': tail_row_number,
        'tail_frac': tail_frac,
        'tail_frac2': tail_frac2,
        'tail_frac3': tail_frac3,
        'sqrt_tail_frac': sqrt_tail_frac,
        'log1p_tail_row': log1p_tail_row,
        'sin_tail_frac_pi': sin_tail_frac_pi,
        'sin_tail_frac_2pi': sin_tail_frac_2pi,
        'cos_tail_frac_3pi': cos_tail_frac_3pi,
        'n_rows': n_rows,
        'row_frac': row_frac,
        'md_tail_span': md_tail_span,
        'md_tail_frac': md_tail_frac,
        'gap_md_span': md_tail_span,
        'gap_x_delta': gap_x_delta,
        'gap_y_delta': gap_y_delta,
        'gap_z_delta': gap_z_delta,
        'gap_xy_span': gap_xy_span,
        'gap_z_over_xy': gap_z_over_xy,
        'gap_gr_mean': gap_gr_mean,
        'gap_gr_std': gap_gr_std,
        'gap_gr_min': gap_gr_min,
        'gap_gr_p05': gap_gr_q[0.05],
        'gap_gr_p25': gap_gr_q[0.25],
        'gap_gr_p50': gap_gr_q[0.50],
        'gap_gr_p75': gap_gr_q[0.75],
        'gap_gr_p95': gap_gr_q[0.95],
        'gap_gr_max': gap_gr_max,
        'tail_gr_missing_rate': tail_gr_missing_rate,
        'md_since_ps': md_since_ps,
        'x_delta_ps': x_delta_ps,
        'y_delta_ps': y_delta_ps,
        'z_delta_ps': z_delta_ps,
        'xy_dist_ps': np.hypot(x_delta_ps, y_delta_ps),
        'dist_xyz_ps': dist_xyz_ps,
        'dx_per_md_since_ps': x_delta_ps / safe_md_since_ps,
        'dy_per_md_since_ps': y_delta_ps / safe_md_since_ps,
        'dz_per_md_since_ps': z_delta_ps / safe_md_since_ps,
        'prefix_len': len(prefix),
        'prefix_azimuth_deg': prefix_azimuth,
        'prefix_gr_missing_rate': float(prefix_gr.isna().mean()),
        'prefix_gr_mean': prefix_gr_mean,
        'prefix_gr_std': prefix_gr_std,
        'prefix_gr_min': safe_stat(prefix_gr, np.min),
        'prefix_gr_max': safe_stat(prefix_gr, np.max),
        'prefix_last_valid_gr': prefix_last_valid_gr,
        'rows_since_prefix_last_valid_gr': pred_idx - prefix_last_valid_gr_index if prefix_last_valid_gr_index >= 0 else np.nan,
        'md_since_prefix_last_valid_gr': tail['MD'].to_numpy(dtype=float) - prefix_last_valid_gr_md,
        'gr_minus_prefix_last_valid_gr': tail_gr - prefix_last_valid_gr,
        'gr_minus_prefix_gr_mean': tail_gr - prefix_gr_mean,
        'prefix_tvt_min': safe_stat(prefix_tvt, np.min),
        'prefix_tvt_max': safe_stat(prefix_tvt, np.max),
        'prefix_tvt_range': safe_stat(prefix_tvt, np.max) - safe_stat(prefix_tvt, np.min),
        'prefix_tvt_mean': safe_stat(prefix_tvt, np.mean),
        'prefix_tvt_std': safe_stat(prefix_tvt, np.std),
        'prefix_tvt_slope_md_all': prefix_tvt_slope_all,
        'prefix_tvt_slope_md_last200': prefix_tvt_slope_last200,
        'prefix_tvt_step20': prefix_tvt_step20,
        'prefix_tvt_step100': prefix_tvt_step100,
        'prefix_tvt_md_slope100': prefix_tvt_md_slope100,
        'prefix_tvt_z_slope100': prefix_tvt_z_slope100,
        'slope_baseline_delta_all': slope_delta_all,
        'slope_baseline_delta_last200': slope_delta_last200,
        'slope_baseline_tvt_all': slope_baseline_tvt_all,
        'slope_baseline_tvt_last200': slope_baseline_tvt_last200,
    })
    
    # Rolling GR features are trailing only for strict mode.
    gr_full = df['GR'].copy()
    if need_trailing_gr_roll:
        for window in [25, 100, 300]:
            roll = gr_full.rolling(window, min_periods=max(5, window // 5))
            out[f'gr_roll_mean_{window}'] = roll.mean().iloc[pred_idx].to_numpy()
            out[f'gr_roll_std_{window}'] = roll.std().iloc[pred_idx].to_numpy()
            out[f'gr_roll_min_{window}'] = roll.min().iloc[pred_idx].to_numpy()
            out[f'gr_roll_max_{window}'] = roll.max().iloc[pred_idx].to_numpy()
            out[f'gr_roll_range_{window}'] = (roll.max() - roll.min()).iloc[pred_idx].to_numpy()
    
    # Offline features use target-free covariates from the full provided horizontal file.
    gr_full_filled = None
    if need_offline_gr_context:
        for window in [5, 21, 51, 151, 301]:
            center_roll = gr_full.rolling(window, center=True, min_periods=max(1, min(5, window // 2)))
            out[f'gr_center_roll_mean_{window}'] = center_roll.mean().iloc[pred_idx].to_numpy()
            out[f'gr_center_roll_std_{window}'] = center_roll.std().iloc[pred_idx].to_numpy()
            out[f'gr_center_roll_range_{window}'] = (center_roll.max() - center_roll.min()).iloc[pred_idx].to_numpy()
        fallback_gr = prefix_gr_mean if np.isfinite(prefix_gr_mean) else 0.0
        gr_full_filled = gr_full_numeric.interpolate(limit_direction='both').fillna(fallback_gr)
        gr_cumsum = gr_full_filled.cumsum()
        cumsum_anchor = float(gr_cumsum.iloc[ps - 1]) if ps > 0 else 0.0
        out['gr_center_grad_1'] = gr_full_filled.diff().fillna(0.0).iloc[pred_idx].to_numpy()
        out['gr_center_lag1'] = gr_full_filled.shift(1).bfill().iloc[pred_idx].to_numpy()
        out['gr_center_lead1'] = gr_full_filled.shift(-1).ffill().iloc[pred_idx].to_numpy()
        out['gr_center_lag5'] = gr_full_filled.shift(5).bfill().iloc[pred_idx].to_numpy()
        out['gr_center_lead5'] = gr_full_filled.shift(-5).ffill().iloc[pred_idx].to_numpy()
        out['gr_cumsum_since_ps'] = (gr_cumsum.iloc[pred_idx] - cumsum_anchor).to_numpy()
    
    # Formation top features use train-only surfaces only as auxiliary labels for a spatial imputer.
    if need_formation_features:
        self_wid = well_id if exclude_query_well_from_formation else None
        prefix_xy = prefix[['X', 'Y']].to_numpy(dtype=float)
        tail_xy = tail[['X', 'Y']].to_numpy(dtype=float)
        prefix_z = prefix['Z'].to_numpy(dtype=float)
        tail_z = tail['Z'].to_numpy(dtype=float)
        plane_delta = np.full(len(tail), np.nan, dtype=float)
        if need_formation_plane and formation_plane_imputer is not None and getattr(formation_plane_imputer, 'ready', False):
            plane_prefix, _ = formation_plane_imputer.impute(prefix_xy, self_wid=self_wid)
            plane_tail, plane_dist = formation_plane_imputer.impute(tail_xy, self_wid=self_wid)
            for label_idx, label in enumerate(FORMATION_LABELS):
                out[f'formation_plane_{label}'] = plane_tail[:, label_idx]
            plane_prefix_ancc = plane_prefix[:, 0]
            plane_tail_ancc = plane_tail[:, 0]
            valid_b = np.isfinite(prefix_tvt_values) & np.isfinite(prefix_z) & np.isfinite(plane_prefix_ancc)
            plane_anchor_b = float(np.nanmedian(prefix_tvt_values[valid_b] + prefix_z[valid_b] - plane_prefix_ancc[valid_b])) if valid_b.any() else np.nan
            plane_tvt_formula = -tail_z + plane_tail_ancc + plane_anchor_b
            plane_prefix_formula = -prefix_z + plane_prefix_ancc + plane_anchor_b
            prefix_valid_formula = valid_b & np.isfinite(plane_prefix_formula)
            plane_prefix_resid = prefix_tvt_values[prefix_valid_formula] - plane_prefix_formula[prefix_valid_formula]
            out['formation_plane_min_dist'] = plane_dist
            out['formation_plane_anchor_b'] = plane_anchor_b
            out['formation_plane_prefix_rmse'] = float(np.sqrt(np.mean(plane_prefix_resid ** 2))) if len(plane_prefix_resid) else np.nan
            out['formation_plane_prefix_mae'] = float(np.mean(np.abs(plane_prefix_resid))) if len(plane_prefix_resid) else np.nan
            out['formation_plane_tvt_formula'] = plane_tvt_formula
            out['formation_plane_delta_formula'] = plane_tvt_formula - last_known_tvt
            out['formation_plane_delta_from_slope_last200'] = (plane_tvt_formula - slope_baseline_tvt_last200)
            plane_delta = np.asarray(out['formation_plane_delta_formula'], dtype=float)
        row_delta = np.full(len(tail), np.nan, dtype=float)
        if need_row_ancc and row_ancc_imputer is not None and getattr(row_ancc_imputer, 'ready', False):
            row_prefix_ancc, row_prefix_std, _ = row_ancc_imputer.impute(prefix_xy, self_wid=self_wid)
            row_tail_ancc, row_tail_std, row_dist = row_ancc_imputer.impute(tail_xy, self_wid=self_wid)
            valid_b = np.isfinite(prefix_tvt_values) & np.isfinite(prefix_z) & np.isfinite(row_prefix_ancc)
            row_anchor_b = float(np.nanmedian(prefix_tvt_values[valid_b] + prefix_z[valid_b] - row_prefix_ancc[valid_b])) if valid_b.any() else np.nan
            row_tvt_formula = -tail_z + row_tail_ancc + row_anchor_b
            row_prefix_formula = -prefix_z + row_prefix_ancc + row_anchor_b
            prefix_valid_formula = valid_b & np.isfinite(row_prefix_formula)
            row_prefix_resid = prefix_tvt_values[prefix_valid_formula] - row_prefix_formula[prefix_valid_formula]
            out['formation_row_ancc'] = row_tail_ancc
            out['formation_row_ancc_std'] = row_tail_std
            out['formation_row_min_dist'] = row_dist
            out['formation_row_anchor_b'] = row_anchor_b
            out['formation_row_prefix_rmse'] = float(np.sqrt(np.mean(row_prefix_resid ** 2))) if len(row_prefix_resid) else np.nan
            out['formation_row_prefix_mae'] = float(np.mean(np.abs(row_prefix_resid))) if len(row_prefix_resid) else np.nan
            out['formation_row_tvt_formula'] = row_tvt_formula
            out['formation_row_delta_formula'] = row_tvt_formula - last_known_tvt
            row_delta = np.asarray(out['formation_row_delta_formula'], dtype=float)
            out['formation_row_delta_from_plane'] = row_delta - plane_delta
            stacked_delta = np.vstack([plane_delta, row_delta])
            finite_count = np.isfinite(stacked_delta).sum(axis=0)
            summed_delta = np.nansum(stacked_delta, axis=0)
            out['formation_formula_mean_delta'] = np.divide(
                summed_delta,
                finite_count,
                out=np.full(len(tail), np.nan, dtype=float),
                where=finite_count > 0,
            )
            out['formation_formula_abs_gap'] = np.abs(row_delta - plane_delta)
    
    # Typewell features are available reference-well features. They are evaluated only at prefix-derived TVT baselines.
    tw_path = split_dir / f'{well_id}__typewell.csv'
    if need_typewell_features and tw_path.exists():
        tw = pd.read_csv(tw_path)
    else:
        tw = pd.DataFrame(columns=['TVT', 'GR'])
    if need_typewell_features and {'TVT', 'GR'}.issubset(tw.columns) and tw[['TVT', 'GR']].dropna().shape[0] >= 2:
        tw_tvt = pd.to_numeric(tw['TVT'], errors='coerce')
        tw_gr = pd.to_numeric(tw['GR'], errors='coerce')
        tw_last_known_gr = typewell_gr_at_tvt(tw, np.array([last_known_tvt]))[0]
        tw_slope_all_gr = typewell_gr_at_tvt(tw, slope_baseline_tvt_all) if need_typewell_slope_features else np.full(len(tail), np.nan)
        tw_slope_last200_gr = typewell_gr_at_tvt(tw, slope_baseline_tvt_last200) if (need_typewell_slope_features or need_local_typewell or need_local_calibrated_typewell or need_calibrated_typewell) else np.full(len(tail), np.nan)
        prefix_tw_gr = typewell_gr_at_tvt(tw, prefix_tvt.to_numpy(dtype=float))
        prefix_gr_values = prefix_gr.to_numpy(dtype=float)
        valid_prefix_align = np.isfinite(prefix_gr_values) & np.isfinite(prefix_tw_gr)
        prefix_align_residual = prefix_gr_values - prefix_tw_gr
        prefix_align_absdiff = (
            float(np.mean(np.abs(prefix_align_residual[valid_prefix_align])))
            if valid_prefix_align.any()
            else np.nan
        )
        prefix_align_rmse = (
            float(np.sqrt(np.mean(prefix_align_residual[valid_prefix_align] ** 2)))
            if valid_prefix_align.any()
            else np.nan
        )
        calib_a = calib_b = np.nan
        prefix_calib_mae = prefix_calib_rmse = np.nan
        tw_last_known_gr_calibrated = np.nan
        tw_slope_all_gr_calibrated = np.full(len(tail), np.nan)
        tw_slope_last200_gr_calibrated = np.full(len(tail), np.nan)
        if need_calibrated_typewell or need_local_calibrated_typewell or need_calibrated_anchor_offsets:
            calib_a, calib_b = safe_affine_fit(prefix_tw_gr, prefix_gr_values)
            prefix_tw_gr_calibrated = apply_affine(prefix_tw_gr, calib_a, calib_b)
            tw_last_known_gr_calibrated = apply_affine(np.array([tw_last_known_gr]), calib_a, calib_b)[0]
            tw_slope_all_gr_calibrated = apply_affine(tw_slope_all_gr, calib_a, calib_b)
            tw_slope_last200_gr_calibrated = apply_affine(tw_slope_last200_gr, calib_a, calib_b)
            valid_prefix_calib = np.isfinite(prefix_gr_values) & np.isfinite(prefix_tw_gr_calibrated)
            prefix_calib_residual = prefix_gr_values - prefix_tw_gr_calibrated
            prefix_calib_mae = (
                float(np.mean(np.abs(prefix_calib_residual[valid_prefix_calib])))
                if valid_prefix_calib.any()
                else np.nan
            )
            prefix_calib_rmse = (
                float(np.sqrt(np.mean(prefix_calib_residual[valid_prefix_calib] ** 2)))
                if valid_prefix_calib.any()
                else np.nan
            )
        local_offsets = np.arange(-60.0, 60.1, 5.0)
        local_raw = {}
        if need_local_typewell:
            local_raw = local_typewell_offset_features(
                tw,
                slope_baseline_tvt_last200,
                tail_gr,
                local_offsets,
                prefix='typewell_local_last200',
            )
        local_calibrated = {}
        if need_local_calibrated_typewell:
            local_calibrated = local_typewell_offset_features(
                tw,
                slope_baseline_tvt_last200,
                tail_gr,
                local_offsets,
                affine_a=calib_a,
                affine_b=calib_b,
                prefix='calibrated_typewell_local_last200',
            )
        out['typewell_tvt_min'] = safe_stat(tw_tvt, np.min)
        out['typewell_tvt_max'] = safe_stat(tw_tvt, np.max)
        out['typewell_tvt_range'] = safe_stat(tw_tvt, np.max) - safe_stat(tw_tvt, np.min)
        out['typewell_gr_mean'] = safe_stat(tw_gr, np.mean)
        out['typewell_gr_std'] = safe_stat(tw_gr, np.std)
        out['typewell_gr_at_last_known_tvt'] = tw_last_known_gr
        out['typewell_gr_at_slope_baseline_all'] = tw_slope_all_gr
        out['typewell_gr_at_slope_baseline_last200'] = tw_slope_last200_gr
        out['gr_minus_typewell_last_known_tvt'] = tail_gr - tw_last_known_gr
        out['gr_minus_typewell_slope_baseline_all'] = tail_gr - tw_slope_all_gr
        out['gr_minus_typewell_slope_baseline_last200'] = tail_gr - tw_slope_last200_gr
        out['prefix_horizontal_vs_typewell_gr_corr'] = safe_corr(prefix_gr_values, prefix_tw_gr)
        out['prefix_horizontal_vs_typewell_gr_mae'] = prefix_align_absdiff
        out['prefix_horizontal_vs_typewell_gr_rmse'] = prefix_align_rmse
        if need_typewell_interval_context:
            last_geo_context = typewell_interval_context_features(tw, np.array([last_known_tvt]), prefix='typewell_last_geo')
            for col, values in last_geo_context.items():
                out[col] = values[0]
            baseline_geo_context = typewell_interval_context_features(tw, slope_baseline_tvt_last200, prefix='typewell_baseline_last200_geo')
            for col, values in baseline_geo_context.items():
                out[col] = values
        anchor_offsets = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], dtype=float)
        if need_typewell_anchor_offsets:
            for anchor_offset in anchor_offsets:
                label = candidate_endpoint_label(float(anchor_offset))
                anchor_gr = typewell_gr_at_tvt(tw, np.array([last_known_tvt + float(anchor_offset)]))[0]
                out[f'typewell_anchor_gr_diff_{label}'] = tail_gr - anchor_gr
        if need_local_typewell:
            for col, values in local_raw.items():
                out[col] = values
            out['typewell_local_last200_gr_resid_best'] = tail_gr - out['typewell_local_last200_best_gr']
            out['typewell_local_last200_best_vs_zero_gain'] = out['typewell_local_last200_zero_abs_resid'] - out['typewell_local_last200_best_abs_resid']
        out['prefix_typewell_gr_affine_a'] = calib_a
        out['prefix_typewell_gr_affine_b'] = calib_b
        out['prefix_horizontal_vs_calibrated_typewell_gr_mae'] = prefix_calib_mae
        out['prefix_horizontal_vs_calibrated_typewell_gr_rmse'] = prefix_calib_rmse
        if need_calibrated_anchor_offsets:
            for anchor_offset in anchor_offsets:
                label = candidate_endpoint_label(float(anchor_offset))
                anchor_gr = typewell_gr_at_tvt(tw, np.array([last_known_tvt + float(anchor_offset)]))[0]
                anchor_gr_calibrated = apply_affine(np.array([anchor_gr]), calib_a, calib_b)[0]
                out[f'calibrated_typewell_anchor_gr_diff_{label}'] = tail_gr - anchor_gr_calibrated
        out['typewell_calibrated_gr_at_last_known_tvt'] = tw_last_known_gr_calibrated
        out['typewell_calibrated_gr_at_slope_baseline_all'] = tw_slope_all_gr_calibrated
        out['typewell_calibrated_gr_at_slope_baseline_last200'] = tw_slope_last200_gr_calibrated
        out['gr_minus_calibrated_typewell_last_known_tvt'] = tail_gr - tw_last_known_gr_calibrated
        out['gr_minus_calibrated_typewell_slope_baseline_all'] = tail_gr - tw_slope_all_gr_calibrated
        out['gr_minus_calibrated_typewell_slope_baseline_last200'] = tail_gr - tw_slope_last200_gr_calibrated
        out['calibrated_typewell_slope_last200_gr_prefix_z'] = (tw_slope_last200_gr_calibrated - prefix_gr_mean) / (prefix_gr_std + 1e-6) if np.isfinite(prefix_gr_std) else np.nan
        if need_local_calibrated_typewell:
            for col, values in local_calibrated.items():
                out[col] = values
            out['calibrated_typewell_local_last200_gr_resid_best'] = tail_gr - out['calibrated_typewell_local_last200_best_gr']
            out['calibrated_typewell_local_last200_best_vs_zero_gain'] = out['calibrated_typewell_local_last200_zero_abs_resid'] - out['calibrated_typewell_local_last200_best_abs_resid']
        if need_candidate_path:
            candidate_path_features = typewell_candidate_path_features(
                tw,
                last_known_tvt,
                tail_frac,
                tail_gr,
                endpoints=CANDIDATE_PATH_ENDPOINTS,
                prefix='tw_path',
            )
            candidate_path_ease_features = typewell_candidate_path_features(
                tw,
                last_known_tvt,
                np.power(np.clip(tail_frac, 0.0, None), 1.45),
                tail_gr,
                endpoints=CANDIDATE_PATH_ENDPOINTS,
                prefix='tw_path_ease',
            )
            for col, values in candidate_path_features.items():
                out[col] = values
            for col, values in candidate_path_ease_features.items():
                out[col] = values
        if need_beam:
            if gr_full_filled is None:
                fallback_gr = prefix_gr_mean if np.isfinite(prefix_gr_mean) else 0.0
                gr_full_filled = gr_full_numeric.interpolate(limit_direction='both').fillna(fallback_gr)
            hidden_gr_for_beam = gr_full_filled.iloc[pred_idx].to_numpy(dtype=float)
            tw_tvt_values = tw_tvt.to_numpy(dtype=float)
            tw_gr_values = tw_gr.to_numpy(dtype=float)
            beam_configs = {
                'tight': dict(beam_size=5, move_cost=50.0, emit_scale=200.0, radius=1),
                'conservative': dict(beam_size=10, move_cost=20.0, emit_scale=144.0, radius=2),
                'loose': dict(beam_size=15, move_cost=8.0, emit_scale=64.0, radius=2),
                'vloose': dict(beam_size=20, move_cost=3.0, emit_scale=25.0, radius=3),
            }
            beam_paths = {
                name: beam_typewell_path(
                    hidden_gr_for_beam,
                    tw_tvt_values,
                    tw_gr_values,
                    last_known_tvt,
                    **config,
                )
                for name, config in beam_configs.items()
            }
            beam_tight = beam_paths['tight']
            beam_cons = beam_paths['conservative']
            beam_loose = beam_paths['loose']
            beam_vloose = beam_paths['vloose']
            beam_deltas = np.vstack([
                beam_tight - last_known_tvt,
                beam_cons - last_known_tvt,
                beam_loose - last_known_tvt,
                beam_vloose - last_known_tvt,
            ])
            out['tw_beam_tight_delta'] = beam_tight - last_known_tvt
            out['tw_beam_conservative_delta'] = beam_cons - last_known_tvt
            out['tw_beam_loose_delta'] = beam_loose - last_known_tvt
            out['tw_beam_vloose_delta'] = beam_vloose - last_known_tvt
            out['tw_beam_gap'] = beam_loose - beam_cons
            out['tw_beam_spread'] = np.nanmax(beam_deltas, axis=0) - np.nanmin(beam_deltas, axis=0)
            out['tw_beam_mean_delta'] = np.nanmean(beam_deltas, axis=0)
            out['tw_beam_std_delta'] = np.nanstd(beam_deltas, axis=0)
            out['tw_beam_tight_step'] = np.r_[np.nan, np.diff(beam_tight)]
            out['tw_beam_conservative_step'] = np.r_[np.nan, np.diff(beam_cons)]
            out['tw_beam_loose_step'] = np.r_[np.nan, np.diff(beam_loose)]
            out['tw_beam_vloose_step'] = np.r_[np.nan, np.diff(beam_vloose)]
            out['tw_beam_gr_at_conservative'] = np.interp(beam_cons, tw_tvt_values, tw_gr_values, left=np.nan, right=np.nan)
            out['tw_beam_gr_at_loose'] = np.interp(beam_loose, tw_tvt_values, tw_gr_values, left=np.nan, right=np.nan)
            out['tw_beam_gr_minus_conservative'] = tail_gr - out['tw_beam_gr_at_conservative']
            out['tw_beam_gr_minus_loose'] = tail_gr - out['tw_beam_gr_at_loose']
            out['beam_tight_delta'] = out['tw_beam_tight_delta']
            out['beam_cons_delta'] = out['tw_beam_conservative_delta']
            out['beam_loose_delta'] = out['tw_beam_loose_delta']
            out['beam_vloose_delta'] = out['tw_beam_vloose_delta']
            out['beam_mean_delta'] = out['tw_beam_mean_delta']
            out['beam_std_delta'] = out['tw_beam_std_delta']
            out['beam_spread'] = out['tw_beam_spread']
            out['beam_gap'] = out['tw_beam_gap']
            out['gr_minus_tw_beam_cons'] = out['tw_beam_gr_minus_conservative']
            out['gr_minus_tw_beam_loose'] = out['tw_beam_gr_minus_loose']
    else:
        for col in [
            'typewell_tvt_min',
            'typewell_tvt_max',
            'typewell_tvt_range',
            'typewell_gr_mean',
            'typewell_gr_std',
            'typewell_gr_at_last_known_tvt',
            'typewell_gr_at_slope_baseline_all',
            'typewell_gr_at_slope_baseline_last200',
            'gr_minus_typewell_last_known_tvt',
            'gr_minus_typewell_slope_baseline_all',
            'gr_minus_typewell_slope_baseline_last200',
            'prefix_horizontal_vs_typewell_gr_corr',
            'prefix_horizontal_vs_typewell_gr_mae',
            'prefix_horizontal_vs_typewell_gr_rmse',
            'typewell_local_last200_best_delta',
            'typewell_local_last200_best_abs_resid',
            'typewell_local_last200_zero_abs_resid',
            'typewell_local_last200_top2_gap',
            'typewell_local_last200_soft_delta_mean',
            'typewell_local_last200_best_gr',
            'typewell_local_last200_gr_resid_best',
            'typewell_local_last200_best_vs_zero_gain',
            'prefix_typewell_gr_affine_a',
            'prefix_typewell_gr_affine_b',
            'prefix_horizontal_vs_calibrated_typewell_gr_mae',
    'prefix_horizontal_vs_typewell_gr_rmse',
    'prefix_tvt_step20',
    'typewell_last_geo_interval_phase',
    'typewell_baseline_last200_geo_boundary_proximity',
            'prefix_horizontal_vs_calibrated_typewell_gr_rmse',
            'typewell_calibrated_gr_at_last_known_tvt',
            'typewell_calibrated_gr_at_slope_baseline_all',
            'typewell_calibrated_gr_at_slope_baseline_last200',
            'gr_minus_calibrated_typewell_last_known_tvt',
            'gr_minus_calibrated_typewell_slope_baseline_all',
            'gr_minus_calibrated_typewell_slope_baseline_last200',
            'calibrated_typewell_slope_last200_gr_prefix_z',
            'calibrated_typewell_local_last200_best_delta',
            'calibrated_typewell_local_last200_best_abs_resid',
            'calibrated_typewell_local_last200_zero_abs_resid',
            'calibrated_typewell_local_last200_top2_gap',
            'calibrated_typewell_local_last200_soft_delta_mean',
            'calibrated_typewell_local_last200_best_gr',
            'calibrated_typewell_local_last200_gr_resid_best',
            'calibrated_typewell_local_last200_best_vs_zero_gain',
        ]:
            out[col] = np.nan
        for col in candidate_path_feature_names(prefix='tw_path'):
            out[col] = np.nan
        for col in candidate_path_feature_names(prefix='tw_path_ease'):
            out[col] = np.nan
        for col in typewell_interval_context_feature_names(prefix='typewell_last_geo'):
            out[col] = np.nan
        for col in typewell_interval_context_feature_names(prefix='typewell_baseline_last200_geo'):
            out[col] = np.nan
        for offset in [-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80]:
            label = candidate_endpoint_label(float(offset))
            out[f'typewell_anchor_gr_diff_{label}'] = np.nan
            out[f'calibrated_typewell_anchor_gr_diff_{label}'] = np.nan
        if need_beam:
            for col in offline_beam_feature_names(prefix='tw_beam'):
                out[col] = np.nan
    
    if include_target and 'TVT' in df.columns:
        out['target_tvt'] = tail['TVT'].to_numpy()
        out['target_delta_from_last_known'] = out['target_tvt'] - last_known_tvt
    return out

preview_well_id = globals().get('representative_well_id')
if preview_well_id is None and globals().get('train_h_ids'):
    preview_well_id = sorted(train_h_ids)[0]

if preview_well_id is not None:
    representative_feature_frame = make_tail_features_for_well(preview_well_id, TRAIN_DIR, include_target=True, use_beam_features=False)
    print('representative_feature_frame shape:', representative_feature_frame.shape)
    display(representative_feature_frame.head())
    preview_columns = [
        'target_delta_from_last_known',
        'md_since_ps',
        'prefix_azimuth_deg',
        'prefix_horizontal_vs_typewell_gr_corr',
        'prefix_horizontal_vs_calibrated_typewell_gr_mae',
        'gr_minus_calibrated_typewell_slope_baseline_last200',
        'typewell_local_last200_best_delta',
        'typewell_local_last200_best_vs_zero_gain',
        'tw_path_min_absdiff',
        'tw_path_best_endpoint',
        'tw_path_top2_absdiff_gap',
        'tw_path_ease_min_absdiff',
        'gap_gr_std',
        'sin_tail_frac_pi',
        'gr_center_roll_mean_21',
        'gr_center_lead1',
        'gr_cumsum_since_ps',
        'calibrated_typewell_local_last200_best_delta',
        'calibrated_typewell_local_last200_best_vs_zero_gain',
        'GR_prefix_z',
        'gr_diff_1',
        'gr_slope_md_1',
        'gr_roll_range_25',
        'rows_since_prefix_last_valid_gr',
        'md_since_prefix_last_valid_gr',
        'trajectory_step_1',
        'z_slope_md_1',
        'GR_isna',
    ]
    preview_columns = [col for col in preview_columns if col in representative_feature_frame.columns]
    display(representative_feature_frame[preview_columns].describe().T)
else:
    print('Representative feature preview skipped because no training wells are available.')


## 12. Curve-Level Target Diagnostics

### Statistical role

Summarize each tail as a fixed number of curve knots for target-shape analysis.

### Why this matters

If TVT curves are smooth, a knot model can reduce row-wise noise and produce more stable long-tail shapes. The labels are allowed for training, but their future-tail construction must not leak into inference features.

<details>
<summary>Knot target definition</summary>

Let $q_j\in[0,1]$ be a tail-fraction grid. The knot target is:

$$
\Delta y_w(q_j)=y_w(q_j)-y_{w,\mathrm{PS}-1}
$$

At prediction time, row-level predictions are reconstructed by interpolation:

$$
\hat{y}_{w,i}=y_{w,\mathrm{PS}-1}+\mathrm{Interp}\left(q_i; \widehat{\Delta y}_w(q_1),\ldots,\widehat{\Delta y}_w(q_m)\right)
$$

</details>

### Leakage boundary

The knot targets in this section use the true future tail TVT and full tail length. They are supervised labels for curve-model training or target-shape diagnostics, not row-level input features. A drilling-time compatible curve model must predict these knots from prefix/current-row-safe features only.


In [ ]:
# Convert each tail TVT curve into fixed-fraction knot delta targets.

def make_knot_targets(path: Path, knots=np.linspace(0, 1, 21)) -> dict:
    wid = well_id_from_path(path)
    df = pd.read_csv(path, usecols=['TVT', 'TVT_input'])
    mask = df['TVT_input'].isna().to_numpy()
    pred_idx = np.flatnonzero(mask)
    if len(pred_idx) == 0:
        return {'well_id': wid}
    last_known = float(df['TVT_input'].iloc[pred_idx[0] - 1])
    tail_y = df['TVT'].iloc[pred_idx].to_numpy()
    x = np.linspace(0, 1, len(tail_y))
    knot_delta = np.interp(knots, x, tail_y - last_known)
    row = {'well_id': wid, 'tail_len': len(tail_y), 'last_known_TVT': last_known}
    for k, value in zip(knots, knot_delta):
        row[f'delta_knot_{k:.2f}'] = float(value)
    row['tail_min_delta'] = float(np.min(tail_y - last_known))
    row['tail_max_delta'] = float(np.max(tail_y - last_known))
    row['tail_end_delta'] = float(tail_y[-1] - last_known)
    return row

knot_targets = pd.DataFrame([make_knot_targets(path) for path in train_horizontal_files])
print('knot_targets shape:', knot_targets.shape)
display(knot_targets.head())

selected_knot_cols = ['delta_knot_0.00', 'delta_knot_0.25', 'delta_knot_0.50', 'delta_knot_0.75', 'delta_knot_1.00', 'tail_min_delta', 'tail_max_delta']
display(knot_targets[selected_knot_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

fig, ax = plt.subplots(figsize=(10, 5))
for _, row in knot_targets.sample(min(80, len(knot_targets)), random_state=42).iterrows():
    values = [row[f'delta_knot_{k:.2f}'] for k in np.linspace(0, 1, 21)]
    ax.plot(np.linspace(0, 1, 21), values, alpha=0.18, color='tab:blue')
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Sample tail delta curves from last known TVT')
ax.set_xlabel('Tail fraction')
ax.set_ylabel('TVT delta from last known')
plt.tight_layout()
plt.show()

## 13. Nearby-Well Spatial Signal

### Statistical role

Diagnose whether spatial drift in nearby wells is related to the target well drift.

### Leakage caution

This diagnostic uses target-derived drift only to measure whether spatial proximity has signal. In real CV features, target-derived drift from validation-fold wells must not enter the neighbor reference table.

<details>
<summary>KNN drift diagnostic</summary>

Distance between Prediction Start coordinates:

$$
d(w,u)=\sqrt{(X_{PS,w}-X_{PS,u})^2+(Y_{PS,w}-Y_{PS,u})^2}
$$

K-nearest-neighbor prior:

$$
\widehat{\Delta}^{NN}_w=\frac{1}{K}\sum_{u\in NN_K(w)}\Delta^{\mathrm{end}}_u
$$

</details>


In [ ]:
# Estimate leave-one-out nearest-neighbor spatial drift signal.

spatial_frame = h_summary[['well_id', 'ps_x', 'ps_y', 'azimuth_deg', 'tail_end_delta_from_last_known', 'tail_tvt_range', 'constant_tail_rmse']].dropna().reset_index(drop=True)
coords = spatial_frame[['ps_x', 'ps_y']].to_numpy()
# Pairwise distance is small enough: 773 x 773.
dx = coords[:, None, 0] - coords[None, :, 0]
dy = coords[:, None, 1] - coords[None, :, 1]
dist = np.sqrt(dx ** 2 + dy ** 2)
np.fill_diagonal(dist, np.inf)

for k in [3, 5, 10, 20]:
    nn = np.argsort(dist, axis=1)[:, :k]
    neighbor_mean = spatial_frame['tail_end_delta_from_last_known'].to_numpy()[nn].mean(axis=1)
    spatial_frame[f'nn{k}_tail_end_delta_mean'] = neighbor_mean
    corr = np.corrcoef(spatial_frame['tail_end_delta_from_last_known'], neighbor_mean)[0, 1]
    print(f'leave-one-out nearest {k} mean vs own tail_end_delta correlation: {corr:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=spatial_frame, x='nn5_tail_end_delta_mean', y='tail_end_delta_from_last_known', hue='constant_tail_rmse', palette='viridis', ax=axes[0])
axes[0].axline((0, 0), slope=1, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Nearest-neighbor tail-end drift signal')
sns.scatterplot(data=spatial_frame, x='ps_x', y='ps_y', hue='tail_end_delta_from_last_known', palette='coolwarm', ax=axes[1], s=35)
axes[1].set_title('Spatial distribution of tail-end drift')
axes[1].set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()

## 14. Representative Well Plot

### Statistical role

Inspect one well visually to catch curve behavior that aggregate statistics can hide.

### What to inspect

- TVT continuity before and after Prediction Start
- GR missingness locations
- TVT-axis similarity between typewell GR and horizontal known-prefix GR
- Whether the tail is flat, increasing, or decreasing

### Why this matters

Aggregate RMSE can hide specific well-level failure modes. Plots are useful for building a failure taxonomy.


In [ ]:
# Plot a representative well across TVT, GR, map trajectory, and typewell-aligned GR axes.

def plot_well_overview(well_id: str):
    h = pd.read_csv(TRAIN_DIR / f'{well_id}__horizontal_well.csv')
    tw = pd.read_csv(TRAIN_DIR / f'{well_id}__typewell.csv')
    mask = h['TVT_input'].isna().to_numpy()
    ps = int(np.flatnonzero(mask)[0]) if mask.any() else len(h)
    last_known_tvt = h['TVT_input'].iloc[ps - 1]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    ax = axes[0, 0]
    ax.plot(h.index, h['TVT'], label='TVT target', color='tab:blue')
    ax.plot(h.index, h['TVT_input'], label='TVT_input known prefix', color='tab:orange')
    ax.axvline(ps, color='red', linestyle='--', label='Prediction Start')
    ax.set_title(f'{well_id}: TVT and Prediction Start')
    ax.set_xlabel('row index')
    ax.set_ylabel('TVT')
    ax.legend()
    
    ax = axes[0, 1]
    ax.plot(h.index, h['GR'], color='black', linewidth=1)
    ax.axvline(ps, color='red', linestyle='--')
    ax.set_title('Horizontal GR along row index')
    ax.set_xlabel('row index')
    ax.set_ylabel('GR')
    
    ax = axes[1, 0]
    ax.plot(h['X'], h['Y'], color='tab:green')
    ax.scatter(h['X'].iloc[ps], h['Y'].iloc[ps], color='red', s=80, label='PS')
    ax.set_title('Map view of horizontal trajectory')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_aspect('equal', adjustable='box')
    ax.legend()
    
    ax = axes[1, 1]
    known = h['TVT_input'].notna() & h['GR'].notna()
    ax.plot(tw['GR'], tw['TVT'], color='red', alpha=0.8, label='Typewell GR')
    ax.scatter(h.loc[known, 'GR'], h.loc[known, 'TVT_input'], s=5, alpha=0.35, color='black', label='Horizontal known prefix GR')
    ax.axhline(last_known_tvt, color='blue', linestyle='--', label='last known TVT')
    ax.invert_yaxis()
    ax.set_title('GR on TVT axis')
    ax.set_xlabel('GR')
    ax.set_ylabel('TVT')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

plot_well_overview(representative_well_id)

## 🧠 15. Model Logic from EDA

### 🧩 Prediction architecture

```text
last_known_TVT anchor
        +
residual model from leakage-checked features
        +
OOF-selected post-processing
        ↓
row-level TVT prediction
```

### 📏 Validation rules

| Rule | Reason |
|---|---|
| `GroupKFold` by `well_id` | estimates unseen-well generalization |
| Evaluate only `TVT_input.isna()` rows | matches the submission target |
| Row-level RMSE | matches the competition metric |
| Row-weighted CV summary | long tails should carry more metric weight |
| Global postprocess selection | avoids fold-specific oracle tuning |

<details>
<summary>Row-weighted CV aggregation</summary>

$$
\mathrm{RMSE}_{CV}
=
\sqrt{\frac{\sum_f n_f\,\mathrm{RMSE}_f^2}{\sum_f n_f}}.
$$

</details>

### 🧬 Signal families

| Signal | Status | Construction |
|---|---:|---|
| Residual target | ✅ | predict `TVT - last_known_TVT`, not raw TVT |
| Typewell GR alignment | ✅ | interpolate reference GR at prefix-derived TVT baselines |
| GR affine calibration | ✅ | prefix-only fit between horizontal GR and typewell GR |
| GR derivative/event features | ✅ | backward differences and trailing ranges |
| Local trajectory steps | ✅ | current and previous survey rows |
| Prefix TVT context | ✅ | prefix-only statistics and slopes |
| Offline tail-position/path features | 🌐 | target-free full-tail covariates |
| Formation-plane/KNN references | 🌐 | train surfaces as auxiliary labels, fold-safe in CV |
| Beam-path alignment | 🧪 | full hidden GR sequence, no future TVT labels |
| Fade-in residual | ✅ | suppresses unsupported early-tail jumps |
| Anchor-aware slope clipping | ✅ | anchor plus previous clipped prediction |
| Centered GR rolling | 🌐 | future GR covariates |
| Direct train-only surfaces | 🚫 | hidden-test feature mismatch |

### 🎯 Statistical objective

| Goal | Practical consequence |
|---|---|
| protect flat wells | shrink residuals and fade in near Prediction Start |
| correct drifting wells | use GR/typewell, trajectory, and formation-plane evidence |
| avoid nonphysical jumps | apply anchor-aware slope limiting when selected |


## 🧮 16. Residual Prediction Model

### 🔁 Residual pipeline

| Stage | Input | Output | Purpose |
|---|---|---|---|
| Anchor | `last_known_TVT` | flat baseline | strong empirical prior |
| Model | feature row | raw residual | learn non-zero drift |
| Residual clip | raw residual | bounded residual | remove extreme model outliers |
| Global alpha | bounded residual | shrunk residual | match OOF residual variance |
| Fade-in | shrunk residual | early-tail guarded residual | keep first tail rows near anchor |
| Slope limiter | TVT curve | clipped TVT curve | suppress implausible jumps |

<details>
<summary>Compact prediction equation</summary>

$$
\hat{y}_{w,i}
=
y_{w,L}
+
\rho_{w,i}(\tau)\,\alpha\,\mathrm{clip}(\widehat{\Delta y}_{w,i}).
$$

The fade-in factor is:

$$
\rho_{w,i}(\tau)
=
1-
\exp\left(-\frac{\max(MD_{w,i}-MD_{w,PS},0)}{\tau}\right).
$$

When no fade-in is selected, $\rho_{w,i}=1$.

</details>

### 🧠 Encoded domain structure

| Feature family | Signal |
|---|---|
| Prefix context | known TVT range, statistics, slopes |
| Typewell alignment | typewell GR at prefix-derived TVT baselines |
| Calibrated typewell | prefix-only affine GR calibration |
| Causal events | backward GR derivatives and trailing GR ranges |
| Trajectory steps | local geometric movement |
| Offline tail covariates | target-free batch-prediction context |
| Formation-plane references | spatial formation-top geometry projected onto hidden rows |

### 🧯 Anchor-aware causal clipping

| Row | Reference value | Constraint |
|---|---|---|
| first tail row | last known prefix TVT | no unsupported jump from anchor |
| later tail rows | previous clipped prediction | smooth row-to-row movement |

The slope bound is estimated from train-fold TVT slope quantiles.


### 16.0 Feature Leakage Review

### Strict policy inputs

| Feature source | Why it is allowed |
|---|---|
| `MD`, `X`, `Y`, `Z`, `GR` at row $i$ | observed at the row |
| Prefix aggregates | known before tail prediction |
| `last_known_TVT` and prefix TVT statistics | known from prefix `TVT_input` |
| Backward GR differences | previous/current rows only |
| Trailing GR rolling features | rows up to $i$ only |
| Typewell GR at prefix-derived baselines | reference log + prefix baseline |
| Prefix-only affine GR calibration | fit only where `TVT_input` is present |
| Anchor-aware slope clipping | anchor, train-fold slope bound, previous clipped prediction |

### Offline policy inputs

| Feature source | Offline reason |
|---|---|
| Tail length and tail fraction | prediction rows are known before writing predictions |
| Full-row fraction and MD tail fraction | test trajectory is provided |
| Gap geometry and full-tail GR summaries | target-free covariates from the provided file |
| Centered GR rolling and lead/lag GR | future GR covariates, not future TVT labels |
| Candidate-path typewell features | tail fraction defines candidate paths, but no true tail TVT is used |
| Spatial formation-plane features | train-only surfaces are auxiliary labels for a reference imputer; validation folds use only training wells |
| Beam-path typewell features | full hidden GR sequence is used; true tail TVT is not used |

### Excluded from all feature policies

| Excluded item | Reason |
|---|---|
| `TVT` in the prediction tail | direct target leakage |
| Train-only surfaces (`ANCC` ... `BUDA`) | hidden-test feature mismatch |
| `TVT_input_bfill` | reads rows after the prediction row |
| Target-derived tail summaries | answer-key information |
| Nearby target drift from validation wells | fold leakage |


In [ ]:
# Configure the residual model, feature sets, validation, and causal post-processing.

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# Execution controls.
NOTEBOOK_RUN_VERSION = 'ROGII_EDA_v2_candidate_path_lgbm_2026_05_06'
# Local research runs keep grouped CV enabled. Kaggle submission runs skip CV by default
# and use the latest stored CV selection before fitting the all-row GPU LightGBM candidate.
QUICK_CHECK_MODE = False
KAGGLE_MEMORY_SAFE_MODE = True
ENABLE_KAGGLE_BEAM_FEATURES = False
RUN_HGB_DIAGNOSTIC_SUBMISSIONS = not KAGGLE_NOTEBOOK_RUN
RUN_GROUPED_CV = not KAGGLE_NOTEBOOK_RUN

MODEL_CONFIG = {
    # Resource-controlled settings for grouped CV and final training.
    'random_state': 42,
    'n_splits': 5,
    'cv_folds_to_run': 5,
    'train_rows_per_well': 350,
    'final_train_rows_per_well': 700,
    'feature_policy_for_selection': 'strict',
    'residual_shrinkage': 0.80,
    'alpha_bounds': (0.0, 1.5),
    'delta_clip_quantiles': (0.005, 0.995),
    'fade_in_tau_md_to_compare': [None, 25.0, 50.0, 100.0, 200.0],
    'slope_clip_quantiles_to_compare': [0.90, 0.95, 0.975, 0.99, 0.995],
    'postprocess_slope_quantile': 0.995,
    'feature_sets_to_compare': None,
    'select_best_feature_set_from_cv': True,
    'apply_slope_clip_if_cv_improves': True,
    'write_submission': True,
    'write_public_clean_submission': True,
}

if QUICK_CHECK_MODE:
    MODEL_CONFIG.update({
        'cv_folds_to_run': 1,
        'train_rows_per_well': 150,
        'final_train_rows_per_well': 150,
        'feature_sets_to_compare': [
            'calibrated_typewell_alignment',
            'offline_candidate_path_alignment',
        ],
    })

BASE_FEATURE_COLUMNS = [
    # Row position and trajectory after Prediction Start.
    'md_since_ps',
    'x_delta_ps',
    'y_delta_ps',
    'z_delta_ps',
    'xy_dist_ps',
    'MD',
    'X',
    'Y',
    'Z',
    # Prefix-only well context.
    'prefix_len',
    'prefix_azimuth_deg',
    'prefix_gr_missing_rate',
    # Row-level GR signal and trailing GR context.
    'GR',
    'GR_isna',
    'GR_prefix_z',
    'gr_diff_1',
    'gr_diff_5',
    'gr_slope_md_1',
    'md_step_1',
    'x_step_1',
    'y_step_1',
    'z_step_1',
    'trajectory_step_1',
    'z_slope_md_1',
    'prefix_gr_mean',
    'prefix_gr_std',
    'gr_roll_mean_25',
    'gr_roll_std_25',
    'gr_roll_min_25',
    'gr_roll_max_25',
    'gr_roll_range_25',
    'gr_roll_mean_100',
    'gr_roll_std_100',
    'gr_roll_range_100',
    'gr_roll_mean_300',
    'gr_roll_std_300',
    'gr_roll_range_300',
    # Known-prefix TVT trend. This can help, but shrinkage/clipping protects against blind extrapolation.
    'prefix_tvt_slope_md_all',
    'prefix_tvt_slope_md_last200',
]

PREFIX_CONTEXT_COLUMNS = [
    'last_known_TVT',
    'prefix_gr_min',
    'prefix_gr_max',
    'prefix_last_valid_gr',
    'rows_since_prefix_last_valid_gr',
    'md_since_prefix_last_valid_gr',
    'gr_minus_prefix_last_valid_gr',
    'gr_minus_prefix_gr_mean',
    'prefix_tvt_min',
    'prefix_tvt_max',
    'prefix_tvt_range',
    'prefix_tvt_mean',
    'prefix_tvt_std',
    'prefix_tvt_step20',
    'prefix_tvt_step100',
    'prefix_tvt_md_slope100',
    'prefix_tvt_z_slope100',
    'slope_baseline_delta_all',
    'slope_baseline_delta_last200',
    'slope_baseline_tvt_all',
    'slope_baseline_tvt_last200',
]

TYPEWELL_INTERVAL_CONTEXT_COLUMNS = (
    typewell_interval_context_feature_names(prefix='typewell_last_geo')
    + typewell_interval_context_feature_names(prefix='typewell_baseline_last200_geo')
)

TYPEWELL_ANCHOR_OFFSET_COLUMNS = [
    f'typewell_anchor_gr_diff_{candidate_endpoint_label(float(offset))}'
    for offset in [-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80]
]

CALIBRATED_TYPEWELL_ANCHOR_OFFSET_COLUMNS = [
    f'calibrated_typewell_anchor_gr_diff_{candidate_endpoint_label(float(offset))}'
    for offset in [-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80]
]

TYPEWELL_ALIGNMENT_COLUMNS = [
    'typewell_tvt_min',
    'typewell_tvt_max',
    'typewell_tvt_range',
    'typewell_gr_mean',
    'typewell_gr_std',
    'typewell_gr_at_last_known_tvt',
    'typewell_gr_at_slope_baseline_all',
    'typewell_gr_at_slope_baseline_last200',
    'gr_minus_typewell_last_known_tvt',
    'gr_minus_typewell_slope_baseline_all',
    'gr_minus_typewell_slope_baseline_last200',
    'prefix_horizontal_vs_typewell_gr_corr',
    'prefix_horizontal_vs_typewell_gr_mae',
    'prefix_horizontal_vs_typewell_gr_rmse',
    *TYPEWELL_INTERVAL_CONTEXT_COLUMNS,
    *TYPEWELL_ANCHOR_OFFSET_COLUMNS,
    'typewell_local_last200_best_delta',
    'typewell_local_last200_best_abs_resid',
    'typewell_local_last200_zero_abs_resid',
    'typewell_local_last200_top2_gap',
    'typewell_local_last200_soft_delta_mean',
    'typewell_local_last200_gr_resid_best',
    'typewell_local_last200_best_vs_zero_gain',
]

CALIBRATED_TYPEWELL_ALIGNMENT_COLUMNS = [
    'prefix_typewell_gr_affine_a',
    'prefix_typewell_gr_affine_b',
    'prefix_horizontal_vs_calibrated_typewell_gr_mae',
    'prefix_horizontal_vs_calibrated_typewell_gr_rmse',
    *CALIBRATED_TYPEWELL_ANCHOR_OFFSET_COLUMNS,
    'typewell_calibrated_gr_at_last_known_tvt',
    'typewell_calibrated_gr_at_slope_baseline_all',
    'typewell_calibrated_gr_at_slope_baseline_last200',
    'gr_minus_calibrated_typewell_last_known_tvt',
    'gr_minus_calibrated_typewell_slope_baseline_all',
    'gr_minus_calibrated_typewell_slope_baseline_last200',
    'calibrated_typewell_slope_last200_gr_prefix_z',
    'calibrated_typewell_local_last200_best_delta',
    'calibrated_typewell_local_last200_best_abs_resid',
    'calibrated_typewell_local_last200_zero_abs_resid',
    'calibrated_typewell_local_last200_top2_gap',
    'calibrated_typewell_local_last200_soft_delta_mean',
    'calibrated_typewell_local_last200_gr_resid_best',
    'calibrated_typewell_local_last200_best_vs_zero_gain',
]

FORMATION_PLANE_COLUMNS = [
    'formation_plane_ancc',
    'formation_plane_astnu',
    'formation_plane_astnl',
    'formation_plane_egfdu',
    'formation_plane_egfdl',
    'formation_plane_buda',
    'formation_plane_min_dist',
    'formation_plane_anchor_b',
    'formation_plane_prefix_rmse',
    'formation_plane_prefix_mae',
    'formation_plane_tvt_formula',
    'formation_plane_delta_formula',
    'formation_plane_delta_from_slope_last200',
]

FORMATION_ROW_ANCC_COLUMNS = [
    'formation_row_ancc',
    'formation_row_ancc_std',
    'formation_row_min_dist',
    'formation_row_anchor_b',
    'formation_row_prefix_rmse',
    'formation_row_prefix_mae',
    'formation_row_tvt_formula',
    'formation_row_delta_formula',
    'formation_row_delta_from_plane',
    'formation_formula_mean_delta',
    'formation_formula_abs_gap',
]

# Beam alignment is expensive during repeated local CV. Keep it off locally by default,
# but enable it for the final Kaggle GPU submission path.
ENABLE_OFFLINE_BEAM_FEATURES = bool(ENABLE_KAGGLE_BEAM_FEATURES and KAGGLE_NOTEBOOK_RUN and not KAGGLE_MEMORY_SAFE_MODE)
OFFLINE_CANDIDATE_PATH_COLUMNS = candidate_path_feature_names(prefix='tw_path')
OFFLINE_CANDIDATE_PATH_EASE_COLUMNS = candidate_path_feature_names(prefix='tw_path_ease')
OFFLINE_BEAM_FEATURE_COLUMNS = offline_beam_feature_names(prefix='tw_beam')

OFFLINE_EXTRA_FEATURE_COLUMNS = [
    'tail_len',
    'tail_row_number',
    'tail_frac',
    'tail_frac2',
    'tail_frac3',
    'sqrt_tail_frac',
    'log1p_tail_row',
    'sin_tail_frac_pi',
    'sin_tail_frac_2pi',
    'cos_tail_frac_3pi',
    'n_rows',
    'row_frac',
    'md_tail_span',
    'md_tail_frac',
    'tail_gr_missing_rate',
    'gap_md_span',
    'gap_x_delta',
    'gap_y_delta',
    'gap_z_delta',
    'gap_xy_span',
    'gap_z_over_xy',
    'gap_gr_mean',
    'gap_gr_std',
    'gap_gr_min',
    'gap_gr_p05',
    'gap_gr_p25',
    'gap_gr_p50',
    'gap_gr_p75',
    'gap_gr_p95',
    'gap_gr_max',
    'dist_xyz_ps',
    'dx_per_md_since_ps',
    'dy_per_md_since_ps',
    'dz_per_md_since_ps',
    'gr_center_roll_mean_5',
    'gr_center_roll_std_5',
    'gr_center_roll_range_5',
    'gr_center_roll_mean_21',
    'gr_center_roll_std_21',
    'gr_center_roll_range_21',
    'gr_center_grad_1',
    'gr_center_lag1',
    'gr_center_lead1',
    'gr_center_lag5',
    'gr_center_lead5',
    'gr_cumsum_since_ps',
    'gr_center_roll_mean_51',
    'gr_center_roll_std_51',
    'gr_center_roll_range_51',
    'gr_center_roll_mean_151',
    'gr_center_roll_std_151',
    'gr_center_roll_range_151',
    'gr_center_roll_mean_301',
    'gr_center_roll_std_301',
    'gr_center_roll_range_301',
]

STRICT_FEATURE_SETS = {
    'causal_base': BASE_FEATURE_COLUMNS,
    'prefix_context': BASE_FEATURE_COLUMNS + PREFIX_CONTEXT_COLUMNS,
    'typewell_alignment': BASE_FEATURE_COLUMNS + PREFIX_CONTEXT_COLUMNS + TYPEWELL_ALIGNMENT_COLUMNS,
    'calibrated_typewell_alignment': (
        BASE_FEATURE_COLUMNS
        + PREFIX_CONTEXT_COLUMNS
        + TYPEWELL_ALIGNMENT_COLUMNS
        + CALIBRATED_TYPEWELL_ALIGNMENT_COLUMNS
    ),
}

OFFLINE_FEATURE_SETS = {
    'offline_prefix_context': STRICT_FEATURE_SETS['prefix_context'] + OFFLINE_EXTRA_FEATURE_COLUMNS,
    'offline_typewell_alignment': STRICT_FEATURE_SETS['typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS,
    'offline_calibrated_typewell_alignment': STRICT_FEATURE_SETS['calibrated_typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS,
    'offline_candidate_path_alignment': STRICT_FEATURE_SETS['typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + OFFLINE_CANDIDATE_PATH_COLUMNS + OFFLINE_CANDIDATE_PATH_EASE_COLUMNS,
    'offline_candidate_path_calibrated_alignment': STRICT_FEATURE_SETS['calibrated_typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + OFFLINE_CANDIDATE_PATH_COLUMNS + OFFLINE_CANDIDATE_PATH_EASE_COLUMNS,
    'offline_formation_plane_alignment': STRICT_FEATURE_SETS['typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + FORMATION_PLANE_COLUMNS,
    'offline_formation_top_alignment': STRICT_FEATURE_SETS['typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + FORMATION_PLANE_COLUMNS + FORMATION_ROW_ANCC_COLUMNS,
}
if ENABLE_OFFLINE_BEAM_FEATURES:
    OFFLINE_FEATURE_SETS['offline_beam_candidate_path_alignment'] = (
        STRICT_FEATURE_SETS['typewell_alignment']
        + OFFLINE_EXTRA_FEATURE_COLUMNS
        + OFFLINE_CANDIDATE_PATH_COLUMNS
        + OFFLINE_CANDIDATE_PATH_EASE_COLUMNS
        + OFFLINE_BEAM_FEATURE_COLUMNS
    )

FEATURE_SETS = {**STRICT_FEATURE_SETS, **OFFLINE_FEATURE_SETS}
FEATURE_SET_POLICY = {name: 'strict' for name in STRICT_FEATURE_SETS}
FEATURE_SET_POLICY.update({name: 'offline' for name in OFFLINE_FEATURE_SETS})

DEFAULT_FEATURE_SETS_TO_COMPARE = [
    name
    for name in FEATURE_SETS
    if name not in {
        # Row-level ANCC KNN is useful as a targeted experiment, but it is heavier than the
        # plane-fit formation features and is not needed for the memory-safe submission path.
        'offline_formation_top_alignment',
    }
]
if MODEL_CONFIG['feature_sets_to_compare'] is None:
    MODEL_CONFIG['feature_sets_to_compare'] = DEFAULT_FEATURE_SETS_TO_COMPARE

# Latest full grouped-CV selection saved in this notebook.
# These defaults make submission-only runs reproducible without rerunning CV.
SELECTED_FEATURE_SET = 'calibrated_typewell_alignment'
FEATURE_COLUMNS = FEATURE_SETS[SELECTED_FEATURE_SET]
SELECTED_POLICY_METRIC = 'global_alpha_0.812_fade_200_0_slope_q_0_9'
SELECTED_SHRINKAGE_ALPHA = 0.811832
SELECTED_FADE_IN_TAU_MD = 200.0
SELECTED_SLOPE_QUANTILE = 0.90
APPLY_SELECTED_SLOPE_CLIP = True
BEST_OVERALL_FEATURE_SET = 'offline_candidate_path_alignment'
BEST_OVERALL_SHRINKAGE_ALPHA = 0.941149
BEST_OVERALL_FADE_IN_TAU_MD = 200.0
BEST_OVERALL_SLOPE_QUANTILE = 0.90
APPLY_BEST_OVERALL_SLOPE_CLIP = True

PUBLIC_LGBM_STYLE_COLUMNS = [
    'row_index', 'last_known_TVT', 'prefix_len', 'tail_len', 'tail_row_number', 'tail_frac',
    'n_rows', 'row_frac', 'md_tail_frac', 'md_tail_span',
    'MD', 'Z', 'X', 'Y', 'GR', 'GR_isna',
    'gr_center_roll_mean_5', 'gr_center_roll_mean_21',
    'gr_center_grad_1', 'gr_center_roll_std_5', 'gr_center_roll_std_21',
    'gr_center_lag1', 'gr_center_lead1', 'gr_center_lag5', 'gr_center_lead5', 'gr_cumsum_since_ps',
    'md_since_ps', 'z_delta_ps', 'x_delta_ps', 'y_delta_ps',
    'dx_per_md_since_ps', 'dy_per_md_since_ps', 'dz_per_md_since_ps', 'xy_dist_ps', 'dist_xyz_ps',
    'prefix_tvt_step20', 'prefix_tvt_step100', 'prefix_tvt_md_slope100', 'prefix_tvt_z_slope100',
    'prefix_horizontal_vs_typewell_gr_rmse', 'prefix_horizontal_vs_typewell_gr_mae',
    'typewell_anchor_gr_diff_m80', 'typewell_anchor_gr_diff_m40', 'typewell_anchor_gr_diff_m20',
    'typewell_anchor_gr_diff_m10', 'typewell_anchor_gr_diff_m5', 'typewell_anchor_gr_diff_p0',
    'typewell_anchor_gr_diff_p5', 'typewell_anchor_gr_diff_p10', 'typewell_anchor_gr_diff_p20',
    'typewell_anchor_gr_diff_p40', 'typewell_anchor_gr_diff_p80',
]
PUBLIC_LGBM_ALWAYS_AVAILABLE_COLUMNS = ['row_index']
PUBLIC_LGBM_COMPACT_EXTRA_COLUMNS = [
    'tail_frac2',
    'tail_frac3',
    'sqrt_tail_frac',
    'log1p_tail_row',
    'sin_tail_frac_pi',
    'sin_tail_frac_2pi',
    'cos_tail_frac_3pi',
    'gap_md_span',
    'gap_x_delta',
    'gap_y_delta',
    'gap_z_delta',
    'gap_xy_span',
    'gap_z_over_xy',
    'gap_gr_mean',
    'gap_gr_std',
    'gap_gr_min',
    'gap_gr_p05',
    'gap_gr_p25',
    'gap_gr_p50',
    'gap_gr_p75',
    'gap_gr_p95',
    'gap_gr_max',
    'tail_gr_missing_rate',
    'gr_center_roll_range_5',
    'gr_center_roll_range_21',
    'gr_center_roll_mean_51',
    'gr_center_roll_std_51',
    'gr_center_roll_range_51',
    'gr_center_roll_mean_151',
    'gr_center_roll_std_151',
    'gr_center_roll_range_151',
]
PUBLIC_LGBM_STYLE_COLUMNS = list(dict.fromkeys(PUBLIC_LGBM_STYLE_COLUMNS + PUBLIC_LGBM_COMPACT_EXTRA_COLUMNS))
PUBLIC_LGBM_STYLE_COLUMNS = [
    col
    for col in PUBLIC_LGBM_STYLE_COLUMNS
    if col in FEATURE_SETS['offline_typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + PUBLIC_LGBM_ALWAYS_AVAILABLE_COLUMNS
]
FEATURE_SETS['offline_public_lgbm_style'] = list(dict.fromkeys(PUBLIC_LGBM_STYLE_COLUMNS))
FEATURE_SET_POLICY['offline_public_lgbm_style'] = 'offline'
PUBLIC_LGBM_FORMATION_STYLE_COLUMNS = list(dict.fromkeys(PUBLIC_LGBM_STYLE_COLUMNS + FORMATION_PLANE_COLUMNS))
FEATURE_SETS['offline_public_lgbm_formation_style'] = PUBLIC_LGBM_FORMATION_STYLE_COLUMNS
FEATURE_SET_POLICY['offline_public_lgbm_formation_style'] = 'offline'
# The public-style feature set is available for targeted experiments but excluded from default CV
# unless explicitly listed in MODEL_CONFIG['feature_sets_to_compare'].

feature_set_report = pd.DataFrame({
    'feature_set': list(FEATURE_SETS.keys()),
    'feature_policy': [FEATURE_SET_POLICY[name] for name in FEATURE_SETS],
    'feature_count': [len(cols) for cols in FEATURE_SETS.values()],
})
display(feature_set_report)
print('NOTEBOOK_RUN_VERSION:', NOTEBOOK_RUN_VERSION)
print('RUN_GROUPED_CV:', RUN_GROUPED_CV)
print('KAGGLE_MEMORY_SAFE_MODE:', KAGGLE_MEMORY_SAFE_MODE)
print('ENABLE_OFFLINE_BEAM_FEATURES:', ENABLE_OFFLINE_BEAM_FEATURES)
print('RUN_HGB_DIAGNOSTIC_SUBMISSIONS:', RUN_HGB_DIAGNOSTIC_SUBMISSIONS)
print('QUICK_CHECK_MODE:', QUICK_CHECK_MODE)
print('selection_feature_policy:', MODEL_CONFIG['feature_policy_for_selection'])
print('configured_feature_set_before_cv:', SELECTED_FEATURE_SET)


### 16.1 Feature Table Builder

### Sampling rule

If well $w$ has $n_w$ eligible tail rows and the cap is $m$:

$$
S_w \subset \mathcal{T}_w, \quad |S_w|=\min(n_w,m).
$$

| Effect | Statistical consequence |
|---|---|
| Reduces millions of rows | lower computation cost |
| Caps rows per well | loss weighting becomes less dominated by very long wells |
| Sampling inside training fold | no validation-label leakage |

Feature construction must obey the active information policy.

In [ ]:
# Build tail feature matrices and define model/post-processing helpers.

def feature_columns_require_beam(feature_columns) -> bool:
    if feature_columns is None:
        return bool(globals().get('ENABLE_OFFLINE_BEAM_FEATURES', False))
    return any(str(col).startswith('tw_beam_') or str(col).startswith('beam_') for col in feature_columns)


def feature_columns_require_formation(feature_columns) -> bool:
    if feature_columns is None:
        return False
    return any(str(col).startswith('formation_') for col in feature_columns)


def feature_columns_require_row_ancc(feature_columns) -> bool:
    if feature_columns is None:
        return False
    return any(str(col).startswith('formation_row_') or str(col).startswith('formation_formula_') for col in feature_columns)


def compact_tail_feature_frame(
    frame: pd.DataFrame,
    keep_columns=None,
    include_target: bool = True,
    keep_metadata: bool = True,
) -> pd.DataFrame:
    if keep_columns is not None:
        required_columns = []
        if keep_metadata:
            required_columns += [
                'well_id',
                'row_index',
                'id',
                'MD',
                'last_known_TVT',
                'last_known_MD',
                'md_since_ps',
            ]
        if include_target:
            required_columns += ['target_tvt', 'target_delta_from_last_known']
        frame = frame.copy()
        for col in keep_columns:
            if col not in frame.columns:
                frame[col] = np.nan
        wanted = list(dict.fromkeys(required_columns + list(keep_columns)))
        frame = frame[[col for col in wanted if col in frame.columns]]
    else:
        frame = frame.copy()

    for col in frame.columns:
        if pd.api.types.is_float_dtype(frame[col]):
            frame[col] = pd.to_numeric(frame[col], downcast='float')
        elif pd.api.types.is_integer_dtype(frame[col]) and col not in {'row_index'}:
            frame[col] = pd.to_numeric(frame[col], downcast='integer')
    return frame


def build_tail_feature_frame(
    well_ids,
    split_dir: Path,
    include_target: bool,
    rows_per_well: int | None = None,
    random_state: int = 42,
    use_beam_features: bool | None = None,
    keep_columns=None,
    formation_plane_imputer=None,
    row_ancc_imputer=None,
    imputer_well_ids=None,
    keep_metadata: bool = True,
    exclude_query_well_from_formation: bool = True,
) -> pd.DataFrame:
    if use_beam_features is None:
        use_beam_features = bool(globals().get('ENABLE_OFFLINE_BEAM_FEATURES', False))
    need_formation = feature_columns_require_formation(keep_columns)
    need_row_ancc = feature_columns_require_row_ancc(keep_columns)
    if need_formation and formation_plane_imputer is None:
        source_wells = imputer_well_ids if imputer_well_ids is not None else globals().get('all_train_wells', globals().get('train_h_ids', []))
        formation_plane_imputer, maybe_row = make_formation_imputers(
            source_wells,
            TRAIN_DIR,
            need_row_ancc=need_row_ancc and row_ancc_imputer is None,
            seed=random_state,
        )
        if row_ancc_imputer is None:
            row_ancc_imputer = maybe_row
    rng = np.random.default_rng(random_state)
    frames = []
    for well_id in sorted(well_ids):
        frame = make_tail_features_for_well(
            well_id,
            split_dir,
            include_target=include_target,
            use_beam_features=use_beam_features,
            required_feature_columns=keep_columns,
            formation_plane_imputer=formation_plane_imputer,
            row_ancc_imputer=row_ancc_imputer,
            exclude_query_well_from_formation=exclude_query_well_from_formation,
        )
        if frame.empty:
            continue
        if rows_per_well is not None and len(frame) > rows_per_well:
            sampled_idx = np.sort(rng.choice(frame.index.to_numpy(), size=rows_per_well, replace=False))
            frame = frame.loc[sampled_idx]
        frame = compact_tail_feature_frame(
            frame,
            keep_columns=keep_columns,
            include_target=include_target,
            keep_metadata=keep_metadata,
        )
        frames.append(frame)
        gc.collect()
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, copy=False)


def model_ready_xy(frame: pd.DataFrame, feature_columns=None):
    if feature_columns is None:
        feature_columns = FEATURE_COLUMNS
    X = frame[feature_columns].copy()
    y_delta = frame['target_delta_from_last_known'].to_numpy()
    y_tvt = frame['target_tvt'].to_numpy()
    return X, y_delta, y_tvt


def make_residual_model(random_state: int = 42):
    # Fixed-iteration HGB avoids row-random internal early stopping inside a grouped-CV fold.
    return HistGradientBoostingRegressor(
        loss='squared_error',
        learning_rate=0.06,
        max_iter=220,
        max_leaf_nodes=31,
        l2_regularization=0.05,
        early_stopping=False,
        random_state=random_state,
    )


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def clip_delta_by_train_quantiles(delta_pred: np.ndarray, train_delta: np.ndarray, config=MODEL_CONFIG) -> np.ndarray:
    lo_q, hi_q = config['delta_clip_quantiles']
    lo, hi = np.nanquantile(train_delta, [lo_q, hi_q])
    return np.clip(delta_pred, lo, hi)


def shrink_delta(delta: np.ndarray, alpha: float | None = None, config=MODEL_CONFIG) -> np.ndarray:
    if alpha is None:
        alpha = config['residual_shrinkage']
    return float(alpha) * np.asarray(delta, dtype=float)


def fade_in_delta(frame: pd.DataFrame, delta: np.ndarray, tau_md: float | None) -> np.ndarray:
    """Dampen residuals close to Prediction Start; tau=None leaves them unchanged."""
    values = np.asarray(delta, dtype=float)
    if tau_md is None or not np.isfinite(tau_md) or tau_md <= 0:
        return values
    md_since = frame['md_since_ps'].to_numpy(dtype=float)
    rho = 1.0 - np.exp(-np.maximum(md_since, 0.0) / float(tau_md))
    return values * rho


def fade_in_tau_label(tau_md: float | None) -> str:
    if tau_md is None or (isinstance(tau_md, float) and not np.isfinite(tau_md)):
        return 'none'
    return str(float(tau_md)).replace('.', '_')


def fit_global_alpha_from_fold_parts(fold_parts, tau_md: float | None, config=MODEL_CONFIG) -> tuple[float, float]:
    """Fit one alpha over all validation folds for a fixed fade-in setting."""
    numerator = 0.0
    denominator = 0.0
    for part in fold_parts:
        frame = part['frame']
        base_delta = fade_in_delta(frame, part['clipped_delta'], tau_md)
        y_delta = frame['target_delta_from_last_known'].to_numpy(dtype=float)
        valid = np.isfinite(base_delta) & np.isfinite(y_delta)
        numerator += float(np.dot(y_delta[valid], base_delta[valid]))
        denominator += float(np.dot(base_delta[valid], base_delta[valid]))
    if denominator <= 1e-12:
        alpha = 0.0
    else:
        alpha = numerator / denominator
    lo, hi = config['alpha_bounds']
    alpha = float(np.clip(alpha, lo, hi))
    score = score_policy_from_fold_parts(fold_parts, alpha=alpha, tau_md=tau_md, slope_quantile=None)['rmse']
    return alpha, score


def score_policy_from_fold_parts(fold_parts, alpha: float, tau_md: float | None, slope_quantile: float | None) -> dict[str, float]:
    sse = 0.0
    n = 0
    for part in fold_parts:
        frame = part['frame']
        delta = shrink_delta(fade_in_delta(frame, part['clipped_delta'], tau_md), alpha=alpha)
        pred = frame['last_known_TVT'].to_numpy(dtype=float) + delta
        if slope_quantile is not None:
            pred = causal_slope_clip_by_well(frame, pred, part['slope_bounds'][float(slope_quantile)])
        y = frame['target_tvt'].to_numpy(dtype=float)
        valid = np.isfinite(y) & np.isfinite(pred)
        err = y[valid] - pred[valid]
        sse += float(np.dot(err, err))
        n += int(valid.sum())
    return {'rmse': float(np.sqrt(sse / n)), 'sse': sse, 'n': n}


def policy_metric_name(alpha: float, tau_md: float | None, slope_quantile: float | None) -> str:
    tau_label = fade_in_tau_label(tau_md)
    if slope_quantile is None:
        slope_label = 'none'
    else:
        slope_label = f'q_{slope_quantile_label(float(slope_quantile))}'
    return f'global_alpha_{alpha:.3f}_fade_{tau_label}_slope_{slope_label}'


def best_alpha_for_clipped_delta(
    last_known_tvt: np.ndarray,
    clipped_delta: np.ndarray,
    y_true: np.ndarray,
    alphas=None,
) -> tuple[float, float]:
    if alphas is None:
        alphas = np.linspace(0.0, 1.5, 61)
    best_score = np.inf
    best_alpha = np.nan
    for alpha in alphas:
        score = rmse(y_true, last_known_tvt + alpha * clipped_delta)
        if score < best_score:
            best_score = score
            best_alpha = float(alpha)
    return best_alpha, float(best_score)


def estimate_abs_tvt_slope_quantiles(paths, quantiles, max_wells=None) -> dict[float, float]:
    slopes = []
    selected_paths = list(paths)[:max_wells] if max_wells is not None else list(paths)
    for path in selected_paths:
        df = pd.read_csv(path, usecols=['MD', 'TVT'])
        md = df['MD'].to_numpy(dtype=float)
        tvt = df['TVT'].to_numpy(dtype=float)
        dmd = np.diff(md)
        dtvt = np.diff(tvt)
        valid = np.isfinite(dmd) & np.isfinite(dtvt) & (np.abs(dmd) > 1e-9)
        if valid.any():
            slopes.append(np.abs(dtvt[valid] / dmd[valid]))
    if not slopes:
        return {float(q): np.nan for q in quantiles}
    values = np.concatenate(slopes)
    return {float(q): float(np.nanquantile(values, q)) for q in quantiles}


def estimate_abs_tvt_slope_quantile(paths, quantile=0.995, max_wells=None) -> float:
    return estimate_abs_tvt_slope_quantiles(paths, [quantile], max_wells=max_wells)[float(quantile)]


def slope_quantile_label(q: float) -> str:
    return str(float(q)).replace('.', '_')


def causal_slope_clip_by_well(frame: pd.DataFrame, pred_tvt: np.ndarray, max_abs_slope: float) -> np.ndarray:
    """Anchor-aware forward slope limiter. It never averages with future rows."""
    clipped = np.asarray(pred_tvt, dtype=float).copy()
    if not np.isfinite(max_abs_slope) or max_abs_slope <= 0:
        return clipped
    required = ['well_id', 'row_index', 'MD', 'last_known_TVT', 'last_known_MD']
    frame_ordered = frame[required].copy()
    frame_ordered['_pos'] = np.arange(len(frame_ordered))
    for _, g in frame_ordered.sort_values(['well_id', 'row_index']).groupby('well_id', sort=False):
        pos = g['_pos'].to_numpy(dtype=int)
        md = g['MD'].to_numpy(dtype=float)
        prev_tvt = float(g['last_known_TVT'].iloc[0])
        prev_md = float(g['last_known_MD'].iloc[0])
        for k in range(len(pos)):
            step_md = abs(md[k] - prev_md)
            limit = max_abs_slope * max(step_md, 1e-6)
            clipped[pos[k]] = np.clip(clipped[pos[k]], prev_tvt - limit, prev_tvt + limit)
            prev_tvt = clipped[pos[k]]
            prev_md = md[k]
    return clipped


MAX_ABS_TVT_SLOPE_BY_QUANTILE = estimate_abs_tvt_slope_quantiles(
    train_horizontal_files,
    quantiles=MODEL_CONFIG['slope_clip_quantiles_to_compare'],
)
MAX_ABS_TVT_SLOPE = MAX_ABS_TVT_SLOPE_BY_QUANTILE[float(MODEL_CONFIG['postprocess_slope_quantile'])]
print('feature builder configured')
print('final_train_max_abs_tvt_slope_by_quantile:', MAX_ABS_TVT_SLOPE_BY_QUANTILE)


### 16.2 GroupKFold Validation

### Fold metric

Validation is evaluated on full prediction tails from held-out wells. Fold summaries are row-weighted because the competition metric is row-level RMSE and long tails contribute more rows.

<details>
<summary>Fold RMSE</summary>

$$
\mathrm{RMSE}^{(k)}
=
\sqrt{
\frac{1}{N_k}
\sum_{w\in\mathcal{W}^{(k)}_{valid}}
\sum_{i\in\mathcal{T}_w}
(y_{w,i}-\hat{y}_{w,i})^2
}.
$$

</details>

### Prediction variants

| Variant | Purpose |
|---|---|
| Raw | checks the residual model before post-processing |
| Delta-clipped | removes extreme residual outliers |
| Global-shrunk | fits one alpha across all OOF rows for a feature set |
| Fade-in | suppresses early-tail residuals near Prediction Start |
| Slope-limited | applies anchor-aware causal clipping after shrinkage and fade-in |

### Selection rule

A single post-processing policy is selected per feature set using all OOF rows. This avoids choosing a different oracle alpha inside each fold.

### CV caution

`GroupKFold` balances well ids, not geology, geography, tail length, or GR missingness. Those variables remain useful for fold-difficulty diagnostics.


In [ ]:
# Run well-level GroupKFold validation and choose one global post-processing policy per feature set.

all_train_wells = np.array(sorted(train_h_ids))
gkf = GroupKFold(n_splits=MODEL_CONFIG['n_splits'])
fold_splits = list(gkf.split(all_train_wells, groups=all_train_wells))[:MODEL_CONFIG['cv_folds_to_run']]
slope_quantiles = [float(q) for q in MODEL_CONFIG['slope_clip_quantiles_to_compare']]
tau_candidates = MODEL_CONFIG['fade_in_tau_md_to_compare']
feature_names_to_compare = list(MODEL_CONFIG['feature_sets_to_compare'])
cv_required_feature_columns = list(dict.fromkeys(
    col
    for name in feature_names_to_compare
    if name in FEATURE_SETS
    for col in FEATURE_SETS[name]
))
cv_requires_beam_features = any(
    feature_columns_require_beam(FEATURE_SETS[name])
    for name in feature_names_to_compare
    if name in FEATURE_SETS
)
cv_requires_formation_features = feature_columns_require_formation(cv_required_feature_columns)
cv_requires_row_ancc_features = feature_columns_require_row_ancc(cv_required_feature_columns)

if not RUN_GROUPED_CV:
    cv_report = pd.DataFrame()
    policy_grid = pd.DataFrame()
    fold_overview = pd.DataFrame()
    feature_fold_parts = {}
    cv_summary = pd.DataFrame([
        {
            'feature_set': SELECTED_FEATURE_SET,
            'feature_policy': FEATURE_SET_POLICY[SELECTED_FEATURE_SET],
            'feature_count': len(FEATURE_SETS[SELECTED_FEATURE_SET]),
            'row_weighted_policy_rmse': np.nan,
            'selected_policy_metric': SELECTED_POLICY_METRIC,
            'policy_alpha': SELECTED_SHRINKAGE_ALPHA,
            'policy_fade_tau_md': SELECTED_FADE_IN_TAU_MD,
            'policy_slope_quantile': SELECTED_SLOPE_QUANTILE,
            'policy_apply_slope_clip': APPLY_SELECTED_SLOPE_CLIP,
        },
        {
            'feature_set': BEST_OVERALL_FEATURE_SET,
            'feature_policy': FEATURE_SET_POLICY[BEST_OVERALL_FEATURE_SET],
            'feature_count': len(FEATURE_SETS[BEST_OVERALL_FEATURE_SET]),
            'row_weighted_policy_rmse': np.nan,
            'selected_policy_metric': 'stored_best_overall',
            'policy_alpha': BEST_OVERALL_SHRINKAGE_ALPHA,
            'policy_fade_tau_md': BEST_OVERALL_FADE_IN_TAU_MD,
            'policy_slope_quantile': BEST_OVERALL_SLOPE_QUANTILE,
            'policy_apply_slope_clip': APPLY_BEST_OVERALL_SLOPE_CLIP,
        },
    ])
    print('Grouped CV skipped. Using stored CV selections from the latest full validation run.')
    display(cv_summary)
else:
    cv_rows = []
    policy_rows = []
    fold_overview_rows = []
    feature_fold_parts = {name: [] for name in feature_names_to_compare}

    for fold, (tr_idx, va_idx) in enumerate(fold_splits, start=1):
        train_wells = all_train_wells[tr_idx]
        valid_wells = all_train_wells[va_idx]
        fold_train_paths = [TRAIN_DIR / f'{well_id}__horizontal_well.csv' for well_id in train_wells]
        fold_slope_bounds = estimate_abs_tvt_slope_quantiles(
            fold_train_paths,
            quantiles=slope_quantiles,
        )
        configured_fold_slope = fold_slope_bounds[float(MODEL_CONFIG['postprocess_slope_quantile'])]
        fold_plane_imputer = fold_row_imputer = None
        if cv_requires_formation_features:
            fold_plane_imputer, fold_row_imputer = make_formation_imputers(
                train_wells,
                TRAIN_DIR,
                need_row_ancc=cv_requires_row_ancc_features,
                seed=MODEL_CONFIG['random_state'] + fold,
            )
    
        train_frame = build_tail_feature_frame(
            train_wells,
            TRAIN_DIR,
            include_target=True,
            rows_per_well=MODEL_CONFIG['train_rows_per_well'],
            random_state=MODEL_CONFIG['random_state'] + fold,
            use_beam_features=cv_requires_beam_features,
            keep_columns=cv_required_feature_columns,
            formation_plane_imputer=fold_plane_imputer,
            row_ancc_imputer=fold_row_imputer,
            imputer_well_ids=train_wells,
        )
        valid_frame = build_tail_feature_frame(
            valid_wells,
            TRAIN_DIR,
            include_target=True,
            rows_per_well=None,
            random_state=MODEL_CONFIG['random_state'] + 1000 + fold,
            use_beam_features=cv_requires_beam_features,
            keep_columns=cv_required_feature_columns,
            formation_plane_imputer=fold_plane_imputer,
            row_ancc_imputer=fold_row_imputer,
            imputer_well_ids=train_wells,
        )
        last_known_valid = valid_frame['last_known_TVT'].to_numpy(dtype=float)
        y_valid_tvt = valid_frame['target_tvt'].to_numpy(dtype=float)
        y_valid_delta = valid_frame['target_delta_from_last_known'].to_numpy(dtype=float)
        fold_baseline_rmse = rmse(y_valid_tvt, last_known_valid)
    
        fold_overview_rows.append({
            'fold': fold,
            'valid_wells': len(valid_wells),
            'valid_rows': len(valid_frame),
            'constant_baseline_rmse': fold_baseline_rmse,
        })
        compact_valid = valid_frame[[
            'well_id',
            'row_index',
            'MD',
            'last_known_MD',
            'last_known_TVT',
            'target_tvt',
            'target_delta_from_last_known',
            'md_since_ps',
            'tail_row_number',
        ]].copy()
    
        for feature_set_name in feature_names_to_compare:
            feature_columns = FEATURE_SETS[feature_set_name]
            X_train, y_train_delta, _ = model_ready_xy(train_frame, feature_columns)
            X_valid, _, _ = model_ready_xy(valid_frame, feature_columns)
            model = make_residual_model(random_state=MODEL_CONFIG['random_state'] + fold)
            model.fit(X_train, y_train_delta)
            raw_delta = model.predict(X_valid)
            clipped_delta = clip_delta_by_train_quantiles(raw_delta, y_train_delta)
            fixed_shrunk_delta = shrink_delta(clipped_delta)
            pred_raw = last_known_valid + raw_delta
            pred_clipped = last_known_valid + clipped_delta
            pred_fixed_shrunk = last_known_valid + fixed_shrunk_delta
            fold_best_alpha, fold_best_alpha_rmse = best_alpha_for_clipped_delta(last_known_valid, clipped_delta, y_valid_tvt)
            pred_configured_slope = causal_slope_clip_by_well(valid_frame, pred_fixed_shrunk, configured_fold_slope)
        
            cv_rows.append({
                'fold': fold,
                'feature_set': feature_set_name,
                'feature_policy': FEATURE_SET_POLICY[feature_set_name],
                'feature_count': len(feature_columns),
                'train_wells': len(train_wells),
                'valid_wells': len(valid_wells),
                'train_rows_sampled': len(train_frame),
                'valid_rows_full': len(valid_frame),
                'configured_fold_max_abs_tvt_slope': configured_fold_slope,
                'baseline_constant_rmse': fold_baseline_rmse,
                'rmse_raw_model': rmse(y_valid_tvt, pred_raw),
                'rmse_delta_clipped': rmse(y_valid_tvt, pred_clipped),
                'rmse_fixed_shrunk_delta': rmse(y_valid_tvt, pred_fixed_shrunk),
                'rmse_anchor_slope_limited': rmse(y_valid_tvt, pred_configured_slope),
                'fixed_shrinkage_alpha': MODEL_CONFIG['residual_shrinkage'],
                'fold_oracle_alpha_from_delta_clip': fold_best_alpha,
                'rmse_fold_oracle_alpha_delta_clip': fold_best_alpha_rmse,
                'delta_target_std_valid': float(np.nanstd(y_valid_delta)),
                'delta_pred_std_raw': float(np.nanstd(raw_delta)),
                'delta_pred_std_clipped': float(np.nanstd(clipped_delta)),
                'delta_pred_std_fixed_shrunk': float(np.nanstd(fixed_shrunk_delta)),
            })
            feature_fold_parts[feature_set_name].append({
                'fold': fold,
                'frame': compact_valid,
                'clipped_delta': clipped_delta,
                'slope_bounds': fold_slope_bounds,
            })

    for feature_set_name in feature_names_to_compare:
        feature_columns = FEATURE_SETS[feature_set_name]
        fold_parts = feature_fold_parts[feature_set_name]
        fade_candidates = []
        for tau_md in tau_candidates:
            alpha, no_slope_rmse = fit_global_alpha_from_fold_parts(fold_parts, tau_md=tau_md)
            fade_candidates.append({
                'tau_md': tau_md,
                'alpha': alpha,
                'no_slope_rmse': no_slope_rmse,
            })
        best_fade = min(fade_candidates, key=lambda row: row['no_slope_rmse'])
    
        slope_options = [None] + slope_quantiles
        for slope_q in slope_options:
            score = score_policy_from_fold_parts(
                fold_parts,
                alpha=best_fade['alpha'],
                tau_md=best_fade['tau_md'],
                slope_quantile=slope_q,
            )
            policy_rows.append({
                'feature_set': feature_set_name,
                'feature_policy': FEATURE_SET_POLICY[feature_set_name],
                'feature_count': len(feature_columns),
                'policy_metric': policy_metric_name(best_fade['alpha'], best_fade['tau_md'], slope_q),
                'policy_alpha': best_fade['alpha'],
                'policy_fade_tau_md': best_fade['tau_md'],
                'policy_slope_quantile': slope_q,
                'policy_apply_slope_clip': slope_q is not None,
                'policy_rmse': score['rmse'],
                'policy_rows': score['n'],
                'global_alpha_no_slope_rmse': best_fade['no_slope_rmse'],
            })

    cv_report = pd.DataFrame(cv_rows)
    policy_grid = pd.DataFrame(policy_rows)
    fold_overview = pd.DataFrame(fold_overview_rows)

    if len(cv_report) and len(policy_grid):
        print('CV folds evaluated:', cv_report['fold'].nunique())
        print('Feature sets compared:', cv_report['feature_set'].nunique())
        print('Postprocess policies compared:', len(policy_grid))
    
        agg_spec = {
            'feature_policy': ('feature_policy', 'first'),
            'feature_count': ('feature_count', 'first'),
            'mean_baseline_rmse': ('baseline_constant_rmse', 'mean'),
            'mean_rmse_raw_model': ('rmse_raw_model', 'mean'),
            'mean_rmse_delta_clipped': ('rmse_delta_clipped', 'mean'),
            'mean_rmse_fixed_shrunk_delta': ('rmse_fixed_shrunk_delta', 'mean'),
            'mean_rmse_anchor_slope_limited': ('rmse_anchor_slope_limited', 'mean'),
            'mean_fold_oracle_alpha': ('fold_oracle_alpha_from_delta_clip', 'mean'),
            'mean_rmse_fold_oracle_alpha': ('rmse_fold_oracle_alpha_delta_clip', 'mean'),
            'std_rmse_fixed_shrunk_delta': ('rmse_fixed_shrunk_delta', 'std'),
            'std_rmse_anchor_slope_limited': ('rmse_anchor_slope_limited', 'std'),
        }
        cv_summary = cv_report.groupby('feature_set', as_index=False).agg(**agg_spec)
    
        row_weighted_metric_columns = [
            'baseline_constant_rmse',
            'rmse_raw_model',
            'rmse_delta_clipped',
            'rmse_fixed_shrunk_delta',
            'rmse_anchor_slope_limited',
            'rmse_fold_oracle_alpha_delta_clip',
        ]
        weighted_rows = []
        for feature_set_name, g in cv_report.groupby('feature_set'):
            weights = g['valid_rows_full'].to_numpy(dtype=float)
            row = {'feature_set': feature_set_name}
            for metric_col in row_weighted_metric_columns:
                values = g[metric_col].to_numpy(dtype=float)
                row[f'row_weighted_{metric_col}'] = float(np.sqrt(np.sum(weights * values ** 2) / np.sum(weights)))
            weighted_rows.append(row)
        cv_summary = cv_summary.merge(pd.DataFrame(weighted_rows), on='feature_set', how='left')
    
        best_policy_by_feature = (
            policy_grid.sort_values('policy_rmse')
            .drop_duplicates('feature_set')
            .rename(columns={
                'policy_rmse': 'row_weighted_policy_rmse',
                'policy_metric': 'selected_policy_metric',
            })
        )
        cv_summary = cv_summary.merge(
            best_policy_by_feature[[
                'feature_set',
                'selected_policy_metric',
                'policy_alpha',
                'policy_fade_tau_md',
                'policy_slope_quantile',
                'policy_apply_slope_clip',
                'row_weighted_policy_rmse',
                'global_alpha_no_slope_rmse',
            ]],
            on='feature_set',
            how='left',
        )
        cv_summary['row_weighted_policy_improvement'] = (
            cv_summary['row_weighted_baseline_constant_rmse'] - cv_summary['row_weighted_policy_rmse']
        )
        cv_summary = cv_summary.sort_values(['feature_policy', 'row_weighted_policy_rmse'])
    
        display(fold_overview.style.format({
            'valid_rows': '{:,.0f}',
            'constant_baseline_rmse': '{:.4f}',
        }))
    
        cv_summary_display = cv_summary[[
            'feature_set',
            'feature_policy',
            'feature_count',
            'row_weighted_baseline_constant_rmse',
            'row_weighted_rmse_raw_model',
            'global_alpha_no_slope_rmse',
            'row_weighted_policy_rmse',
            'policy_alpha',
            'policy_fade_tau_md',
            'policy_slope_quantile',
            'selected_policy_metric',
        ]].rename(columns={
            'row_weighted_baseline_constant_rmse': 'baseline_rmse',
            'row_weighted_rmse_raw_model': 'raw_rmse',
            'global_alpha_no_slope_rmse': 'global_alpha_rmse',
            'row_weighted_policy_rmse': 'policy_rmse',
            'policy_alpha': 'alpha',
            'policy_fade_tau_md': 'fade_tau_md',
            'policy_slope_quantile': 'slope_q',
        })
        display(cv_summary_display.style.format({
            'baseline_rmse': '{:.4f}',
            'raw_rmse': '{:.4f}',
            'global_alpha_rmse': '{:.4f}',
            'policy_rmse': '{:.4f}',
            'alpha': '{:.3f}',
            'fade_tau_md': lambda x: 'none' if pd.isna(x) else f'{x:.0f}',
            'slope_q': lambda x: 'none' if pd.isna(x) else f'{x:.3f}',
        }))
    
        best_overall = cv_summary.sort_values('row_weighted_policy_rmse').iloc[0]
        BEST_OVERALL_FEATURE_SET = str(best_overall['feature_set'])
        BEST_OVERALL_SHRINKAGE_ALPHA = float(best_overall['policy_alpha'])
        BEST_OVERALL_FADE_IN_TAU_MD = None if pd.isna(best_overall['policy_fade_tau_md']) else float(best_overall['policy_fade_tau_md'])
        BEST_OVERALL_SLOPE_QUANTILE = None if pd.isna(best_overall['policy_slope_quantile']) else float(best_overall['policy_slope_quantile'])
        APPLY_BEST_OVERALL_SLOPE_CLIP = bool(best_overall['policy_apply_slope_clip'])
    
        if MODEL_CONFIG['select_best_feature_set_from_cv']:
            selection_policy = MODEL_CONFIG['feature_policy_for_selection']
            selection_summary = cv_summary[cv_summary['feature_policy'].eq(selection_policy)].copy()
            if selection_summary.empty:
                raise ValueError(f'No feature sets available for selection policy: {selection_policy}')
            selected = selection_summary.sort_values('row_weighted_policy_rmse').iloc[0]
            SELECTED_FEATURE_SET = str(selected['feature_set'])
            FEATURE_COLUMNS = FEATURE_SETS[SELECTED_FEATURE_SET]
            SELECTED_POLICY_METRIC = str(selected['selected_policy_metric'])
            SELECTED_SHRINKAGE_ALPHA = float(selected['policy_alpha'])
            SELECTED_FADE_IN_TAU_MD = None if pd.isna(selected['policy_fade_tau_md']) else float(selected['policy_fade_tau_md'])
            SELECTED_SLOPE_QUANTILE = None if pd.isna(selected['policy_slope_quantile']) else float(selected['policy_slope_quantile'])
            APPLY_SELECTED_SLOPE_CLIP = bool(selected['policy_apply_slope_clip'])
            selected_summary = pd.Series({
                'selection_policy': selection_policy,
                'selected_feature_set': SELECTED_FEATURE_SET,
                'selected_feature_count': len(FEATURE_COLUMNS),
                'selected_policy_metric': SELECTED_POLICY_METRIC,
                'selected_shrinkage_alpha': SELECTED_SHRINKAGE_ALPHA,
                'selected_fade_in_tau_md': SELECTED_FADE_IN_TAU_MD,
                'selected_slope_quantile': SELECTED_SLOPE_QUANTILE,
                'apply_slope_clip': APPLY_SELECTED_SLOPE_CLIP,
                'selected_policy_rmse': float(selected['row_weighted_policy_rmse']),
                'best_overall_feature_set': BEST_OVERALL_FEATURE_SET,
                'best_overall_policy': best_overall['feature_policy'],
                'best_overall_policy_rmse': float(best_overall['row_weighted_policy_rmse']),
            })
            display(selected_summary.to_frame('value'))
        
        artifact_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR
        cv_report.to_csv(artifact_dir / 'v2_cv_report.csv', index=False)
        policy_grid.to_csv(artifact_dir / 'v2_policy_grid.csv', index=False)
        cv_summary.to_csv(artifact_dir / 'v2_cv_summary.csv', index=False)
        fold_overview.to_csv(artifact_dir / 'v2_cv_fold_overview.csv', index=False)
        print('CV artifacts written to:', artifact_dir)


### 16.2.1 Selected Policy Diagnostics

The selected policy should improve row-level RMSE without damaging wells that are already well explained by the flat anchor. This diagnostic checks three questions:

| Diagnostic | Question |
|---|---|
| Post-processing stages | Which step reduces RMSE: clipping, shrinkage, fade-in, or slope limiting? |
| Tail-position bins | Does the model help immediately after Prediction Start and farther into the tail? |
| Well-level gain | Does the model improve many wells, or only a few drifting wells while hurting flat wells? |

These diagnostics use OOF predictions from grouped validation only. They are not computed on the test labels.


In [ ]:
if not RUN_GROUPED_CV:
    selected_policy_stage_report = pd.DataFrame()
    selected_policy_tail_md_diagnostics = pd.DataFrame()
    selected_policy_tail_row_diagnostics = pd.DataFrame()
    selected_policy_well_gain = pd.DataFrame()
    print('OOF diagnostics skipped because grouped CV was skipped. Run with RUN_GROUPED_CV=True to refresh diagnostics.')
else:
    # Diagnose the selected OOF policy by stage, tail position, and well-level gain.

    required_diagnostic_vars = [
        'feature_fold_parts',
        'SELECTED_FEATURE_SET',
        'SELECTED_SHRINKAGE_ALPHA',
        'SELECTED_FADE_IN_TAU_MD',
        'SELECTED_SLOPE_QUANTILE',
        'APPLY_SELECTED_SLOPE_CLIP',
    ]
    missing_diagnostic_vars = [name for name in required_diagnostic_vars if name not in globals()]
    if missing_diagnostic_vars:
        raise RuntimeError(f'Selected-policy diagnostic variables are missing: {missing_diagnostic_vars}. Run the validation cell first.')

    if SELECTED_FEATURE_SET not in feature_fold_parts:
        raise KeyError(f'Selected feature set is not available in OOF fold parts: {SELECTED_FEATURE_SET}')

    selected_fold_parts = feature_fold_parts[SELECTED_FEATURE_SET]
    stage_sse = {
        'constant_anchor': 0.0,
        'delta_clipped': 0.0,
        'global_alpha': 0.0,
        'fade_in': 0.0,
        'selected_policy': 0.0,
    }
    stage_n = {name: 0 for name in stage_sse}
    tail_md_bin_rows = []
    tail_row_bin_rows = []
    well_gain_rows = []

    md_bins = [0, 10, 50, 100, 250, 500, 1000, 2000, 5000, np.inf]
    row_bins = [-0.5, 0.5, 4.5, 9.5, 24.5, 49.5, 99.5, 249.5, 499.5, 999.5, np.inf]
    row_labels = ['0', '1-4', '5-9', '10-24', '25-49', '50-99', '100-249', '250-499', '500-999', '1000+']

    for part in selected_fold_parts:
        frame = part['frame']
        y = frame['target_tvt'].to_numpy(dtype=float)
        anchor = frame['last_known_TVT'].to_numpy(dtype=float)
        clipped_delta = np.asarray(part['clipped_delta'], dtype=float)
        pred_delta_clipped = anchor + clipped_delta
        pred_global_alpha = anchor + shrink_delta(clipped_delta, alpha=SELECTED_SHRINKAGE_ALPHA)
        faded_delta = shrink_delta(
            fade_in_delta(frame, clipped_delta, SELECTED_FADE_IN_TAU_MD),
            alpha=SELECTED_SHRINKAGE_ALPHA,
        )
        pred_fade_in = anchor + faded_delta
        pred_selected = pred_fade_in.copy()
        if APPLY_SELECTED_SLOPE_CLIP and SELECTED_SLOPE_QUANTILE is not None:
            slope_bound = part['slope_bounds'][float(SELECTED_SLOPE_QUANTILE)]
            pred_selected = causal_slope_clip_by_well(frame, pred_selected, slope_bound)
        stage_predictions = {
            'constant_anchor': anchor,
            'delta_clipped': pred_delta_clipped,
            'global_alpha': pred_global_alpha,
            'fade_in': pred_fade_in,
            'selected_policy': pred_selected,
        }
        for stage_name, pred in stage_predictions.items():
            valid = np.isfinite(y) & np.isfinite(pred)
            err = y[valid] - pred[valid]
            stage_sse[stage_name] += float(np.dot(err, err))
            stage_n[stage_name] += int(valid.sum())
        diag = pd.DataFrame({
            'well_id': frame['well_id'].to_numpy(),
            'md_since_ps': frame['md_since_ps'].to_numpy(dtype=float),
            'tail_row_number': frame['tail_row_number'].to_numpy(dtype=float),
            'target_delta': frame['target_delta_from_last_known'].to_numpy(dtype=float),
            'constant_error_sq': (y - anchor) ** 2,
            'selected_error_sq': (y - pred_selected) ** 2,
        })
        diag['md_bin'] = pd.cut(diag['md_since_ps'], bins=md_bins, include_lowest=True)
        md_group = diag.groupby('md_bin', observed=False).agg(
            rows=('selected_error_sq', 'size'),
            constant_sse=('constant_error_sq', 'sum'),
            selected_sse=('selected_error_sq', 'sum'),
            target_delta_std=('target_delta', 'std'),
        ).reset_index()
        tail_md_bin_rows.append(md_group)
        diag['tail_row_bin'] = pd.cut(
            diag['tail_row_number'],
            bins=row_bins,
            labels=row_labels,
            include_lowest=True,
        )
        row_group = diag.groupby('tail_row_bin', observed=False).agg(
            rows=('selected_error_sq', 'size'),
            constant_sse=('constant_error_sq', 'sum'),
            selected_sse=('selected_error_sq', 'sum'),
            target_delta_std=('target_delta', 'std'),
        ).reset_index()
        tail_row_bin_rows.append(row_group)
        well_group = diag.groupby('well_id', as_index=False).agg(
            rows=('selected_error_sq', 'size'),
            constant_sse=('constant_error_sq', 'sum'),
            selected_sse=('selected_error_sq', 'sum'),
            target_delta_min=('target_delta', 'min'),
            target_delta_max=('target_delta', 'max'),
            target_delta_std=('target_delta', 'std'),
            max_md_since_ps=('md_since_ps', 'max'),
        )
        well_gain_rows.append(well_group)

    selected_policy_stage_report = pd.DataFrame([
        {
            'stage': stage_name,
            'rows': stage_n[stage_name],
            'rmse': float(np.sqrt(stage_sse[stage_name] / stage_n[stage_name])),
        }
        for stage_name in stage_sse
    ])
    constant_rmse = float(selected_policy_stage_report.loc[
        selected_policy_stage_report['stage'].eq('constant_anchor'),
        'rmse',
    ].iloc[0])
    selected_policy_stage_report['gain_vs_constant'] = constant_rmse - selected_policy_stage_report['rmse']


    def combine_bin_diagnostics(rows, bin_col):
        out = pd.concat(rows, ignore_index=True).groupby(bin_col, observed=False).agg(
            rows=('rows', 'sum'),
            constant_sse=('constant_sse', 'sum'),
            selected_sse=('selected_sse', 'sum'),
            target_delta_std=('target_delta_std', 'mean'),
        ).reset_index()
        out['constant_rmse'] = np.sqrt(out['constant_sse'] / out['rows'])
        out['selected_rmse'] = np.sqrt(out['selected_sse'] / out['rows'])
        out['gain_vs_constant'] = out['constant_rmse'] - out['selected_rmse']
        out[bin_col] = out[bin_col].astype(str)
        return out[[bin_col, 'rows', 'constant_rmse', 'selected_rmse', 'gain_vs_constant', 'target_delta_std']]

    selected_policy_tail_md_diagnostics = combine_bin_diagnostics(tail_md_bin_rows, 'md_bin')
    selected_policy_tail_row_diagnostics = combine_bin_diagnostics(tail_row_bin_rows, 'tail_row_bin')

    selected_policy_well_gain = pd.concat(well_gain_rows, ignore_index=True)
    selected_policy_well_gain['constant_rmse'] = np.sqrt(
        selected_policy_well_gain['constant_sse'] / selected_policy_well_gain['rows']
    )
    selected_policy_well_gain['selected_rmse'] = np.sqrt(
        selected_policy_well_gain['selected_sse'] / selected_policy_well_gain['rows']
    )
    selected_policy_well_gain['gain_vs_constant'] = (
        selected_policy_well_gain['constant_rmse'] - selected_policy_well_gain['selected_rmse']
    )
    selected_policy_well_gain['target_delta_range'] = (
        selected_policy_well_gain['target_delta_max'] - selected_policy_well_gain['target_delta_min']
    )

    well_gain_distribution = selected_policy_well_gain['gain_vs_constant'].describe(
        percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    ).to_frame('gain_vs_constant')
    well_gain_distribution.loc['improved_well_count', 'gain_vs_constant'] = float((selected_policy_well_gain['gain_vs_constant'] > 0).sum())
    well_gain_distribution.loc['hurt_well_count', 'gain_vs_constant'] = float((selected_policy_well_gain['gain_vs_constant'] < 0).sum())

    artifact_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR
    selected_policy_stage_report.to_csv(artifact_dir / 'v2_selected_policy_stage_report.csv', index=False)
    selected_policy_tail_md_diagnostics.to_csv(artifact_dir / 'v2_selected_policy_tail_md_diagnostics.csv', index=False)
    selected_policy_tail_row_diagnostics.to_csv(artifact_dir / 'v2_selected_policy_tail_row_diagnostics.csv', index=False)
    selected_policy_well_gain.to_csv(artifact_dir / 'v2_selected_policy_well_gain.csv', index=False)

    print('Selected diagnostic feature set:', SELECTED_FEATURE_SET)
    print('Selected diagnostic artifacts written to:', artifact_dir)
    display(selected_policy_stage_report.style.format({
        'rmse': '{:.4f}',
        'gain_vs_constant': '{:.4f}',
    }))
    display(selected_policy_tail_md_diagnostics.style.format({
        'constant_rmse': '{:.4f}',
        'selected_rmse': '{:.4f}',
        'gain_vs_constant': '{:.4f}',
        'target_delta_std': '{:.4f}',
    }))
    display(well_gain_distribution.style.format({'gain_vs_constant': '{:.4f}'}))
    display(selected_policy_well_gain.sort_values('gain_vs_constant').head(8)[[
        'well_id',
        'rows',
        'constant_rmse',
        'selected_rmse',
        'gain_vs_constant',
        'target_delta_range',
    ]].style.format({
        'constant_rmse': '{:.4f}',
        'selected_rmse': '{:.4f}',
        'gain_vs_constant': '{:.4f}',
        'target_delta_range': '{:.4f}',
    }))


### 16.3 Prediction Table Construction

### Prediction rule

For each test row, the prediction starts from `last_known_TVT` and adds the selected residual correction. The correction is clipped, globally shrunk, optionally faded in near Prediction Start, and optionally passed through the anchor-aware slope limiter.

<details>
<summary>Compact prediction equation</summary>

$$
\hat{y}_{w,i}
=
\mathcal{S}_q\!\left(
\mathrm{TVT}_{\mathrm{input},w,L}
+
\rho_{w,i}(\tau)\,\alpha\,\mathrm{clip}(\widehat{\Delta y}_{w,i})
\right),
$$

where $\mathcal{S}_q$ is the anchor-aware slope limiter when selected by CV, and the identity map otherwise.

</details>

### Output tables from the HGB residual pipeline

| Prediction table | Training wells | Diagnostic meaning |
|---|---:|---|
| selected all-train | 773 | selected strict feature policy with maximum HGB training data |
| selected public-clean | 770 | excludes visible test-overlap wells for public-LB interpretation |
| best-overall all-train | 773 | best stored-CV HGB policy, including offline features if selected |



In [ ]:
# Fit residual models and write id/tvt prediction files.

required_selection_vars = [
    'SELECTED_FEATURE_SET',
    'SELECTED_POLICY_METRIC',
    'SELECTED_SHRINKAGE_ALPHA',
    'SELECTED_FADE_IN_TAU_MD',
    'SELECTED_SLOPE_QUANTILE',
    'APPLY_SELECTED_SLOPE_CLIP',
    'BEST_OVERALL_FEATURE_SET',
    'BEST_OVERALL_SHRINKAGE_ALPHA',
    'BEST_OVERALL_FADE_IN_TAU_MD',
    'BEST_OVERALL_SLOPE_QUANTILE',
    'APPLY_BEST_OVERALL_SLOPE_CLIP',
]
missing_selection_vars = [name for name in required_selection_vars if name not in globals()]
if missing_selection_vars:
    raise RuntimeError(f'CV selection variables are missing: {missing_selection_vars}. Run the validation cell before submission construction.')

all_train_wells = np.array(sorted(train_h_ids))

final_model_settings = pd.Series({
    'selected_feature_set': SELECTED_FEATURE_SET,
    'selected_feature_policy': FEATURE_SET_POLICY[SELECTED_FEATURE_SET],
    'selected_feature_count': len(FEATURE_SETS[SELECTED_FEATURE_SET]),
    'selected_policy_metric': SELECTED_POLICY_METRIC,
    'selected_shrinkage_alpha': SELECTED_SHRINKAGE_ALPHA,
    'selected_fade_in_tau_md': SELECTED_FADE_IN_TAU_MD,
    'apply_causal_slope_clip': APPLY_SELECTED_SLOPE_CLIP,
    'selected_slope_quantile': SELECTED_SLOPE_QUANTILE,
    'best_overall_feature_set': BEST_OVERALL_FEATURE_SET,
    'best_overall_feature_policy': FEATURE_SET_POLICY[BEST_OVERALL_FEATURE_SET],
    'best_overall_shrinkage_alpha': BEST_OVERALL_SHRINKAGE_ALPHA,
    'best_overall_fade_in_tau_md': BEST_OVERALL_FADE_IN_TAU_MD,
    'best_overall_apply_slope_clip': APPLY_BEST_OVERALL_SLOPE_CLIP,
    'best_overall_slope_quantile': BEST_OVERALL_SLOPE_QUANTILE,
}).to_frame('value')
display(final_model_settings)

visible_test_overlap_wells = np.array(sorted(set(train_h_ids) & set(test_h_ids)))
public_clean_train_wells = np.array(sorted(set(all_train_wells) - set(visible_test_overlap_wells)))
prediction_file_summaries = []


def train_and_predict_submission(
    training_wells,
    output_path: Path,
    label: str,
    feature_set_name: str,
    alpha: float,
    fade_tau_md: float | None,
    slope_quantile: float | None,
    apply_slope_clip: bool,
):
    feature_columns = FEATURE_SETS[feature_set_name]
    final_plane_imputer = final_row_imputer = None
    if feature_columns_require_formation(feature_columns):
        final_plane_imputer, final_row_imputer = make_formation_imputers(
            training_wells,
            TRAIN_DIR,
            need_row_ancc=feature_columns_require_row_ancc(feature_columns),
            seed=MODEL_CONFIG['random_state'] + 999,
        )
    train_frame = build_tail_feature_frame(
        training_wells,
        TRAIN_DIR,
        include_target=True,
        rows_per_well=MODEL_CONFIG['final_train_rows_per_well'],
        random_state=MODEL_CONFIG['random_state'] + 999,
        use_beam_features=feature_columns_require_beam(feature_columns),
        keep_columns=feature_columns,
        formation_plane_imputer=final_plane_imputer,
        row_ancc_imputer=final_row_imputer,
        imputer_well_ids=training_wells,
    )
    X_train = train_frame[feature_columns].copy()
    y_train_delta = train_frame['target_delta_from_last_known'].to_numpy(dtype=float)
    model = make_residual_model(random_state=MODEL_CONFIG['random_state'] + 999)
    model.fit(X_train, y_train_delta)
    
    if not test_horizontal_files:
        prediction_file_summaries.append({
            'label': label,
            'feature_set': feature_set_name,
            'train_wells': len(training_wells),
            'train_rows': len(train_frame),
            'prediction_rows': 0,
            'output_file': None,
            'missing_predictions': np.nan,
        })
        return None
    
    test_wells = sorted(test_h_ids)
    test_frame = build_tail_feature_frame(
        test_wells,
        TEST_DIR,
        include_target=False,
        rows_per_well=None,
        random_state=MODEL_CONFIG['random_state'] + 2026,
        use_beam_features=feature_columns_require_beam(feature_columns),
        keep_columns=feature_columns,
        formation_plane_imputer=final_plane_imputer,
        row_ancc_imputer=final_row_imputer,
        imputer_well_ids=training_wells,
    )
    test_delta_raw = model.predict(test_frame[feature_columns])
    test_delta_clipped = clip_delta_by_train_quantiles(test_delta_raw, y_train_delta)
    test_delta_final = shrink_delta(
        fade_in_delta(test_frame, test_delta_clipped, fade_tau_md),
        alpha=alpha,
    )
    test_pred = test_frame['last_known_TVT'].to_numpy(dtype=float) + test_delta_final
    max_abs_slope = np.nan
    if apply_slope_clip and slope_quantile is not None:
        training_slope_paths = [TRAIN_DIR / f'{well_id}__horizontal_well.csv' for well_id in training_wells]
        training_slope_bounds = estimate_abs_tvt_slope_quantiles(
            training_slope_paths,
            quantiles=[float(slope_quantile)],
        )
        max_abs_slope = training_slope_bounds[float(slope_quantile)]
        test_pred = causal_slope_clip_by_well(test_frame, test_pred, max_abs_slope)
    submission = pd.DataFrame({'id': test_frame['id'].to_numpy(), 'tvt': test_pred})
    
    if SAMPLE_SUBMISSION.exists():
        sample = pd.read_csv(SAMPLE_SUBMISSION)
        submission = sample[['id']].merge(submission, on='id', how='left')
        missing = int(submission['tvt'].isna().sum())
        if missing:
            missing_ids = submission.loc[submission['tvt'].isna(), 'id'].head(10).tolist()
            raise ValueError(f'Missing predictions after sample alignment: {missing}. First missing ids: {missing_ids}')
    else:
        missing = int(submission['tvt'].isna().sum())
    
    if MODEL_CONFIG['write_submission']:
        submission.to_csv(output_path, index=False)
    prediction_file_summaries.append({
        'label': label,
        'feature_set': feature_set_name,
        'feature_policy': FEATURE_SET_POLICY[feature_set_name],
        'alpha': alpha,
        'fade_tau_md': fade_tau_md,
        'slope_quantile': slope_quantile,
        'apply_slope_clip': apply_slope_clip,
        'max_abs_slope_used': max_abs_slope,
        'train_wells': len(training_wells),
        'train_rows': len(train_frame),
        'target_delta_mean': float(np.mean(y_train_delta)),
        'target_delta_std': float(np.std(y_train_delta)),
        'prediction_rows': len(submission),
        'missing_predictions': missing,
        'tvt_mean': float(submission['tvt'].mean()),
        'tvt_std': float(submission['tvt'].std()),
        'tvt_min': float(submission['tvt'].min()),
        'tvt_max': float(submission['tvt'].max()),
        'output_file': output_path.name,
    })
    return submission

KAGGLE_WORKING_DIR = Path('/kaggle/working')
OUTPUT_DIR = KAGGLE_WORKING_DIR if KAGGLE_WORKING_DIR.exists() else DATA_DIR
selected_main_file = (
    'submission_hgb_strict_v2.csv'
    if KAGGLE_WORKING_DIR.exists()
    else 'submission_simple_residual_v2.csv'
)
selected_public_clean_file = (
    'submission_hgb_public_clean_v2.csv'
    if KAGGLE_WORKING_DIR.exists()
    else 'submission_simple_residual_public_clean_v2.csv'
)
best_overall_file = (
    'submission_hgb_offline_candidate_path_v2.csv'
    if KAGGLE_WORKING_DIR.exists()
    else 'submission_simple_residual_best_overall_v2.csv'
)

if RUN_HGB_DIAGNOSTIC_SUBMISSIONS:
    submission_all_train = train_and_predict_submission(
        all_train_wells,
        OUTPUT_DIR / selected_main_file,
        'selected_all_train',
        SELECTED_FEATURE_SET,
        SELECTED_SHRINKAGE_ALPHA,
        SELECTED_FADE_IN_TAU_MD,
        SELECTED_SLOPE_QUANTILE,
        APPLY_SELECTED_SLOPE_CLIP,
    )

    if MODEL_CONFIG['write_public_clean_submission'] and len(public_clean_train_wells):
        submission_public_clean = train_and_predict_submission(
            public_clean_train_wells,
            OUTPUT_DIR / selected_public_clean_file,
            'selected_public_clean',
            SELECTED_FEATURE_SET,
            SELECTED_SHRINKAGE_ALPHA,
            SELECTED_FADE_IN_TAU_MD,
            SELECTED_SLOPE_QUANTILE,
            APPLY_SELECTED_SLOPE_CLIP,
        )

    if BEST_OVERALL_FEATURE_SET != SELECTED_FEATURE_SET:
        submission_best_overall = train_and_predict_submission(
            all_train_wells,
            OUTPUT_DIR / best_overall_file,
            'best_overall_all_train',
            BEST_OVERALL_FEATURE_SET,
            BEST_OVERALL_SHRINKAGE_ALPHA,
            BEST_OVERALL_FADE_IN_TAU_MD,
            BEST_OVERALL_SLOPE_QUANTILE,
            APPLY_BEST_OVERALL_SLOPE_CLIP,
        )

else:
    print('HGB diagnostic submissions skipped in memory-safe Kaggle mode.')
    submission_all_train = None

prediction_file_summary = pd.DataFrame(prediction_file_summaries)
prediction_file_summary.to_csv(OUTPUT_DIR / 'v2_prediction_file_summary.csv', index=False)
display(prediction_file_summary)
if submission_all_train is not None:
    display(submission_all_train.head())


### 16.4 Statistical Extensions

### Pipeline contract

| Stage | Statistical role |
|---|---|
| Well files | observed trajectory, GR, prefix TVT, and typewell reference |
| Feature policy | separates strict drilling-time inputs from Kaggle offline inputs |
| Feature table | converts each prediction row into leakage-checked covariates |
| Residual model | predicts movement away from `last_known_TVT` |
| Global postprocess | shrinkage, fade-in, and optional slope limiting |
| Prediction file | aligned `id,tvt` table for submission |

### Candidate signal families

| Candidate | Statistical purpose | Leakage condition |
|---|---|---|
| Candidate-path typewell endpoints | compare plausible offline tail trajectories against typewell GR and boundary context | offline only, because tail fraction is known only in batch mode |
| Formation-top plane/KNN references | estimate spatial formation geometry and plane-style TVT from train surfaces | fold-safe imputer; no direct surface columns in hidden test |
| Multi-prior beam alignment | turn GR matching into a constrained sequence-path signal | offline only; uses hidden GR, not hidden TVT |
| LightGBM / XGBoost / CatBoost | stronger tabular residual learners | same feature policies |
| Top2/plane blend | combine GR-sequence and spatial-formation models with different failure modes | use OOF or conservative target-free gates |
| Trailing-window typewell correlation | compare local GR shapes | strict if trailing only |
| Typewell `Geology` encoding | categorical stratigraphic context | fold-safe encoding or CatBoost |
| Knot / curve model | reduce row-wise noise | target knots are labels only |
| Residual smoothing | enforce curve smoothness | offline unless causal |
| Sample-weighted training | align sampled training rows with row-level RMSE | use tail length, not target values |


## 17. Offline Candidate-Path Model Check

This section keeps the strict residual pipeline intact and isolates batch-only feature families.

### 🌐 Offline signal map

| Addition | Policy | Statistical role |
|---|---:|---|
| Recent prefix step/slope | ✅ strict-safe | short-horizon TVT behavior before Prediction Start |
| Typewell anchor-offset residuals | ✅ strict-safe | current GR vs nearby typewell anchors |
| Position / gap / GR distribution | 🌐 offline | full hidden interval shape and texture |
| Centered GR texture + lead/lag | 🌐 offline | local batch-only GR shape |
| Typewell boundary phase | ✅ strict-safe | interval position in typewell Geology |
| Linear/eased candidate paths | 🌐 offline | plausible tail-end TVT shifts against typewell GR |
| Formation-plane imputation | 🌐 reference | spatial formation geometry projected onto hidden rows |
| Beam-path alignment | 🧪 offline | full hidden GR sequence as a constrained path |
| Stronger tabular models | ✅/🌐 same policy | LGBM/XGB/CatBoost residual learners |

### 🛡️ Leakage guardrails

| Feature family | Uses future GR/geometry? | Uses future TVT? | Uses direct train-only surfaces? |
|---|---:|---:|---:|
| Candidate paths | ✅ | 🚫 | 🚫 |
| Beam paths | ✅ | 🚫 | 🚫 |
| Formation-plane outputs | target-free coordinates | 🚫 | 🚫 as direct test features |

Formation-plane features use train-only surfaces only as **auxiliary labels** in a spatial imputer. In grouped CV, validation wells are excluded from the imputer fit.

### ⚙️ Runtime policy

| Component | Default | Reason |
|---|---:|---|
| HGB candidate-path CV | on locally | reproducible offline feature ablation |
| strong-model experiment | off | heavier model-family comparison |
| beam features | controlled separately | repeated GroupKFold feature construction is expensive |


In [ ]:
# Optional comparison: offline candidate-path features with a stronger residual model.

RUN_V2_STRONG_MODEL_EXPERIMENT = False

V2_STRONG_MODEL_CONFIG = {
    'feature_sets': ['offline_candidate_path_alignment'],
    'models': ['lightgbm'],
    'folds_to_run': min(2, MODEL_CONFIG['cv_folds_to_run']),
    'train_rows_per_well': 350,
    'random_state': MODEL_CONFIG['random_state'] + 2606,
}


def available_strong_models() -> dict[str, object]:
    models = {}
    try:
        from lightgbm import LGBMRegressor
        models['lightgbm'] = LGBMRegressor
    except Exception as exc:
        print(f'LightGBM unavailable: {exc}')
    try:
        from xgboost import XGBRegressor
        models['xgboost'] = XGBRegressor
    except Exception as exc:
        print(f'XGBoost unavailable: {exc}')
    try:
        from catboost import CatBoostRegressor
        models['catboost'] = CatBoostRegressor
    except Exception as exc:
        print(f'CatBoost unavailable: {exc}')
    return models


def make_strong_residual_model(model_name: str, random_state: int):
    registry = available_strong_models()
    if model_name == 'lightgbm' and 'lightgbm' in registry:
        LGBMRegressor = registry['lightgbm']
        params = dict(
            objective='regression_l2',
            metric='rmse',
            n_estimators=500,
            learning_rate=0.035,
            num_leaves=63,
            min_child_samples=80,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.80,
            reg_alpha=0.05,
            reg_lambda=1.0,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        params.update(lightgbm_accelerator_params())
        return LGBMRegressor(**params)
    if model_name == 'xgboost' and 'xgboost' in registry:
        XGBRegressor = registry['xgboost']
        params = dict(
            objective='reg:squarederror',
            eval_metric='rmse',
            n_estimators=700,
            learning_rate=0.035,
            max_depth=7,
            min_child_weight=20,
            subsample=0.80,
            colsample_bytree=0.80,
            reg_alpha=1.0,
            reg_lambda=20.0,
            tree_method='hist',
            random_state=random_state,
            n_jobs=-1,
        )
        if LIGHTGBM_DEVICE_TYPE == 'gpu':
            params['device'] = 'cuda'
        return XGBRegressor(**params)
    if model_name == 'catboost' and 'catboost' in registry:
        CatBoostRegressor = registry['catboost']
        return CatBoostRegressor(
            loss_function='RMSE',
            iterations=900,
            learning_rate=0.035,
            depth=8,
            l2_leaf_reg=8.0,
            random_seed=random_state,
            task_type='GPU' if LIGHTGBM_DEVICE_TYPE == 'gpu' else 'CPU',
            verbose=False,
            allow_writing_files=False,
        )
    raise ValueError(f'Model is not available or not configured: {model_name}')


def run_v2_strong_model_cv() -> pd.DataFrame:
    available = available_strong_models()
    rows = []
    strong_fold_parts = []
    selected_feature_sets = [name for name in V2_STRONG_MODEL_CONFIG['feature_sets'] if name in FEATURE_SETS]
    selected_models = [name for name in V2_STRONG_MODEL_CONFIG['models'] if name in available]
    if not selected_feature_sets or not selected_models:
        return pd.DataFrame(columns=['model', 'feature_set', 'rmse', 'alpha', 'fade_tau_md', 'slope_quantile', 'n_rows'])

    for feature_name in selected_feature_sets:
        feature_cols = FEATURE_SETS[feature_name]
        for model_name in selected_models:
            fold_parts = []
            for fold_idx, (train_idx, valid_idx) in enumerate(fold_splits[:V2_STRONG_MODEL_CONFIG['folds_to_run']], start=1):
                train_wells = all_train_wells[train_idx]
                valid_wells = all_train_wells[valid_idx]
                fold_plane_imputer = fold_row_imputer = None
                if feature_columns_require_formation(feature_cols):
                    fold_plane_imputer, fold_row_imputer = make_formation_imputers(
                        train_wells,
                        TRAIN_DIR,
                        need_row_ancc=feature_columns_require_row_ancc(feature_cols),
                        seed=V2_STRONG_MODEL_CONFIG['random_state'] + fold_idx,
                    )
                train_frame = build_tail_feature_frame(
                    train_wells,
                    TRAIN_DIR,
                    include_target=True,
                    rows_per_well=V2_STRONG_MODEL_CONFIG['train_rows_per_well'],
                    random_state=V2_STRONG_MODEL_CONFIG['random_state'] + fold_idx,
                    use_beam_features=feature_columns_require_beam(feature_cols),
                    keep_columns=feature_cols,
                    formation_plane_imputer=fold_plane_imputer,
                    row_ancc_imputer=fold_row_imputer,
                    imputer_well_ids=train_wells,
                )
                valid_frame = build_tail_feature_frame(
                    valid_wells,
                    TRAIN_DIR,
                    include_target=True,
                    rows_per_well=None,
                    random_state=V2_STRONG_MODEL_CONFIG['random_state'] + fold_idx,
                    use_beam_features=feature_columns_require_beam(feature_cols),
                    keep_columns=feature_cols,
                    formation_plane_imputer=fold_plane_imputer,
                    row_ancc_imputer=fold_row_imputer,
                    imputer_well_ids=train_wells,
                )
                model = make_strong_residual_model(model_name, V2_STRONG_MODEL_CONFIG['random_state'] + fold_idx)
                X_train, y_train_delta, _ = model_ready_xy(train_frame, feature_cols)
                X_valid, _, _ = model_ready_xy(valid_frame, feature_cols)
                model.fit(X_train, y_train_delta)
                raw_delta = model.predict(X_valid)
                clipped_delta = clip_delta_by_train_quantiles(raw_delta, y_train_delta)
                fold_train_paths = [TRAIN_DIR / f'{well_id}__horizontal_well.csv' for well_id in train_wells]
                slope_bounds = estimate_abs_tvt_slope_quantiles(fold_train_paths, quantiles=slope_quantiles)
                fold_parts.append({
                    'fold': fold_idx,
                    'frame': valid_frame,
                    'raw_delta': np.asarray(raw_delta, dtype=float),
                    'clipped_delta': np.asarray(clipped_delta, dtype=float),
                    'slope_bounds': slope_bounds,
                })
                rows.append({
                    'model': model_name,
                    'feature_set': feature_name,
                    'fold': fold_idx,
                    'n_train_rows': len(train_frame),
                    'n_valid_rows': len(valid_frame),
                    'raw_rmse': rmse(
                        valid_frame['target_tvt'].to_numpy(dtype=float),
                        valid_frame['last_known_TVT'].to_numpy(dtype=float) + raw_delta,
                    ),
                    'clipped_rmse': rmse(
                        valid_frame['target_tvt'].to_numpy(dtype=float),
                        valid_frame['last_known_TVT'].to_numpy(dtype=float) + clipped_delta,
                    ),
                })

            policy_rows = []
            for tau in MODEL_CONFIG['fade_in_tau_md_to_compare']:
                alpha, _ = fit_global_alpha_from_fold_parts(fold_parts, tau_md=tau)
                score = score_policy_from_fold_parts(
                    fold_parts,
                    alpha=alpha,
                    tau_md=tau,
                    slope_quantile=None,
                )
                score.update({'alpha': alpha, 'tau_md': tau, 'slope_quantile': np.nan})
                policy_rows.append(score)
                for slope_q in MODEL_CONFIG['slope_clip_quantiles_to_compare']:
                    score = score_policy_from_fold_parts(
                        fold_parts,
                        alpha=alpha,
                        tau_md=tau,
                        slope_quantile=float(slope_q),
                    )
                    score.update({'alpha': alpha, 'tau_md': tau, 'slope_quantile': float(slope_q)})
                    policy_rows.append(score)
            policy = pd.DataFrame(policy_rows).sort_values('rmse').reset_index(drop=True)
            best = policy.iloc[0].to_dict()
            strong_fold_parts.append({
                'model': model_name,
                'feature_set': feature_name,
                'fold_parts': fold_parts,
                'best_policy': best,
            })
            rows.append({
                'model': model_name,
                'feature_set': feature_name,
                'fold': 'global_policy',
                'n_train_rows': np.nan,
                'n_valid_rows': int(sum(len(part['frame']) for part in fold_parts)),
                'raw_rmse': np.nan,
                'clipped_rmse': np.nan,
                'best_policy_rmse': best['rmse'],
                'best_policy_alpha': best['alpha'],
                'best_policy_fade_tau_md': best['tau_md'],
                'best_policy_slope_quantile': best['slope_quantile'],
            })

    result = pd.DataFrame(rows)
    globals()['v2_strong_fold_parts'] = strong_fold_parts
    return result


if RUN_V2_STRONG_MODEL_EXPERIMENT:
    v2_strong_model_report = run_v2_strong_model_cv()
else:
    v2_strong_model_report = pd.DataFrame()

if len(v2_strong_model_report):
    v2_strong_model_report.to_csv(OUTPUT_DIR / 'v2_strong_model_cv_report.csv', index=False)
    display(v2_strong_model_report.tail(10))
else:
    print('No v2 strong-model experiment was run. Check availability and configuration.')


## 18. Optional All-Row GPU LightGBM Candidate

### 🚀 Purpose

| Goal | Design choice |
|---|---|
| use all train tail rows | train LightGBM on millions of residual rows |
| avoid Kaggle OOM | compact feature set, float32 matrices, reduced GPU bins/leaves |
| keep a valid file even on failure | write constant-anchor fallback before training |
| make the default score-seeking | default `submission.csv` comes from the public-aggressive formation blend |
| test formation safely | write private-safe and public-aggressive formation probe files separately |

### 🧭 Run flow

```text
constant fallback
      ↓
build compact all-row public-style features
      ↓
GPU LightGBM residual model
      ↓
raw / mild / stored-policy candidates
      ↓
optional formation probe candidates
      ↓
copy FINAL_LGBM_CANDIDATE → submission.csv
```

### ⚙️ Memory-safe defaults

| Setting | Default | Reason |
|---|---:|---|
| `KAGGLE_MEMORY_SAFE_MODE` | `True` | avoid wide all-row feature tables and reduce LightGBM bins/leaves |
| `ENABLE_KAGGLE_BEAM_FEATURES` | `False` | beam features are expensive on full rows |
| `ENABLE_KAGGLE_FORMATION_FEATURES` | `False` | keep the model matrix compact by default |
| `WRITE_FORMATION_PROBE_SUBMISSIONS` | `True` | still write formation formula/blend probe files from test-only aux features |
| `PUBLIC_AGGRESSIVE_FORMATION` | `True` | also write a same-well-allowed public diagnostic candidate |
| `ALL_ROW_LGBM_FEATURE_SET` | `offline_public_lgbm_style` | compact public-style offline features |
| `FINAL_LGBM_CANDIDATE` | `plane_blend_public_aggressive` | score-seeking blend of mild LightGBM and same-well-allowed formation formula |
| `REQUIRE_FINAL_LGBM_CANDIDATE` | `True` | fail loudly if the intended final candidate is not produced |

Recommended Kaggle accelerator: **P100 first**, with **T4 x2** as a fallback.

### 📦 Candidate files

| Candidate | File | Purpose |
|---|---|---|
| fallback | `submission_constant_fallback_v2.csv` | last-known TVT anchor safety net |
| raw | `submission_lgbm_raw_v2.csv` | raw all-row LightGBM residual |
| mild | `submission_lgbm_mild_v2.csv` | safer compact LightGBM candidate |
| stored-HGB-policy | `submission_lgbm_hgb_policy_v2.csv` | applies stored HGB offline postprocess to LGBM residual |
| formation private-safe | `submission_formation_plane_formula_private_safe_v2.csv` | formation formula with same-well exclusion |
| blend private-safe | `submission_lgbm_plane_blend_private_safe_v2.csv` | mild LGBM blended with private-safe formation formula |
| formation public-aggressive | `submission_formation_plane_formula_public_aggressive_v2.csv` | formation formula allowing same-well surface references |
| blend public-aggressive | `submission_lgbm_plane_blend_public_aggressive_v2.csv` | mild LGBM blended with public-aggressive formation formula |
| legacy aliases | `submission_formation_plane_formula_v2.csv`, `submission_lgbm_plane_blend_safe_v2.csv` | private-safe aliases for compatibility |

### ⚠️ Interpretation

| Point | Meaning |
|---|---|
| public-aggressive blend is the default | it targets the visible public-test overlap signal; compare against `mild` for safety |
| formation files are written separately | `mild`, private-safe, and formula-only files remain available for comparison |
| final candidate is required | if public-aggressive blend is missing, the run raises instead of silently submitting `mild` |
| public-aggressive is diagnostic | useful for public LB probing, less informative for private generalization |
| gate is conservative | it now reduces plane weight when plane and mild LGBM strongly disagree |
| beam is opt-in | test only after the compact and formation branches are stable |


In [ ]:
# Optional all-row GPU LightGBM candidate. Local runs stay off; Kaggle GPU runs create final submission.csv.

RUN_ALL_ROW_LIGHTGBM_CANDIDATE = KAGGLE_NOTEBOOK_RUN
# Use the compact public-style LightGBM features by default. Formation-plane and
# beam variants remain explicit experiments.
ENABLE_KAGGLE_FORMATION_FEATURES = False
WRITE_FORMATION_PROBE_SUBMISSIONS = True
PUBLIC_AGGRESSIVE_FORMATION = True
ALL_ROW_LGBM_FEATURE_SET = (
    'offline_public_lgbm_formation_style'
    if (
        KAGGLE_MEMORY_SAFE_MODE
        and ENABLE_KAGGLE_FORMATION_FEATURES
        and 'offline_public_lgbm_formation_style' in FEATURE_SETS
    )
    else (
        'offline_public_lgbm_style'
        if KAGGLE_MEMORY_SAFE_MODE and 'offline_public_lgbm_style' in FEATURE_SETS
        else (
            'offline_beam_candidate_path_alignment'
            if ENABLE_OFFLINE_BEAM_FEATURES and 'offline_beam_candidate_path_alignment' in FEATURE_SETS
            else BEST_OVERALL_FEATURE_SET
        )
    )
)
FINAL_LGBM_CANDIDATE = 'plane_blend_public_aggressive'
REQUIRE_FINAL_LGBM_CANDIDATE = True
LGBM_CANDIDATE_FILES = {
    'raw': OUTPUT_DIR / 'submission_lgbm_raw_v2.csv',
    'mild': OUTPUT_DIR / 'submission_lgbm_mild_v2.csv',
    'stored_hgb_policy': OUTPUT_DIR / 'submission_lgbm_hgb_policy_v2.csv',
    'formation_formula': OUTPUT_DIR / 'submission_formation_plane_formula_v2.csv',
    'plane_blend_safe': OUTPUT_DIR / 'submission_lgbm_plane_blend_safe_v2.csv',
    'formation_formula_private_safe': OUTPUT_DIR / 'submission_formation_plane_formula_private_safe_v2.csv',
    'plane_blend_private_safe': OUTPUT_DIR / 'submission_lgbm_plane_blend_private_safe_v2.csv',
    'formation_formula_public_aggressive': OUTPUT_DIR / 'submission_formation_plane_formula_public_aggressive_v2.csv',
    'plane_blend_public_aggressive': OUTPUT_DIR / 'submission_lgbm_plane_blend_public_aggressive_v2.csv',
}
FINAL_SUBMISSION_OUTPUT = OUTPUT_DIR / 'submission.csv'


def make_lgb_residual_model(seed: int = 42):
    try:
        from lightgbm import LGBMRegressor
    except Exception as exc:
        raise RuntimeError(f'LightGBM is not available: {exc}') from exc
    memory_safe = bool(globals().get('KAGGLE_MEMORY_SAFE_MODE', False))
    params = dict(
        objective='regression_l2',
        metric='rmse',
        n_estimators=1200 if memory_safe else 1700,
        learning_rate=0.03 if memory_safe else 0.025,
        num_leaves=63 if memory_safe else 128,
        min_child_samples=100 if memory_safe else 80,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.70 if memory_safe else 0.75,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )
    params.update(lightgbm_accelerator_params())
    return LGBMRegressor(**params)


def align_submission_to_sample(prediction_frame: pd.DataFrame) -> pd.DataFrame:
    submission = prediction_frame[['id', 'tvt']].copy()
    if SAMPLE_SUBMISSION.exists():
        sample = pd.read_csv(SAMPLE_SUBMISSION)
        submission = sample[['id']].merge(submission, on='id', how='left')
    missing_predictions = int(submission['tvt'].isna().sum())
    if missing_predictions:
        missing_ids = submission.loc[submission['tvt'].isna(), 'id'].head(10).tolist()
        raise RuntimeError(f'Submission has missing predictions, for example: {missing_ids}')
    return submission


def write_constant_anchor_fallback_submission() -> pd.DataFrame:
    rows = []
    for well_id in sorted(test_h_ids):
        path = TEST_DIR / f'{well_id}__horizontal_well.csv'
        if not path.exists():
            continue
        df = pd.read_csv(path, usecols=lambda col: col in {'TVT_input'})
        mask = df['TVT_input'].isna().to_numpy()
        pred_idx = np.flatnonzero(mask)
        if len(pred_idx) == 0:
            continue
        ps = int(pred_idx[0])
        last_known = float(df['TVT_input'].iloc[ps - 1])
        rows.extend({'id': f'{well_id}_{idx}', 'tvt': last_known} for idx in pred_idx)
    fallback = align_submission_to_sample(pd.DataFrame(rows))
    fallback.to_csv(OUTPUT_DIR / 'submission_constant_fallback_v2.csv', index=False)
    fallback.to_csv(FINAL_SUBMISSION_OUTPUT, index=False)
    print('Constant-anchor fallback submission written:', FINAL_SUBMISSION_OUTPUT)
    return fallback


def postprocess_lgbm_delta(
    test_frame: pd.DataFrame,
    raw_delta: np.ndarray,
    train_delta: np.ndarray,
    *,
    clip_delta: bool,
    alpha: float,
    fade_tau_md: float | None,
    slope_quantile: float | None,
    slope_bounds: dict[float, float],
) -> np.ndarray:
    delta = np.asarray(raw_delta, dtype=float)
    if clip_delta:
        delta = clip_delta_by_train_quantiles(delta, train_delta)
    delta = shrink_delta(fade_in_delta(test_frame, delta, fade_tau_md), alpha=alpha)
    pred = test_frame['last_known_TVT'].to_numpy(dtype=float) + delta
    if slope_quantile is not None:
        pred = causal_slope_clip_by_well(test_frame, pred, slope_bounds[float(slope_quantile)])
    return pred


def formation_plane_gate(test_aux: pd.DataFrame, mild_pred: np.ndarray | None = None) -> np.ndarray:
    """Return a conservative row-wise blend weight for the formation-plane formula."""
    if test_aux.empty or 'formation_plane_tvt_formula' not in test_aux.columns:
        return np.zeros(len(test_aux), dtype=float)
    formula = test_aux['formation_plane_tvt_formula'].to_numpy(dtype=float)
    rmse = test_aux.get('formation_plane_prefix_rmse', pd.Series(np.nan, index=test_aux.index)).to_numpy(dtype=float)
    dist = test_aux.get('formation_plane_min_dist', pd.Series(np.nan, index=test_aux.index)).to_numpy(dtype=float)
    rmse_conf = 1.0 / (1.0 + (np.nan_to_num(rmse, nan=25.0, posinf=25.0) / 8.0) ** 2)
    dist_conf = 1.0 / (1.0 + (np.nan_to_num(dist, nan=2.0, posinf=2.0) / 1.2) ** 2)
    gate = 0.05 + 0.55 * rmse_conf * dist_conf
    if mild_pred is not None:
        diff = np.abs(formula - np.asarray(mild_pred, dtype=float))
        diff_conf = np.clip((35.0 - diff) / 25.0, 0.0, 1.0)
        gate *= diff_conf
    gate = np.clip(gate, 0.0, 0.55)
    gate[~np.isfinite(formula)] = 0.0
    return gate.astype(float)


def slope_clip_absolute_candidate(test_meta: pd.DataFrame, pred: np.ndarray, slope_bounds: dict[float, float], slope_quantile: float = 0.99) -> np.ndarray:
    if slope_quantile is None or float(slope_quantile) not in slope_bounds:
        return np.asarray(pred, dtype=float)
    return causal_slope_clip_by_well(test_meta, np.asarray(pred, dtype=float), slope_bounds[float(slope_quantile)])


def train_and_predict_all_row_lgbm():
    if not test_horizontal_files:
        print('No test files available; skipping all-row LightGBM candidate.')
        return None

    feature_columns = FEATURE_SETS[ALL_ROW_LGBM_FEATURE_SET]
    lgb_plane_imputer = lgb_row_imputer = None
    if feature_columns_require_formation(feature_columns):
        lgb_plane_imputer, lgb_row_imputer = make_formation_imputers(
            all_train_wells,
            TRAIN_DIR,
            need_row_ancc=feature_columns_require_row_ancc(feature_columns),
            seed=MODEL_CONFIG['random_state'] + 404,
        )
    train_frame = build_tail_feature_frame(
        all_train_wells,
        TRAIN_DIR,
        include_target=True,
        rows_per_well=None,
        random_state=MODEL_CONFIG['random_state'] + 404,
        use_beam_features=feature_columns_require_beam(feature_columns),
        keep_columns=feature_columns,
        formation_plane_imputer=lgb_plane_imputer,
        row_ancc_imputer=lgb_row_imputer,
        imputer_well_ids=all_train_wells,
        keep_metadata=False,
    )
    test_frame = build_tail_feature_frame(
        sorted(test_h_ids),
        TEST_DIR,
        include_target=False,
        rows_per_well=None,
        random_state=MODEL_CONFIG['random_state'] + 405,
        use_beam_features=feature_columns_require_beam(feature_columns),
        keep_columns=feature_columns,
        formation_plane_imputer=lgb_plane_imputer,
        row_ancc_imputer=lgb_row_imputer,
        imputer_well_ids=all_train_wells,
    )

    train_rows = len(train_frame)
    test_rows = len(test_frame)
    train_memory_mb = float(train_frame.memory_usage(deep=True).sum() / 1024**2)
    test_memory_mb = float(test_frame.memory_usage(deep=True).sum() / 1024**2)
    y_train_delta = train_frame['target_delta_from_last_known'].to_numpy(dtype=np.float32)
    X_train = train_frame[feature_columns].astype(np.float32, copy=False)
    x_train_memory_mb = float(X_train.memory_usage(deep=True).sum() / 1024**2)
    del train_frame
    gc.collect()

    test_meta_columns = ['id', 'well_id', 'row_index', 'MD', 'last_known_TVT', 'last_known_MD', 'md_since_ps']
    test_meta = test_frame[test_meta_columns].copy()
    formation_aux_columns = [
        col
        for col in [
            'formation_plane_tvt_formula',
            'formation_plane_delta_formula',
            'formation_plane_prefix_rmse',
            'formation_plane_prefix_mae',
            'formation_plane_min_dist',
        ]
        if col in test_frame.columns
    ]
    test_aux = test_frame[formation_aux_columns].copy() if formation_aux_columns else pd.DataFrame(index=test_frame.index)
    X_test = test_frame[feature_columns].astype(np.float32, copy=False)
    x_test_memory_mb = float(X_test.memory_usage(deep=True).sum() / 1024**2)
    del test_frame
    gc.collect()

    print('All-row LightGBM feature count:', len(feature_columns))
    print('Train compact frame memory MB:', round(train_memory_mb, 1))
    print('Train matrix memory MB:', round(x_train_memory_mb, 1))
    print('Test compact frame memory MB:', round(test_memory_mb, 1))
    print('Test matrix memory MB:', round(x_test_memory_mb, 1))

    model = make_lgb_residual_model(seed=MODEL_CONFIG['random_state'] + 404)
    model.fit(X_train, y_train_delta)
    del X_train
    gc.collect()
    raw_delta = model.predict(X_test)
    del X_test, model
    gc.collect()

    slope_quantiles_needed = [0.99]
    if APPLY_BEST_OVERALL_SLOPE_CLIP and BEST_OVERALL_SLOPE_QUANTILE is not None:
        slope_quantiles_needed.append(float(BEST_OVERALL_SLOPE_QUANTILE))
    slope_quantiles_needed = sorted(set(float(q) for q in slope_quantiles_needed))
    train_paths = [TRAIN_DIR / f'{well_id}__horizontal_well.csv' for well_id in all_train_wells]
    slope_bounds = estimate_abs_tvt_slope_quantiles(train_paths, quantiles=slope_quantiles_needed)

    candidate_policies = {
        'raw': {
            'clip_delta': False,
            'alpha': 1.0,
            'fade_tau_md': None,
            'slope_quantile': None,
            'description': 'raw LightGBM residual without residual clipping, fade-in, or slope limiting',
        },
        'mild': {
            'clip_delta': True,
            'alpha': 0.98,
            'fade_tau_md': 100.0,
            'slope_quantile': 0.99,
            'description': 'mild residual protection for stronger LightGBM residuals',
        },
        'stored_hgb_policy': {
            'clip_delta': True,
            'alpha': BEST_OVERALL_SHRINKAGE_ALPHA,
            'fade_tau_md': BEST_OVERALL_FADE_IN_TAU_MD,
            'slope_quantile': BEST_OVERALL_SLOPE_QUANTILE if APPLY_BEST_OVERALL_SLOPE_CLIP else None,
            'description': 'stored HGB offline post-processing policy applied to LightGBM residuals',
        },
    }

    summary_rows = []
    saved_candidates = {}
    candidate_predictions = {}
    for name, policy in candidate_policies.items():
        pred = postprocess_lgbm_delta(
            test_meta,
            raw_delta,
            y_train_delta,
            clip_delta=policy['clip_delta'],
            alpha=policy['alpha'],
            fade_tau_md=policy['fade_tau_md'],
            slope_quantile=policy['slope_quantile'],
            slope_bounds=slope_bounds,
        )
        prediction_frame = pd.DataFrame({'id': test_meta['id'].to_numpy(), 'tvt': pred})
        submission = align_submission_to_sample(prediction_frame)
        output_path = LGBM_CANDIDATE_FILES[name]
        submission.to_csv(output_path, index=False)
        saved_candidates[name] = output_path
        candidate_predictions[name] = np.asarray(pred, dtype=float)
        summary_rows.append({
            'candidate': name,
            'description': policy['description'],
            'feature_set': ALL_ROW_LGBM_FEATURE_SET,
            'postprocess_source_feature_set': BEST_OVERALL_FEATURE_SET if name == 'stored_hgb_policy' else None,
            'clip_delta': policy['clip_delta'],
            'alpha': policy['alpha'],
            'fade_tau_md': policy['fade_tau_md'],
            'slope_quantile': policy['slope_quantile'],
            'train_rows': train_rows,
            'test_rows': test_rows,
            'train_compact_frame_memory_mb': train_memory_mb,
            'train_matrix_memory_mb': x_train_memory_mb,
            'test_compact_frame_memory_mb': test_memory_mb,
            'test_matrix_memory_mb': x_test_memory_mb,
            'prediction_rows': len(submission),
            'missing_predictions': int(submission['tvt'].isna().sum()),
            'tvt_mean': float(submission['tvt'].mean()),
            'tvt_std': float(submission['tvt'].std()),
            'tvt_min': float(submission['tvt'].min()),
            'tvt_max': float(submission['tvt'].max()),
            'lightgbm_device_type': LIGHTGBM_DEVICE_TYPE,
            'output_file': str(output_path),
        })

    formation_probe_columns = [col for col in FORMATION_PLANE_COLUMNS if col in FEATURE_SETS.get('offline_public_lgbm_formation_style', FORMATION_PLANE_COLUMNS)]

    def build_formation_probe_aux(exclude_self: bool) -> pd.DataFrame:
        nonlocal lgb_plane_imputer
        if not WRITE_FORMATION_PROBE_SUBMISSIONS and 'formation_plane_tvt_formula' not in test_aux.columns:
            return pd.DataFrame(index=test_meta.index)
        if lgb_plane_imputer is None:
            lgb_plane_imputer, _ = make_formation_imputers(
                all_train_wells,
                TRAIN_DIR,
                need_row_ancc=False,
                seed=MODEL_CONFIG['random_state'] + 505,
            )
        probe_frame = build_tail_feature_frame(
            sorted(test_h_ids),
            TEST_DIR,
            include_target=False,
            rows_per_well=None,
            random_state=MODEL_CONFIG['random_state'] + (506 if exclude_self else 507),
            use_beam_features=False,
            keep_columns=formation_probe_columns,
            formation_plane_imputer=lgb_plane_imputer,
            row_ancc_imputer=None,
            imputer_well_ids=all_train_wells,
            exclude_query_well_from_formation=exclude_self,
        )
        available = ['id'] + [col for col in formation_probe_columns if col in probe_frame.columns]
        aligned = test_meta[['id']].merge(probe_frame[available], on='id', how='left')
        return aligned.drop(columns=['id'])

    def save_formation_candidates(
        aux: pd.DataFrame,
        label: str,
        formula_key: str,
        blend_key: str,
        alias_generic: bool = False,
    ):
        if aux.empty or 'formation_plane_tvt_formula' not in aux.columns or 'mild' not in candidate_predictions:
            return
        plane_raw = aux['formation_plane_tvt_formula'].to_numpy(dtype=float)
        mild_pred = candidate_predictions['mild']
        plane_pred = np.where(np.isfinite(plane_raw), plane_raw, mild_pred)
        plane_pred = slope_clip_absolute_candidate(test_meta, plane_pred, slope_bounds, slope_quantile=0.99)
        formula_submission = align_submission_to_sample(pd.DataFrame({'id': test_meta['id'].to_numpy(), 'tvt': plane_pred}))
        formula_keys = [formula_key] + (['formation_formula'] if alias_generic else [])
        for key in formula_keys:
            formula_path = LGBM_CANDIDATE_FILES[key]
            formula_submission.to_csv(formula_path, index=False)
            saved_candidates[key] = formula_path
            candidate_predictions[key] = plane_pred
        summary_rows.append({
            'candidate': formula_key,
            'description': f'{label} formation-plane closed-form TVT formula with anchor-aware slope guard',
            'feature_set': ALL_ROW_LGBM_FEATURE_SET,
            'postprocess_source_feature_set': f'formation_plane_imputer_{label}',
            'clip_delta': False,
            'alpha': np.nan,
            'fade_tau_md': np.nan,
            'slope_quantile': 0.99,
            'train_rows': train_rows,
            'test_rows': test_rows,
            'train_compact_frame_memory_mb': train_memory_mb,
            'train_matrix_memory_mb': x_train_memory_mb,
            'test_compact_frame_memory_mb': test_memory_mb,
            'test_matrix_memory_mb': x_test_memory_mb,
            'prediction_rows': len(formula_submission),
            'missing_predictions': int(formula_submission['tvt'].isna().sum()),
            'tvt_mean': float(formula_submission['tvt'].mean()),
            'tvt_std': float(formula_submission['tvt'].std()),
            'tvt_min': float(formula_submission['tvt'].min()),
            'tvt_max': float(formula_submission['tvt'].max()),
            'lightgbm_device_type': LIGHTGBM_DEVICE_TYPE,
            'output_file': str(LGBM_CANDIDATE_FILES[formula_key]),
            'formation_self_exclusion': label,
        })

        gate = formation_plane_gate(aux, mild_pred=mild_pred)
        blend_pred = (1.0 - gate) * mild_pred + gate * plane_pred
        blend_pred = slope_clip_absolute_candidate(test_meta, blend_pred, slope_bounds, slope_quantile=0.99)
        blend_submission = align_submission_to_sample(pd.DataFrame({'id': test_meta['id'].to_numpy(), 'tvt': blend_pred}))
        blend_keys = [blend_key] + (['plane_blend_safe'] if alias_generic else [])
        for key in blend_keys:
            blend_path = LGBM_CANDIDATE_FILES[key]
            blend_submission.to_csv(blend_path, index=False)
            saved_candidates[key] = blend_path
            candidate_predictions[key] = blend_pred
        summary_rows.append({
            'candidate': blend_key,
            'description': f'{label} conservative row-wise gate between mild LightGBM and formation-plane formula',
            'feature_set': ALL_ROW_LGBM_FEATURE_SET,
            'postprocess_source_feature_set': f'mild_lgbm_plus_formation_plane_{label}',
            'clip_delta': True,
            'alpha': np.nan,
            'fade_tau_md': np.nan,
            'slope_quantile': 0.99,
            'train_rows': train_rows,
            'test_rows': test_rows,
            'train_compact_frame_memory_mb': train_memory_mb,
            'train_matrix_memory_mb': x_train_memory_mb,
            'test_compact_frame_memory_mb': test_memory_mb,
            'test_matrix_memory_mb': x_test_memory_mb,
            'prediction_rows': len(blend_submission),
            'missing_predictions': int(blend_submission['tvt'].isna().sum()),
            'tvt_mean': float(blend_submission['tvt'].mean()),
            'tvt_std': float(blend_submission['tvt'].std()),
            'tvt_min': float(blend_submission['tvt'].min()),
            'tvt_max': float(blend_submission['tvt'].max()),
            'lightgbm_device_type': LIGHTGBM_DEVICE_TYPE,
            'output_file': str(LGBM_CANDIDATE_FILES[blend_key]),
            'formation_gate_mean': float(np.mean(gate)),
            'formation_gate_max': float(np.max(gate)),
            'formation_self_exclusion': label,
        })

    if 'formation_plane_tvt_formula' in test_aux.columns:
        save_formation_candidates(
            test_aux,
            label='private_safe',
            formula_key='formation_formula_private_safe',
            blend_key='plane_blend_private_safe',
            alias_generic=True,
        )
    elif WRITE_FORMATION_PROBE_SUBMISSIONS:
        private_aux = build_formation_probe_aux(exclude_self=True)
        save_formation_candidates(
            private_aux,
            label='private_safe',
            formula_key='formation_formula_private_safe',
            blend_key='plane_blend_private_safe',
            alias_generic=True,
        )
        del private_aux
        gc.collect()

    if WRITE_FORMATION_PROBE_SUBMISSIONS and PUBLIC_AGGRESSIVE_FORMATION:
        public_aux = build_formation_probe_aux(exclude_self=False)
        save_formation_candidates(
            public_aux,
            label='public_aggressive',
            formula_key='formation_formula_public_aggressive',
            blend_key='plane_blend_public_aggressive',
            alias_generic=False,
        )
        del public_aux
        gc.collect()

    final_candidate = FINAL_LGBM_CANDIDATE
    if final_candidate not in saved_candidates:
        available_candidates = list(saved_candidates.keys())
        message = (
            f'FINAL_LGBM_CANDIDATE={final_candidate!r} was not produced. '
            f'Available candidates: {available_candidates}'
        )
        if REQUIRE_FINAL_LGBM_CANDIDATE:
            raise RuntimeError(message)
        fallback_candidate = 'mild' if 'mild' in saved_candidates else next(iter(saved_candidates))
        print(message + f'; using {fallback_candidate!r} instead.')
        final_candidate = fallback_candidate
    final_source = saved_candidates[final_candidate]
    final_submission = pd.read_csv(final_source)
    final_submission.to_csv(FINAL_SUBMISSION_OUTPUT, index=False)

    summary = pd.DataFrame(summary_rows)
    summary['final_submission_candidate'] = final_candidate
    summary['final_submission_output'] = str(FINAL_SUBMISSION_OUTPUT)
    summary.to_csv(OUTPUT_DIR / 'submission_lgbm_offline_allrows_summary_v2.csv', index=False)

    print('All-row LightGBM device:', LIGHTGBM_DEVICE_TYPE)
    print('All-row LightGBM feature set:', ALL_ROW_LGBM_FEATURE_SET)
    print('Final submission source:', final_source)
    print('Final submission output:', FINAL_SUBMISSION_OUTPUT)
    return summary


if RUN_ALL_ROW_LIGHTGBM_CANDIDATE:
    if KAGGLE_NOTEBOOK_RUN:
        constant_anchor_fallback_submission = write_constant_anchor_fallback_submission()
    all_row_lgbm_summary = train_and_predict_all_row_lgbm()
    display(all_row_lgbm_summary)
else:
    print('All-row LightGBM candidate is disabled for this local/non-submission run.')
    print('For Kaggle submission: enable a GPU accelerator (T4 x2 or P100); this cell will run automatically and write submission.csv.')


### 📄 Prediction Tables and Artifacts

| Environment | File | Fit wells | Purpose |
|---|---|---:|---|
| Kaggle GPU fallback | `submission_constant_fallback_v2.csv` | - | last-known TVT anchor written before LightGBM as an OOM/failure safety net |
| Kaggle GPU final | `submission.csv` | 773 | final copy selected by `FINAL_LGBM_CANDIDATE` in Section 18 |
| Kaggle LightGBM candidate | `submission_lgbm_raw_v2.csv` | 773 | all-row LightGBM residual without post-processing |
| Kaggle LightGBM candidate | `submission_lgbm_mild_v2.csv` | 773 | safer compact LightGBM fallback candidate |
| Kaggle LightGBM candidate | `submission_lgbm_hgb_policy_v2.csv` | 773 | all-row LightGBM residual with stored HGB offline post-processing |
| Kaggle formation probe | `submission_formation_plane_formula_private_safe_v2.csv` | 773 | formation formula with same-well exclusion |
| Kaggle formation probe | `submission_lgbm_plane_blend_private_safe_v2.csv` | 773 | mild LGBM plus private-safe formation formula gate |
| Kaggle formation probe | `submission_formation_plane_formula_public_aggressive_v2.csv` | 773 | formation formula allowing same-well references for public diagnostic probing |
| Kaggle formation probe | `submission_lgbm_plane_blend_public_aggressive_v2.csv` | 773 | default score-seeking final candidate when produced |
| Kaggle formation aliases | `submission_formation_plane_formula_v2.csv`, `submission_lgbm_plane_blend_safe_v2.csv` | 773 | private-safe compatibility aliases |
| Kaggle HGB diagnostic | `submission_hgb_strict_v2.csv` | 773 | selected strict HGB prediction table; skipped by default in memory-safe Kaggle mode |
| Kaggle HGB diagnostic | `submission_hgb_public_clean_v2.csv` | 770 | selected strict public-clean diagnostic; skipped by default in memory-safe Kaggle mode |
| Kaggle HGB diagnostic | `submission_hgb_offline_candidate_path_v2.csv` | 773 | HGB best stored-CV policy; skipped by default in memory-safe Kaggle mode |
| Local | `submission_simple_residual_v2.csv` | 773 | selected strict all-train prediction table |
| Local | `submission_simple_residual_public_clean_v2.csv` | 770 | selected strict public-clean diagnostic |
| Local | `submission_simple_residual_best_overall_v2.csv` | 773 | best stored-CV policy, including offline features if selected |
| Local/Kaggle | `v2_cv_summary.csv`, `v2_policy_grid.csv`, `v2_cv_report.csv` | - | reproducible validation artifacts |
| Local/Kaggle | `v2_selected_policy_stage_report.csv`, `v2_selected_policy_tail_md_diagnostics.csv`, `v2_selected_policy_well_gain.csv` | - | selected OOF diagnostics |

By default, `submission.csv` uses `submission_lgbm_plane_blend_public_aggressive_v2.csv`. If that candidate is not produced, the run raises because `REQUIRE_FINAL_LGBM_CANDIDATE=True`. Use `submission_lgbm_mild_v2.csv` as the safer comparison file, and compare it against private-safe and formula-only formation probes.
